# RNN 파일별 1 · RNN 강의자료 (rnn_lecture.pdf)

**한 줄 요약**

- 강의 '순서가 중요하다'의 7부를 원본 순서대로 정리
- 3과 7 순서 과제로 합 · 마지막 값
- 펼치기의 한계를 본 뒤 hidden state · 모양 · padding · 기울기 소실 · LSTM 문 · GRU · 통제된 비교까지

- **원본** · `7. 딥러닝/04_RNN/강의자료/rnn_lecture.pdf` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/08_RNN` RNN-1 (순서 데이터 · RNN · LSTM · GRU) 1 ~ 6장

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 합 · 평균 · 마지막 값이 3과 7 순서 과제에서 각각 50% · 60% 근처인 이유를 계산으로 말한다
2. hidden state와 파라미터 공유를 설명하고 `RNNCell` + for 문을 `nn.RNN`과 맞춰 본다
3. 입력 세 겹 `(개수, 길이, 표현)` · `batch_first` · `output`과 `h_n`의 모양을 계산 전에 말한다
4. padding한 배치에서 `-1` 대신 `lens - 1` · mask로 읽어야 하는 이유를 보인다
5. 기울기 소실을 칸별로 재고, LSTM의 forget 문을 열었을 때만 기울기가 남는 것과 LSTM · GRU 비교법을 말한다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **새 시퀀스 데이터를 받았을 때** · 순서 섞기 검사 · 길이 확인 (1장 · 8장 아홉 단계)
- **RNN 코드에서 shape 에러가 났을 때** · 세 겹 · `batch_first` · one_hot (3장)
- **길이가 다른 문장 · 기록을 배치로 묶을 때** · padding · `lens - 1` · mask (4장)
- **긴 시퀀스 학습이 안 될 때** · 칸별 기울기 측정 · clipping · LSTM forget bias (5 · 6장)
- **LSTM과 GRU 중 하나를 고를 때** · 통제된 비교 두 번 · 비용 (7장)

## 학습 지도

<img src="images/rl_map.png" alt="rnn_lecture 학습 지도" width="763">

**예상 학습 시간** · 개념 2시간 + 실습 40분 (전체 실행 1분 이내 · 학습은 8장만)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>실행 환경 · 강의 구성</td></tr>
<tr><td>1</td><td>순서가 중요하다</td><td>숫자가 같은 두 줄의 답이 왜 반대이고, 순서가 중요한지는 무엇이 정하는가?</td></tr>
<tr><td>2</td><td>무엇을 넘길 것인가</td><td>하나씩 읽을 때 앞에서 뒤로 무엇을 넘겨야 하고, 그것을 누가 정하는가?</td></tr>
<tr><td>3</td><td>코드로 옮기기</td><td>그림 속 RNN을 코드로 옮기면 입력 · 출력 모양은 어떻게 되는가?</td></tr>
<tr><td>4</td><td>길이가 다른 줄</td><td>길이가 제각각인 줄을 한 텐서에 담고 올바르게 읽으려면?</td></tr>
<tr><td>5</td><td>넘긴 것이 끝까지 닿는가</td><td>멀리 떨어진 정보를 못 쓰는 것은 기억 탓인가, 학습 탓인가?</td></tr>
<tr><td>6</td><td>LSTM · 남길 것을 학습</td><td>LSTM은 무엇을 더 달았고, 언제 기울기가 살아나는가?</td></tr>
<tr><td>7</td><td>GRU와 고르는 법</td><td>LSTM과 GRU 중 무엇을 고르고, 비교는 어떻게 해야 공정한가?</td></tr>
<tr><td>8</td><td>종합 실습</td><td>새 시퀀스 문제를 만나면 어떤 순서로 묻고, 각 단계에서 무엇을 확인하는가?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

강의 슬라이드의 숫자는 `rnn.ipynb` 실행 결과임. 이 노트북은 학습 없이 바로 확인되는 계산으로 같은 숫자를 다시 냄 (8장만 짧게 학습). **셀은 위에서부터 차례로 실행**.

*항목 → 설정*

- **라이브러리** · `torch` · `random` · `collections`
- **seed** · `SEED = 20260912` (강의 노트북과 같음)
- **장치** · 1 ~ 7장은 CPU 계산 · 8장 학습만 GPU (없으면 CPU)
- **데이터** · 강의의 3과 7 순서 과제 (학습 6,000줄 · 평가 2,000줄)를 같은 난수 순서로 만듦
- **강의 숫자** · 학습이 필요한 정확도(48.8% 등)는 강의 화면 값 · 직접 학습은 파일별 2

**실행 흐름**

1. 도구 불러오기 · seed · 결정적 설정 · 장치
2. 순서 과제 · 앞부분 과제 만드는 함수 정의
3. 학습 · 평가 데이터 만들기

In [1]:
# 라이브러리 · seed · 장치 · 강의 과제 데이터 (강의 숫자는 rnn.ipynb 실행 결과)
# 왜 · 강의와 같은 규칙으로 데이터를 만들어 화면의 숫자를 직접 재현하기 위해
import random                                 # 파이썬 난수 (데이터 만들기)
from collections import Counter, defaultdict  # 개수 세기 · 묶어 세기

import torch                     # 텐서 · 자동 미분
import torch.nn as nn            # 층: RNN · LSTM · GRU · Linear
import torch.nn.functional as F  # one_hot

# 왜 · seed와 결정적 설정을 한 곳에서 정해 같은 환경에서 같은 결과를 얻기 위해
SEED = 20260912                                                     # 강의 노트북과 같은 seed
torch.backends.cudnn.deterministic = True                           # 같은 GPU · 같은 버전에서 매번 같은 계산 결과
torch.backends.cudnn.benchmark = False                              # 실행마다 계산 방법을 새로 고르지 않음
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 8장 학습만 GPU 사용
OTHERS = [d for d in range(10) if d not in (3, 7)]                  # 3 · 7을 뺀 채울 숫자


# 왜 · '3과 7 중 무엇이 먼저 나왔나'처럼 순서가 답을 정하는 과제를 만들기 위해
def make_order_task(n, length=10, seed=None):                # 3이 먼저면 1 (강의와 같은 난수 순서)
    rng = random.Random(seed)                                # 이 데이터 전용 난수
    xs, ys = [], []                                          # 입력 줄 · 정답 목록
    for _ in range(n):                                       # 줄 n개
        s = [rng.choice(OTHERS) for _ in range(length - 2)]  # 채울 숫자
        i, j = sorted(rng.sample(range(length), 2))          # 3 · 7 자리
        y = rng.randint(0, 1)                                # 정답을 먼저 정함
        s.insert(i, 3 if y else 7)                           # 앞 자리
        s.insert(j, 7 if y else 3)                           # 뒤 자리
        xs.append(s)                                         # 줄 저장
        ys.append(y)                                         # 정답 저장
    return xs, ys                                            # 줄 목록 · 정답 목록


# 왜 · 중요한 정보가 맨 앞에 있어 멀리 기억해야 하는 과제를 만들기 위해
def make_front_task(n, length, seed=None):  # 3 · 7을 맨 앞 두 칸에 (5장)
    rng = random.Random(seed)               # 전용 난수
    xs, ys = [], []                         # 입력 줄 · 정답 목록
    for _ in range(n):                      # 줄 n개
        y = rng.randint(0, 1)               # 정답 먼저
        # 뒤는 상관없는 숫자
        xs.append(([3, 7] if y else [7, 3]) + [rng.choice(OTHERS) for _ in range(length - 2)])
        ys.append(y)                        # 정답 저장
    return xs, ys                           # 줄 목록 · 정답 목록


# 왜 · 숫자 줄을 모델이 받는 (개수, 길이, 10) 원-핫 텐서로 바꾸기 위해
def to_tensor(xs):                                              # 숫자 줄 → (줄, 길이, 10) one-hot
    return F.one_hot(torch.tensor(xs), num_classes=10).float()  # 숫자 하나 = 열 칸


# 왜 · 학습 · 평가 데이터를 서로 다른 seed로 만들어 겹치지 않게 하기 위해
xtr, ytr = make_order_task(6000, 10, seed=1)  # 학습 6,000줄 (강의와 같은 크기)
xte, yte = make_order_task(2000, 10, seed=2)  # 평가 2,000줄
# 환경
print(f"torch {torch.__version__} · device {DEV.type} · 학습 {len(xtr)}줄 · 평가 {len(xte)}줄")

torch 2.11.0+cu128 · device cuda · 학습 6000줄 · 평가 2000줄


### 원본 구성

- **1부 · 순서가 중요하다** · 3과 7 중 무엇이 먼저인가 · 합으로 48.8% · 질문이 순서를 정함
- **2부 · 무엇을 넘길 것인가** · 마지막 값 60.4% · 전부 넘기기 · '봤는가' 51.2% · hidden state · 파라미터 공유
- **3부 · 코드로 옮기면** · RNNCell · nn.RNN · 세 겹 모양 · batch_first · output vs h_n · UCI HAR · padding
- **4부 · 넘긴 것이 안 닿는다** · 거리 실험 · 귓속말 · 기울기 소실 · BPTT · 0.9를 50번
- **5부 · 남길 것을 학습시키자** · 문(gate) · cell state · LSTM 기본도 죽음 · 문을 열어두면
- **6부 · 더 적은 손잡이** · GRU · 구조만 바꾼 비교 · 조건을 맞춘 비교 · 비용
- **7부 · 남는 것** · 아홉 단계 · 양방향 · 세 문장

**원본 힌트**

- 첫 문장 · 순서를 섞어도 답이 같은가? 다르다면 길이도 매번 다른가?
- 두 번째 문장 · 무엇을 넘길 것인가? 사람이 정할까, 학습시킬까?
- 세 번째 문장 · 넘긴 것이 끝까지 닿는가? 못 닿는 게 기억 탓인가, 학습 탓인가?
- 마지막 문장 · 구조가 다르다고 성능이 다른 건 아님

---
## 1. 순서가 중요하다

> **이 장의 흐름**
>
> - 강의 1부 · 정의 대신 문제부터
> - 합으로 풀어 보고 → 어디서 막히는지 → 질문을 바꾸면 어떻게 되는지

<img src="images/rl_order.png" alt="같은 숫자 10개 · 순서만 달라 답이 반대" width="718">

**파이프라인 위치** · ① 문제 정의 — 순서가 답을 정하는지 확인

### 1-1. `A` 순서만 다른 두 줄 · `sum()` · `sorted()` <sub>강의 1부 · '9 3 1 0 7 5 8 6 8 4'</sub>

> 🎯 **숫자가 똑같은 두 줄의 답이 왜 반대가 되고, 합 · 평균은 왜 이 차이를 못 보는가?**

💡 **쉽게 말하면**

- 두 줄은 **같은 재료로 만든 다른 요리**.
- 합 · 평균 · 최댓값 · 개수는 재료만 보고 순서는 안 봄.

**실행 흐름**

1. 두 줄의 합 · 평균 · 최댓값 · 개수
2. 정렬해서 재료 비교
3. 3 · 7의 자리로 답 계산

**먼저 예측** · 두 줄의 합은 같을까? — 답은 코드 실행 결과 아래에

In [2]:
# 강의 첫 두 줄: 들어 있는 숫자는 같고 순서만 다름
# 왜 · 합 · 정렬처럼 순서를 버리는 요약으로는 두 줄을 구분할 수 없음을 보이기 위해
a = [9, 3, 1, 0, 7, 5, 8, 6, 8, 4]                      # 3이 두 번째 · 7이 다섯 번째 → 네
b = [9, 7, 1, 0, 3, 5, 8, 6, 8, 4]                      # 7이 두 번째 · 3이 다섯 번째 → 아니요
# 순서 안 보는 계산
for name, f in [("합", sum), ("평균", lambda s: sum(s) / len(s)), ("최댓값", max), ("개수", len)]:
    print(f"{name:<4} {f(a)} vs {f(b)}")                # 둘 다 같음
print("재료(정렬)가 같은가 :", sorted(a) == sorted(b),  # 재료 비교
      "· 8 의 개수 :", Counter(a)[8], Counter(b)[8])
ans_a, ans_b = int(a.index(3) < a.index(7)), int(b.index(3) < b.index(7))  # 3이 먼저면 1
print("3 의 자리 :", a.index(3), "vs", b.index(3), "→ 답", ans_a, ans_b)  # 순서만 다름

합    51 vs 51
평균   5.1 vs 5.1
최댓값  9 vs 9
개수   10 vs 10
재료(정렬)가 같은가 : True · 8 의 개수 : 2 2
3 의 자리 : 1 vs 4 → 답 1 0


**결과** · 예측 정답은 같음 — 둘 다 51

- `합    51 vs 51` — 합이 같음 → 합만 보면 두 줄을 구분할 수 없음
- `재료(정렬)가 같은가 : True · 8 의 개수 : 2 2` — 8이 두 번인 것까지 같은 재료
- `3 의 자리 : 1 vs 4 → 답 1 0` — 다른 것은 3과 7의 순서뿐 → 답이 뒤집힘

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>합 · 평균 · 최댓값 · 개수</td><td>순서를 <strong>보지 않는</strong> 계산</td><td>순서가 상관없는 질문 (합이 45 이상인가)</td></tr>
<tr><td><code>sorted(a) == sorted(b)</code></td><td>재료가 같은지</td><td>순서만 다른지 확인할 때</td></tr>
<tr><td><code>a.index(3) &lt; a.index(7)</code></td><td>순서를 <strong>보는</strong> 계산</td><td>어느 쪽이 먼저인지 물을 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - '장바구니 → 결제'와 '결제 → 장바구니'처럼 순서가 뒤집히면 의미가 뒤집힘
> - 재료가 같다고 같은 데이터가 아님

> ✅ **한 줄 정리** · 합 · 평균 · 최댓값 · 개수는 순서를 버리는 계산이라, 순서만 다른 두 줄을 구분하지 못한다.

**딥러닝 코드에서는**

- 문장 · 센서 · 클릭 기록은 순서가 의미를 만듦
- 순서를 버리는 평균 풀링(`x.mean(1)`)은 쓰기 전에 질문부터 확인

### 1-2. `A` 순서가 중요한지는 질문이 정함 · `defaultdict(Counter)` <sub>강의 1부 · '합이 나쁜 방법인 것은 아님'</sub>

> 🎯 **같은 데이터인데 합 하나로 질문 A는 48.8%, 질문 B는 96%가 나오는 이유는?**

💡 **쉽게 말하면**

- 합 하나만 받은 모델이 할 수 있는 최선은 **합마다 많은 쪽 답을 외우는 것**.
- 그 최선이 질문 A에서는 찍기, 질문 B에서는 만점.

**실행 흐름**

1. 학습 줄로 '합 → 다수결 답' 표 만들기
2. 평가 줄에 적용 (질문 A · B)
3. 합이 51인 줄의 정답 개수 보기

**먼저 예측** · 질문 A에서 합으로 낼 수 있는 최고 정확도는 얼마일까? — 답은 코드 실행 결과 아래에

In [3]:
# 합 하나만 보고 낼 수 있는 최고 정확도: 학습에서 합마다 많은 쪽 답을 외워 평가에 적용
# 왜 · 순서 정보 없이 얻을 수 있는 점수의 상한을 기준으로 두기 위해
def best_by_sum(xs, ys, xs_e, ys_e):                            # 합 → 다수결 표
    table = defaultdict(Counter)                                # 합마다 정답 개수
    for s, y in zip(xs, ys):                                    # 학습 줄
        table[sum(s)][y] += 1                                   # 합 칸에 정답 세기
    pred = [table[sum(s)].most_common(1)[0][0] if sum(s) in table else 0 for s in xs_e]  # 다수결 답
    return sum(p == y for p, y in zip(pred, ys_e)) / len(ys_e)  # 정확도


# 왜 · 합이 정답을 정하는 다른 질문과 비교해 순서가 중요한지는 질문이 정한다는 것을 보이기 위해
q45 = [int(sum(s) >= 45) for s in xtr]                      # 질문 B: 합이 45 이상인가 (학습)
q45e = [int(sum(s) >= 45) for s in xte]                     # 질문 B (평가)
# 찍기 근처
print(f"질문 A '3 다음에 7인가'  · 합으로 낼 수 있는 최고 {best_by_sum(xtr, ytr, xte, yte):.3f}")
# 완벽
print(f"질문 B '합이 45 이상인가' · 합으로 낼 수 있는 최고 {best_by_sum(xtr, q45, xte, q45e):.3f}")
c51 = Counter(y for s, y in zip(xtr, ytr) if sum(s) == 51)  # 합 51인 줄의 정답 개수
print("같은 합 51 인 학습 줄의 정답 개수 :", dict(sorted(c51.items())))  # 반반

질문 A '3 다음에 7인가'  · 합으로 낼 수 있는 최고 0.501
질문 B '합이 45 이상인가' · 합으로 낼 수 있는 최고 1.000
같은 합 51 인 학습 줄의 정답 개수 : {0: 114, 1: 120}


**결과** · 예측 정답은 약 0.5 — 같은 합 안에 정답 0 · 1이 반반

- `질문 A '3 다음에 7인가'  · 합으로 낼 수 있는 최고 0.501` — 어떤 모델이라도 합만 받으면 찍기 수준 (강의 48.8%)
- `질문 B '합이 45 이상인가' · 합으로 낼 수 있는 최고 1.000` — 답이 합으로 정해짐 (강의 MLP 96%)
- `같은 합 51 인 학습 줄의 정답 개수 : {0: 114, 1: 120}` — 합 51 안에 두 답이 섞여 있음 → 합으로는 못 가름

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>입력에 답이 없음</td><td>데이터 · 층 · 학습을 늘려도 그대로</td><td>모델을 키우기 전에 입력부터 의심</td></tr>
<tr><td>입력에 답이 있음</td><td>작은 모델로도 맞힘</td><td>질문 B처럼 요약이 질문과 맞을 때</td></tr>
<tr><td>다수결 표 (이 섹션)</td><td>그 입력으로 낼 수 있는 상한</td><td>기준선(baseline) 해석</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 50%는 '아무것도 배우지 못했다'는 뜻 → 모델 탓이 아니라 입력 탓일 수 있음

> ✅ **한 줄 정리** · 합 하나에는 순서 정보가 없어 질문 A의 상한이 찍기이고, 순서가 중요한지는 데이터가 아니라 질문이 정한다.

**딥러닝 코드에서는**

- 새 특징을 만들 때 '이 특징만으로 낼 수 있는 상한'을 먼저 계산

### 1-3. `B` 순서 섞기 검사와 길이 문제 · `random.sample()` <sub>강의 1부 · '새 데이터를 만나면 먼저 할 것'</sub>

> 🎯 **새 데이터에서 순서가 중요한지 어떻게 확인하고, 길이가 매번 다르면 무엇이 문제인가?**

💡 **쉽게 말하면**

- **한 줄 안에서 칸 순서만 섞어** 답이 바뀌는지 봄 (줄끼리 섞는 게 아님).
- 길이가 다르면 입력 칸 수를 몇으로 고정할지가 새 문제.

**실행 흐름**

1. 줄마다 칸 순서를 섞음
2. 질문 A · B 답을 다시 계산
3. 사용자별 클릭 수를 15칸에 맞춰 봄

**먼저 예측** · 섞은 뒤 질문 B의 답이 바뀐 줄은 몇 개일까? — 답은 코드 실행 결과 아래에

In [4]:
# 새 데이터를 만나면 먼저: 한 줄 안에서 칸 순서를 섞어 답이 바뀌는지
# 왜 · 순서가 중요한 데이터인지 섞어 보기로 먼저 확인하기 위해
rng = random.Random(0)                                 # 섞기 전용 난수
shuffled = [rng.sample(s, len(s)) for s in xte]        # 줄마다 칸 순서만 섞음 (줄끼리는 그대로)
qa = [int(s.index(3) < s.index(7)) for s in shuffled]  # 질문 A 답 다시 계산
qb = [int(sum(s) >= 45) for s in shuffled]             # 질문 B 답 다시 계산
# 절반가량
print(f"질문 A · 섞은 뒤 답이 바뀐 줄 {sum(x != y for x, y in zip(qa, yte)) / len(yte):.3f}")
qb0 = [int(sum(s) >= 45) for s in xte]                 # 섞기 전 질문 B 답
print(f"질문 B · 섞은 뒤 답이 바뀐 줄 {sum(x != y for x, y in zip(qb, qb0))}")  # 0줄


# 두 번째 갈래: 길이가 매번 다르면 입력 칸을 몇 개로 고정하나
# 왜 · 길이가 다른 입력을 고정 크기 모델에 넣기 어렵다는 문제를 보이기 위해
clicks = {"A": 3, "B": 15, "C": 7, "D": 22}  # 사용자별 클릭 수 (예)
for user, n in clicks.items():               # 15칸으로 고정했다면
    # 채우거나 잘라야 함
    print(f"사용자 {user} · 클릭 {n:>2} → 빈 칸 {max(0, 15 - n):>2} · 넘침 {max(0, n - 15)}")

질문 A · 섞은 뒤 답이 바뀐 줄 0.494
질문 B · 섞은 뒤 답이 바뀐 줄 0
사용자 A · 클릭  3 → 빈 칸 12 · 넘침 0
사용자 B · 클릭 15 → 빈 칸  0 · 넘침 0
사용자 C · 클릭  7 → 빈 칸  8 · 넘침 0
사용자 D · 클릭 22 → 빈 칸  0 · 넘침 7


**결과** · 예측 정답은 0줄 — 합은 순서와 무관

- `질문 A · 섞은 뒤 답이 바뀐 줄 0.494` — 절반가량이 바뀜 → 순서가 답을 정함
- `질문 B · 섞은 뒤 답이 바뀐 줄 0` — 하나도 안 바뀜 → 순서가 상관없음
- `사용자 A · 클릭  3 → 빈 칸 12 · 넘침 0` — 15칸 중 12칸을 무엇으로 채울지 (4장)
- `사용자 D · 클릭 22 → 빈 칸  0 · 넘침 7` — 15칸보다 길면 잘라야 함

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>한 줄 안 칸 섞기</td><td>순서만 파괴 · 값의 모음은 그대로</td><td>순서가 중요한지 검사</td></tr>
<tr><td>줄끼리 섞기</td><td>학습 순서만 바뀜</td><td>미니배치 셔플 (검사 아님)</td></tr>
<tr><td>고정 길이로 채우기 · 자르기</td><td>모양은 맞지만 빈 칸 · 잘림이 생김</td><td>길이가 다른 데이터 (4장)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 0으로 채우면 모델은 그 0도 값으로 읽음 → 4장에서 다시 다룸

> ✅ **한 줄 정리** · 한 줄 안의 칸 순서를 섞어 성능이 떨어지면 순서가 중요하고, 그다음 길이가 매번 다른지를 묻는다.

**딥러닝 코드에서는**

- HAR 실전 검사 — 128칸 순서를 섞으면 LSTM 정확도가 떨어짐 (파일별 2 8-4)

---
## 2. 무엇을 넘길 것인가

> **이 장의 흐름**
>
> - 강의 2부 · 사람이 넘길 것을 정해 보고(마지막 값 · 전부 · 봤는가) 깨지는 곳 확인
> - 그다음 크기만 정하고 안은 학습에 맡김 = hidden state

<img src="images/rl_candidates.png" alt="넘길 것별 정확도 (강의 숫자)" width="782">

**파이프라인 위치** · ② 표현 설계 — 무엇을 요약해 넘길지

### 2-1. `A` 첫 후보: 마지막 값 하나 · `s[-1]` <sub>강의 2부 · '왜 하필 60% 인가'</sub>

> 🎯 **마지막 값 하나만 넘기면 왜 하필 60% 근처가 나오는가?**

💡 **쉽게 말하면**

- 끝 칸이 7이면 3은 반드시 앞 → **답 확정**.
- 끝 칸이 3 · 7이 아니면 단서가 없어 **찍기**.

**실행 흐름**

1. 끝 칸 · 첫 칸이 3 · 7인 비율
2. 기대 정확도 계산
3. 끝 칸 7 · 3일 때의 정답 확인

**먼저 예측** · 열 칸 중 3 · 7이 끝 칸에 올 확률은? — 답은 코드 실행 결과 아래에

In [5]:
# 마지막 값 하나만 넘기면: 끝 칸이 3 · 7이면 답이 확정, 나머지는 찍기
# 왜 · 마지막 값만 보면 답이 정해지는 경우와 아닌 경우를 나눠 한계를 확인하기 위해
last_hit = sum(s[-1] in (3, 7) for s in xte) / len(xte)                 # 끝 칸이 3 · 7인 비율
first_hit = sum(s[0] in (3, 7) for s in xte) / len(xte)                 # 첫 칸이 3 · 7인 비율
# 확정 + 반은 맞힘
print(f"마지막 칸이 3 · 7 : {last_hit:.3f} → 기대 정확도 {last_hit + (1 - last_hit) / 2:.3f}")
# 첫 값도 같은 이유
print(f"첫 칸이 3 · 7     : {first_hit:.3f} → 기대 정확도 {first_hit + (1 - first_hit) / 2:.3f}")
print("이론 : 0.2 × 1 + 0.8 × 0.5 =", round(0.2 * 1 + 0.8 * 0.5, 3))    # 강의 계산
print("끝 칸이 7 인 줄의 정답 :", set(y for s, y in zip(xte, yte) if s[-1] == 7),  # 항상 네
      "· 끝 칸이 3 :", set(y for s, y in zip(xte, yte) if s[-1] == 3))  # 항상 아니요

마지막 칸이 3 · 7 : 0.211 → 기대 정확도 0.606
첫 칸이 3 · 7     : 0.212 → 기대 정확도 0.606
이론 : 0.2 × 1 + 0.8 × 0.5 = 0.6
끝 칸이 7 인 줄의 정답 : {1} · 끝 칸이 3 : {0}


**결과** · 예측 정답은 10분의 2 — 그래서 0.2 × 1 + 0.8 × 0.5 = 0.6

- `마지막 칸이 3 · 7 : 0.211 → 기대 정확도 0.606` — 평가 데이터로 센 값 (강의 60.4%)
- `첫 칸이 3 · 7     : 0.212 → 기대 정확도 0.606` — 첫 값도 같은 이유 (강의 60.9%)
- `이론 : 0.2 × 1 + 0.8 × 0.5 = 0.6` — 확정 2할 + 찍기 8할의 절반
- `끝 칸이 7 인 줄의 정답 : {1} · 끝 칸이 3 : {0}` — 끝 칸이 표식이면 예외 없이 확정

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>마지막 값</td><td>가장 싸고 단순 · 직전 것만</td><td>최근 값이 답을 정할 때</td></tr>
<tr><td>첫 값</td><td>맨 앞만</td><td>시작 상태가 답을 정할 때</td></tr>
<tr><td>합 · 평균</td><td>전체 재료 · 순서 없음</td><td>순서가 상관없는 질문</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 모델을 키워도 60% 근처 (강의 '모델을 키우면' 셀 · 파일별 2 1-4)
> - 문제는 모델이 아니라 넘긴 것

> ✅ **한 줄 정리** · 마지막 값 하나는 표식이 끝 칸에 온 2할만 맞히므로 0.2 × 1 + 0.8 × 0.5 = 0.6이 상한이다.

**딥러닝 코드에서는**

- 시계열에서 '마지막 값 그대로' 예측은 꼭 재 보는 기준선 (RNN-2)

### 2-2. `B` 두 번째 후보: 전부 넘기기 · `L × 10` <sub>강의 2부 · '먼저 — 통째로 펼쳐서 한 번에'</sub>

> 🎯 **줄을 통째로 펼쳐 넣으면 길이가 길어질 때 왜 무너지는가?**

💡 **쉽게 말하면**

- 전부 넘기면 정보는 안 버리지만 **짐(입력 칸)이 길이만큼 커짐**.
- 모델이 커지는데 데이터는 그대로라 배울 기회가 모자람.

**실행 흐름**

1. 길이별 입력 칸 수 · 파라미터 수
2. 칸마다 다른 모델이면 몇 개인지

**먼저 예측** · 길이 50을 펼친 MLP의 파라미터 수는? — 답은 코드 실행 결과 아래에

In [6]:
# 전부 넘기기 = 줄을 통째로 펼쳐 한 번에: 입력 L × 10칸짜리 MLP (은닉 32 · 32)
# 왜 · 모두 펼쳐 넣으면 길이에 따라 모델이 커지는 문제를 숫자로 보기 위해
def mlp_params(n_in, width=32):  # Linear 3개 (입력 → 32 → 32 → 2)
    return (n_in * width + width) + (width * width + width) + (width * 2 + 2)  # 가중치 + bias


# 왜 · 길이가 늘 때 파라미터가 얼마나 커지는지 비교하기 위해
for L in (10, 20, 50, 100, 200):  # 길이만 바꿈
    # 길이만큼 커짐
    print(f"길이 {L:>3} → 입력 {L * 10:>4}칸 · 파라미터 {mlp_params(L * 10):>6,}개")


# 칸마다 다른 모델이면: k번째 칸 모델은 앞 k - 1칸을 입력으로 받음
# 왜 · 칸마다 따로 모델을 두면 생기는 문제를 확인하기 위해
# 가중치 공유 불가
print("3번째 칸 모델 입력", 3 - 1, "개 · 10번째", 10 - 1, "개 · 200칸이면 모델", 200, "개")

길이  10 → 입력  100칸 · 파라미터  4,354개
길이  20 → 입력  200칸 · 파라미터  7,554개
길이  50 → 입력  500칸 · 파라미터 17,154개
길이 100 → 입력 1000칸 · 파라미터 33,154개
길이 200 → 입력 2000칸 · 파라미터 65,154개
3번째 칸 모델 입력 2 개 · 10번째 9 개 · 200칸이면 모델 200 개


**결과** · 예측 정답은 17,154개 — 500 × 32 + 32 + 1,056 + 66

- `길이  50 → 입력  500칸 · 파라미터 17,154개` — 강의의 17,154개 (길이 50 · 정확도 95.9%)
- `길이 200 → 입력 2000칸 · 파라미터 65,154개` — 길이만큼 커짐 · 더 긴 줄은 넣을 수 없음
- `3번째 칸 모델 입력 2 개 · 10번째 9 개 · 200칸이면 모델 200 개` — 칸마다 크기가 달라 가중치를 같이 못 씀

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>통째로 펼치기</td><td>정보 손실 없음 · 길이에 묶임</td><td>길이가 짧고 고정 · 데이터가 많을 때</td></tr>
<tr><td>칸마다 다른 모델</td><td>칸마다 입력 크기가 다름</td><td>쓰지 않음 (모델 수 = 길이)</td></tr>
<tr><td>크기 고정 요약 (2-4)</td><td>길이와 무관한 같은 모델</td><td>RNN</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 강의 숫자 · 길이 10 → 100% · 100 → 88.4% · 길이 50에서 학습 500건 → 67.0%
> - 순서 문제를 길이 문제로 바꿔 놓았을 뿐

> ✅ **한 줄 정리** · 전부 넘기기는 길이만큼 모델이 커져 데이터가 모자라지고, 가장 긴 길이보다 긴 줄은 넣을 수도 없다.

**딥러닝 코드에서는**

- 이미지를 펼쳐 MLP에 넣으면 같은 문제 → CNN은 같은 필터를 위치마다 다시 씀

### 2-3. `A` 무엇을 기억해야 하나 · `3 in s` · `상태 셋` <sub>강의 2부 · '그럼 실제로 무엇이 필요한가'</sub>

> 🎯 **'3을 봤는가 · 7을 봤는가'는 왜 51.2%이고, 상태 셋은 왜 100%인가?**

💡 **쉽게 말하면**

- 3 · 7은 **언제나 둘 다** 들어 있어 '봤는가'는 늘 [1, 1].
- 필요한 것은 **어느 쪽을 먼저 봤는가** — 읽으면서 상태를 바꿔 끝까지 들고 감.

**실행 흐름**

1. 봤는가 두 칸의 종류 세기
2. 왼쪽부터 읽으며 상태 갱신
3. 끝 상태로 답 맞히기

**먼저 예측** · '봤는가' 두 칸 입력은 몇 종류일까? — 답은 코드 실행 결과 아래에

In [7]:
# 무엇을 기억해야 하나: '봤는가' 두 칸 vs '먼저 봤는가' 상태 셋
# 왜 · 끝까지 넘겨야 할 정보가 무엇인지 상태로 정리하기 위해
seen = Counter((int(3 in s), int(7 in s)) for s in xtr)       # 봤는가 두 칸의 종류
print("'3 을 봤는가 · 7 을 봤는가' 입력 종류 :", dict(seen))  # 모든 줄이 (1, 1)


# 왜 · 줄을 앞에서부터 읽으며 상태 하나만 갱신해도 답을 낼 수 있음을 보이기 위해
def first_state(s):                     # 0 = 아직 안 봄 · 1 = 3이 먼저 · 2 = 7이 먼저
    state = 0                           # 처음엔 아무것도 안 봄
    for d in s:                         # 왼쪽부터 하나씩 읽음
        if state == 0 and d in (3, 7):  # 처음 본 표식만 기록
            state = 1 if d == 3 else 2  # 상태 갱신
    return state                        # 끝까지 읽은 뒤 상태


# 왜 · 손으로 만든 상태 규칙이 정답과 맞는지 확인하기 위해
states = [first_state(s) for s in xte]                               # 평가 줄마다 끝 상태
print("끝 상태 개수 :", dict(sorted(Counter(states).items())))       # 1 · 2만 남음
rule = sum((st == 1) == y for st, y in zip(states, yte)) / len(yte)  # 상태 1이면 네
print(f"상태 1 → 네 규칙의 정확도 : {rule:.3f}")                     # 상태 하나로 충분

'3 을 봤는가 · 7 을 봤는가' 입력 종류 : {(1, 1): 6000}
끝 상태 개수 : {1: 1024, 2: 976}
상태 1 → 네 규칙의 정확도 : 1.000


**결과** · 예측 정답은 한 종류 — 모든 줄이 (1, 1)

- `'3 을 봤는가 · 7 을 봤는가' 입력 종류 : {(1, 1): 6000}` — 6000줄이 모두 같은 입력 → 찍기 (강의 51.2%)
- `끝 상태 개수 : {1: 1024, 2: 976}` — '3이 먼저' · '7이 먼저' 두 상태만 남음
- `상태 1 → 네 규칙의 정확도 : 1.000` — 상태 하나로 답이 정해짐 (강의도 만점)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>봤는가 (2칸)</td><td>있는지만</td><td>포함 여부가 답일 때</td></tr>
<tr><td>먼저 봤는가 (상태 셋)</td><td>읽는 순서대로 상태 갱신</td><td>순서가 답일 때</td></tr>
<tr><td>사람이 정한 상태</td><td>문제마다 다시 설계</td><td>규칙이 분명할 때만</td></tr>
<tr><td>학습한 상태 (2-4)</td><td>무엇을 담을지 모델이 정함</td><td>규칙을 모를 때 (보통)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 넘길 것을 사람이 정하면 맞을 때도 있고 아닐 때도 있음 → 맞는지 확인을 안 했음

> ✅ **한 줄 정리** · 필요한 것은 '봤는가'가 아니라 '어느 쪽을 먼저 봤는가'라는 상태이며, 이 상태를 읽으면서 갱신해 끝까지 들고 가야 한다.

**딥러닝 코드에서는**

- 이미지에서 무엇을 볼지 사람이 안 정하듯, 넘길 것도 학습시키는 것이 RNN

### 2-4. `A` hidden state와 파라미터 공유 · `nn.RNN` · `parameters()` <sub>강의 2부 · '이 요약에 붙은 이름' · '파라미터 공유'</sub>

> 🎯 **넘기는 요약의 크기를 고정하면 왜 길이와 상관없이 같은 모델을 쓸 수 있는가?**

💡 **쉽게 말하면**

- 요약 크기가 늘 32칸이면 1번째 칸도 200번째 칸도 **같은 상자**를 쓸 수 있음.
- 상자 안에 무엇을 담을지는 학습이 정함 = **hidden state**.

<img src="images/rl_hidden_flow.png" alt="hidden state 시간축 흐름 · 같은 상자를 칸마다 다시 씀" width="627">

**실행 흐름**

1. RNN 모델 하나 만들기
2. 길이 10 · 50 · 200을 넣어 출력 · 파라미터 수 보기
3. 손계산

**먼저 예측** · 길이가 20배가 되면 파라미터 수는? — 답은 코드 실행 결과 아래에

In [8]:
# 넘기는 요약의 크기를 32로 고정 → 같은 상자(같은 가중치)를 칸마다 다시 씀
# 왜 · 길이와 상관없이 같은 상자(같은 가중치)를 칸마다 쓰는 RNN을 만들기 위해
class Net(nn.Module):                              # RNN → 마지막 요약 → 네 · 아니요
    def __init__(self, kind="RNN", hid=32):        # 셀 종류 · 요약 크기
        super().__init__()                         # nn.Module 준비 (필수)
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]  # 이름으로 층 고르기
        self.r = layer(10, hid, batch_first=True)  # 입력 10 · 요약 hid
        self.o = nn.Linear(hid, 2)                 # 요약 → 점수 2개

    # 왜 · 칸마다 요약을 만든 뒤 마지막 요약으로 답을 내기 위해
    def forward(self, x):           # 입력이 지나가는 길
        o, _ = self.r(x)            # 칸마다 요약
        return self.o(o[:, -1, :])  # 마지막 칸 요약 → 점수


torch.manual_seed(SEED)                           # 같은 시작점
net = Net("RNN")                                  # 모델 하나
for L in (10, 50, 200):                           # 길이만 바꿔 넣음
    xs, _ = make_order_task(4, L, seed=3)         # 길이 L인 4줄
    n = sum(p.numel() for p in net.parameters())  # 파라미터 수
    print(f"길이 {L:>3} → 출력 {tuple(net(to_tensor(xs)).shape)} · 파라미터 {n:,}개")  # 길이와 무관
hand = 32 * 10 + 32 * 32 + 32 + 32 + 32 * 2 + 2   # W_ih + W_hh + bias 2개 + Linear
print("손계산 : 32 × 10 + 32 × 32 + 32 + 32 + 32 × 2 + 2 =", hand)  # 1,474

길이  10 → 출력 (4, 2) · 파라미터 1,474개
길이  50 → 출력 (4, 2) · 파라미터 1,474개
길이 200 → 출력 (4, 2) · 파라미터 1,474개
손계산 : 32 × 10 + 32 × 32 + 32 + 32 + 32 × 2 + 2 = 1474


**결과** · 예측 정답은 그대로 1,474개

- `길이  10 → 출력 (4, 2) · 파라미터 1,474개` — 줄 4개 · 답 2개
- `길이 200 → 출력 (4, 2) · 파라미터 1,474개` — 길이가 20배여도 그대로 (펼치기는 길이에 따라 커졌음)
- `손계산 : 32 × 10 + 32 × 32 + 32 + 32 + 32 × 2 + 2 = 1474` — 입력 가중치 + 요약 가중치 + bias 2개 + 분류기

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>hidden state</td><td>매 칸 새로 쓰는 요약 (창고 아님)</td><td>RNN이 넘기는 것</td></tr>
<tr><td>파라미터 공유 (RNN)</td><td>같은 W를 순간마다 다시 씀</td><td>시간을 훑을 때</td></tr>
<tr><td>파라미터 공유 (CNN)</td><td>같은 필터를 위치마다 다시 씀</td><td>공간을 훑을 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - hidden state는 본 것이 쌓이는 창고가 아님 → 매 칸 새로 쓰고, 이전 요약의 흔적만 남음

> ✅ **한 줄 정리** · 새 요약 = 계산(이전 요약, 새 숫자)이고, 요약 크기를 고정해 같은 상자를 칸마다 다시 쓰므로 파라미터 수가 길이와 무관하다.

**딥러닝 코드에서는**

- 학습은 길이 10으로 하고 쓸 때는 길이 200도 넣을 수 있음
- `h_t = f(h_(t-1), x_t)` — 모든 RNN 계열의 공통 틀

---
## 3. 코드로 옮기기

> **이 장의 흐름**
>
> - 강의 3부 앞 · 손으로 짠 for 문 → PyTorch nn.RNN과 맞춰 봄
> - 거의 모두가 막히는 에러에서 세 겹 모양 · batch_first · output vs h_n으로

<img src="images/rl_shapes.png" alt="입력 세 겹 (개수, 길이, 표현)과 batch_first" width="719">

**파이프라인 위치** · ③ 모델 입력 · 출력 — shape 맞추기

### 3-1. `A` 상자 하나 + for 문 · `nn.RNNCell` · `for t in range(10)` <sub>강의 3부 · '상자 하나 만들기' · 'for 문으로 돌리기'</sub>

> 🎯 **`nn.RNNCell`과 for 문 세 줄이 어떻게 그림 속 RNN과 같은가?**

💡 **쉽게 말하면**

- `cell` 하나 = **상자 하나**, for 열 번 = **시간을 펼친 것**, `h =` = **덮어쓰기**.
- 상자 안 계산은 `tanh(입력 쪽 + 이전 요약 쪽 + bias)`.

**실행 흐름**

1. RNNCell(10, 32) 만들기
2. 처음 요약 0에서 10칸 반복
3. 같은 계산을 식으로 직접 해서 비교

**먼저 예측** · 다 돌고 난 `h`의 모양은? — 답은 코드 실행 결과 아래에

In [9]:
# 상자 하나(RNNCell) + for 문 세 줄 = 손으로 펼친 RNN
# 왜 · RNN이 하는 일을 for 문으로 직접 펼쳐 이해하기 위해
torch.manual_seed(SEED)             # 같은 시작점
cell = nn.RNNCell(10, 32)           # 10 = 칸 하나의 표현 크기 · 32 = 넘길 요약 크기
x = to_tensor(xtr[:4])              # 학습 4줄 → (4, 10, 10)
h = torch.zeros(4, 32)              # 처음 요약 = 0
for t in range(10):                 # 시간을 펼친 것
    h = cell(x[:, t, :], h)         # 덮어쓰기 = 매 순간 새 요약
print("h.shape :", tuple(h.shape))  # 4 = 줄 수 · 32 = 요약 · 길이 10은 사라짐


# 상자 안의 계산을 직접: h_t = tanh(x_t W_ih^T + b_ih + h_(t-1) W_hh^T + b_hh)
# 왜 · RNNCell 안의 수식을 직접 계산해 결과가 같은지 검증하기 위해
h1 = torch.zeros(4, 32)                                                    # 다시 처음부터
for t in range(10):                                                        # 같은 10칸
    pre = x[:, t] @ cell.weight_ih.T + cell.bias_ih                        # 입력 쪽
    h1 = torch.tanh(pre + h1 @ cell.weight_hh.T + cell.bias_hh)            # + 이전 요약 쪽 → tanh
print("식으로 계산한 h == cell 결과 :", torch.allclose(h, h1, atol=1e-6))  # 같은 계산
# (32, 10) · (32, 32)
print("가중치 모양 W_ih", tuple(cell.weight_ih.shape), "· W_hh", tuple(cell.weight_hh.shape))

h.shape : (4, 32)
식으로 계산한 h == cell 결과 : True
가중치 모양 W_ih (32, 10) · W_hh (32, 32)


**결과** · 예측 정답은 `(4, 32)` — 줄 4개 · 요약 32 · 길이 10은 사라짐

- `h.shape : (4, 32)` — 4 = 한 번에 처리한 줄 수 · 32 = 요약 크기
- `식으로 계산한 h == cell 결과 : True` — `h_t = tanh(x_t W_ih^T + b_ih + h_(t-1) W_hh^T + b_hh)`
- `가중치 모양 W_ih (32, 10) · W_hh (32, 32)` — 입력 10 → 요약 32 · 요약 32 → 요약 32

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>nn.RNNCell</code></td><td>한 칸만 · for 문은 직접</td><td>칸마다 값을 잡아야 할 때 (기울기 측정 5장)</td></tr>
<tr><td><code>nn.RNN</code></td><td>for 문이 안에 들어감</td><td>보통의 학습 (3-2)</td></tr>
<tr><td>식으로 직접 계산</td><td>가장 투명 · 느림</td><td>구조 이해 · 검증</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - for 문 밖에 `cell`이 하나뿐이라는 것 = 파라미터 공유
> - for 문 안에서 `cell`을 새로 만들면 칸마다 다른 모델

> ✅ **한 줄 정리** · `cell` 하나를 for 문으로 칸마다 다시 쓰고 `h`를 덮어쓰는 것이 RNN이며, 상자 안은 tanh 한 번이다.

**딥러닝 코드에서는**

- 어텐션 · 디코더처럼 칸마다 개입해야 하면 `RNNCell` · `LSTMCell` + for 문을 직접 씀

### 3-2. `A` PyTorch의 nn.RNN · `nn.RNN` · `output, h_n` <sub>강의 3부 · 'PyTorch 가 이미 만들어 둔 것'</sub>

> 🎯 **`nn.RNN`은 손으로 짠 for 문과 같은 값을 내고, 무엇을 두 개 돌려주는가?**

💡 **쉽게 말하면**

- for 문이 **안으로 들어간 판**.
- 돌려주는 것 두 개 · `output` = 매 칸 요약 전부 · `h_n` = 마지막 요약.

**실행 흐름**

1. 같은 가중치를 cell에 복사
2. 손 계산 · nn.RNN 둘 다 실행
3. 차이 · 모양 · 마지막 칸 비교

**먼저 예측** · 두 결과의 최대 차이는? — 답은 코드 실행 결과 아래에

In [10]:
# PyTorch가 감싸 둔 판 nn.RNN: for 문이 안으로 들어감
# 왜 · 직접 만든 반복과 nn.RNN 결과가 같은지 확인하기 위해
torch.manual_seed(SEED)                          # 같은 시작점
rnn = nn.RNN(10, 32, batch_first=True)           # 판 전체
cell = nn.RNNCell(10, 32)                        # 비교할 상자
with torch.no_grad():                            # 같은 가중치를 복사
    for a, b in [("weight_ih", "weight_ih_l0"), ("weight_hh", "weight_hh_l0"),  # 이름 짝 4개
                 ("bias_ih", "bias_ih_l0"), ("bias_hh", "bias_hh_l0")]:
        getattr(cell, a).copy_(getattr(rnn, b))  # rnn → cell
x = to_tensor(xtr[:4])                           # (4, 10, 10)
h, hs = torch.zeros(4, 32), []                   # 처음 요약 · 칸별 요약
for t in range(10):                              # 손으로 펼친 판
    h = cell(x[:, t, :], h)                      # 새 요약
    hs.append(h)                                 # 저장
output, h_n = rnn(x)                             # 감싼 판
print("(output - manual).abs().max() :", (output - torch.stack(hs, 1)).abs().max().item())  # 0.0
# 돌려주는 것 두 개
print("x", tuple(x.shape), "· output", tuple(output.shape), "· h_n", tuple(h_n.shape))
# 사실상 같음
print("torch.allclose(output[:, -1, :], h_n[0]) :", torch.allclose(output[:, -1, :], h_n[0]))

(output - manual).abs().max() : 0.0
x (4, 10, 10) · output (4, 10, 32) · h_n (1, 4, 32)
torch.allclose(output[:, -1, :], h_n[0]) : True


**결과** · 예측 정답은 `0.0` — 완전히 같은 계산

- `(output - manual).abs().max() : 0.0` — 같은 가중치 · 같은 입력 → 같은 값
- `x (4, 10, 10) · output (4, 10, 32) · h_n (1, 4, 32)` — output은 길이 10이 살아 있고 h_n은 없음
- `torch.allclose(output[:, -1, :], h_n[0]) : True` — 마지막 칸 요약 = 마지막 요약 (지금 조건에서)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>output</code></td><td>매 칸 요약 <code>(N, L, H)</code></td><td>칸마다 답이 필요할 때</td></tr>
<tr><td><code>h_n</code></td><td>마지막 요약 <code>(층 × 방향, N, H)</code></td><td>끝에 한 번 답할 때</td></tr>
<tr><td><code>output[:, -1, :]</code></td><td>마지막 칸 요약 <code>(N, H)</code></td><td>1층 · 한 방향 · 빈 칸 없음일 때만 h_n과 같음</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 가중치 이름이 다름 · cell `weight_ih` ↔ RNN `weight_ih_l0` (층 번호가 붙음)

> ✅ **한 줄 정리** · `nn.RNN`은 손으로 편 for 문과 같은 값을 내고, 매 칸 요약 `output`과 마지막 요약 `h_n`을 함께 돌려준다.

**딥러닝 코드에서는**

- `output, h_n = self.rnn(x)` → 분류는 `self.fc(output[:, -1])` (RNN-1 · RNN-3)

### 3-3. `A` 세 겹 모양과 one_hot · `F.one_hot()` · `unsqueeze()` <sub>강의 3부 · '코드로 옮기면' · '세 겹이 각각 무엇인가'</sub>

> 🎯 **`Expected 10, got 1` 에러는 왜 나고, 어떻게 고치는가?**

💡 **쉽게 말하면**

- 시퀀스 입력은 **세 겹** · 시퀀스 개수 · 길이 · 표현 크기.
- 숫자 하나를 열 칸에 적어야 표현 크기가 10이 됨 (축을 붙이는 게 아님).

**실행 흐름**

1. 숫자 그대로 축만 붙여 넣기 → 에러
2. one_hot으로 열 칸에 적기
3. 숫자 3의 열 칸 보기

**먼저 예측** · `digits.unsqueeze(-1)`의 모양은? — 답은 코드 실행 결과 아래에

In [11]:
# 거의 모두가 한 번 막히는 에러: 칸마다 숫자 1개를 넣음
# 왜 · 칸마다 숫자 하나만 넣을 때의 모양 에러를 미리 보기 위해
digits = torch.tensor([xtr[0]])                             # (1, 10) · 숫자 그대로
rnn = nn.RNN(10, 32, batch_first=True)                      # 칸마다 10개를 기대
x_bad = digits.unsqueeze(-1).float()                        # 축만 붙임 → (1, 10, 1)
try:
    rnn(x_bad)                                              # 마지막 겹이 1
except RuntimeError as e:                                   # 에러를 잡아 이어서 실행
    print(tuple(x_bad.shape), "→", str(e).splitlines()[0])  # 에러 첫 줄


# 고치는 법: 축을 붙이는 게 아니라 숫자를 열 칸으로 적음
# 왜 · 범주 숫자는 원-핫으로 바꿔야 한다는 것을 확인하기 위해
x = F.one_hot(digits, num_classes=10).float()                   # (1, 10) → (1, 10, 10)
output, h_n = rnn(x)                                            # 통과
print(tuple(x.shape), "→ output", tuple(output.shape), "· h_n", tuple(h_n.shape))  # 세 겹
print("숫자", xtr[0][1], "의 열 칸 :", x[0, 1].int().tolist())  # 그 자리만 1

(1, 10, 1) → input.size(-1) must be equal to input_size. Expected 10, got 1
(1, 10, 10) → output (1, 10, 32) · h_n (1, 1, 32)
숫자 3 의 열 칸 : [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]


**결과** · 예측 정답은 `(1, 10, 1)` — 세 겹은 맞지만 마지막 겹이 1

- `(1, 10, 1) → input.size(-1) must be equal to input_size. Expected 10, got 1` — 칸마다 1개를 넣었는데 10개를 기대
- `(1, 10, 10) → output (1, 10, 32) · h_n (1, 1, 32)` — one_hot으로 열 칸에 적으면 통과
- `숫자 3 의 열 칸 : [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]` — 그 자리만 1

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>unsqueeze(-1)</code></td><td>크기 1인 축 추가</td><td>값 하나가 진짜 특성 1개일 때 (시계열 값)</td></tr>
<tr><td><code>F.one_hot(x, 10)</code></td><td>번호 → 열 칸 표시</td><td>번호가 범주일 때 (숫자 · 토큰)</td></tr>
<tr><td><code>nn.Embedding</code></td><td>번호 → 학습되는 벡터</td><td>범주가 많을 때 (RNN-3)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 막히면 먼저 모양을 찍어 볼 것
> - MLP 입력은 두 겹이었지만 시퀀스는 세 겹

> ✅ **한 줄 정리** · `nn.RNN(입력 크기, …)`의 입력 크기는 마지막 겹 길이와 같아야 하며, 번호 입력은 one_hot으로 열 칸에 적는다.

**딥러닝 코드에서는**

- `x.shape` → `(배치, 길이, 특성)`을 먼저 출력하는 습관
- 텍스트는 one_hot 대신 `nn.Embedding` (RNN-3)

### 3-4. `A` batch_first 함정 · `batch_first=True` · `transpose(0, 1)` <sub>강의 3부 · 'batch_first 가 정하는 것' · '껐을 때가 더 무서움'</sub>

> 🎯 **`batch_first`를 끄면 같은 텐서가 왜 에러 없이 다르게 읽히는가?**

💡 **쉽게 말하면**

- `batch_first`는 **앞 두 겹의 순서**만 정함 · 켜면 (개수, 길이, 표현) · 끄면 (길이, 개수, 표현).
- 기본값은 **꺼짐**.

**실행 흐름**

1. 같은 (8, 10, 10)을 켜고 끄고
2. h_n 모양 비교
3. transpose로 바꿔 넣기

**먼저 예측** · 끈 채로 넣으면 h_n의 가운데 숫자는? — 답은 코드 실행 결과 아래에

In [12]:
# batch_first: 앞 두 겹(시퀀스 개수 · 길이)의 순서만 바꿈 · 기본값은 False
# 왜 · 축 순서를 잘못 두면 에러 없이 틀린 결과가 나오는 함정을 보이기 위해
x = torch.randn(8, 10, 10)                           # 8줄 · 10칸 · 칸마다 10개
for bf in (True, False):                             # 켜기 · 끄기
    output, h_n = nn.RNN(10, 32, batch_first=bf)(x)  # 둘 다 에러 없음
    # 8 ↔ 10
    print(f"batch_first={str(bf):<5} → output {tuple(output.shape)} · h_n {tuple(h_n.shape)}")
x_tf = x.transpose(0, 1)  # (8, 10, 10) → (10, 8, 10) = 길이 · 개수 · 표현
output, h_n = nn.RNN(10, 32)(x_tf)                   # False에 맞게 바꿔 넣으면
print("transpose 후 False → h_n", tuple(h_n.shape))  # 다시 8줄

batch_first=True  → output (8, 10, 32) · h_n (1, 8, 32)
batch_first=False → output (8, 10, 32) · h_n (1, 10, 32)
transpose 후 False → h_n (1, 8, 32)


**결과** · 예측 정답은 10 — 10을 줄 수로 읽음

- `batch_first=True  → output (8, 10, 32) · h_n (1, 8, 32)` — 8줄 · 10칸으로 읽음 (의도대로)
- `batch_first=False → output (8, 10, 32) · h_n (1, 10, 32)` — 8을 길이, 10을 줄 수로 바꿔 읽음 · 에러 없음
- `transpose 후 False → h_n (1, 8, 32)` — False에 맞게 축을 바꾸면 다시 8줄

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>batch_first=True</code></td><td>(N, L, 입력)</td><td>DataLoader 배치를 바로 넣을 때 (권장)</td></tr>
<tr><td><code>batch_first=False</code> (기본)</td><td>(L, N, 입력)</td><td>옛 코드 · 문서 예제</td></tr>
<tr><td><code>x.transpose(0, 1)</code></td><td>앞 두 축 맞바꿈</td><td>설정과 데이터 순서가 다를 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - output 모양이 같게 나와 눈치채기 어려움 → h_n의 가운데 숫자로 확인

> ✅ **한 줄 정리** · `batch_first`는 앞 두 겹의 순서만 바꾸며, 끈 채로 (개수, 길이, …)를 넣으면 에러 없이 틀리게 읽는다.

**딥러닝 코드에서는**

- `nn.RNN(..., batch_first=True)`를 기본으로 쓰고 h_n의 배치 축으로 확인

### 3-5. `B` output과 h_n은 정말 같은가 · `torch.equal()` · `torch.allclose()` <sub>강의 3부 · '정말 같은 값일까' · 'h_n 앞에 붙은 1'</sub>

> 🎯 **`output[:, -1]`과 `h_n[0]`은 언제 같고, `h_n` 앞의 1은 무엇인가?**

💡 **쉽게 말하면**

- **층 1 · 한 방향 · 빈 칸 없음**이면 비트까지 같음.
- `h_n` 첫 축 = **층 수 × 방향 수** · 층이 늘거나 양방향이면 달라짐.

<img src="images/rl_out_hn.png" alt="output은 매 칸 요약 전부, h_n은 마지막 칸 요약" width="733">

**실행 흐름**

1. equal · allclose 비교
2. 층 · 방향 조합 4가지 모양
3. 칸마다 답 · 줄마다 답

**먼저 예측** · 2층 양방향이면 h_n의 첫 축은? — 답은 코드 실행 결과 아래에

In [13]:
# output = 칸마다 요약 전부 · h_n = 마지막 요약 · 정말 같은 값일까
# 왜 · 두 출력이 언제 같은지 직접 비교하기 위해
torch.manual_seed(SEED)                                                  # 같은 시작점
x = to_tensor(xtr[:8])                                                   # (8, 10, 10)
output, h_n = nn.RNN(10, 32, batch_first=True)(x)                        # 1층 · 한 방향
print("equal :", torch.equal(output[:, -1, :], h_n[0]), "· allclose :",  # 비트까지 · 오차 허용
      torch.allclose(output[:, -1, :], h_n[0]))


# 조건이 깨지면: 층 여러 개 · 양방향 → h_n 첫 축 = 층 수 × 방향 수
# 왜 · 층 · 방향이 늘면 h_n 모양이 달라진다는 것을 확인하기 위해
for layers, bi in [(1, False), (2, False), (1, True), (2, True)]:  # 조합 4가지
    out, h_n = nn.RNN(10, 32, num_layers=layers, bidirectional=bi, batch_first=True)(x)  # 실행
    # 모양
    print(f"층 {layers} · 양방향 {str(bi):<5} → output {tuple(out.shape)} · h_n {tuple(h_n.shape)}")


# 칸마다 답 → output 전체 · 끝에 한 번 → 마지막 요약
# 왜 · 과제에 따라 어느 출력을 쓸지 고르는 기준을 세우기 위해
head = nn.Linear(32, 2)  # 요약 → 점수
# 두 경우
print("칸마다 답", tuple(head(output).shape), "· 줄마다 답", tuple(head(output[:, -1]).shape))

equal : True · allclose : True
층 1 · 양방향 False → output (8, 10, 32) · h_n (1, 8, 32)
층 2 · 양방향 False → output (8, 10, 32) · h_n (2, 8, 32)
층 1 · 양방향 True  → output (8, 10, 64) · h_n (2, 8, 32)
층 2 · 양방향 True  → output (8, 10, 64) · h_n (4, 8, 32)
칸마다 답 (8, 10, 2) · 줄마다 답 (8, 2)


**결과** · 예측 정답은 4 — 층 2 × 방향 2

- `equal : True · allclose : True` — 1층 · 한 방향에서는 비트까지 같음
- `층 2 · 양방향 False → output (8, 10, 32) · h_n (2, 8, 32)` — h_n은 층마다 하나
- `층 1 · 양방향 True  → output (8, 10, 64) · h_n (2, 8, 32)` — output은 두 방향을 이어 붙여 64
- `층 2 · 양방향 True  → output (8, 10, 64) · h_n (4, 8, 32)` — 2 × 2 = 4
- `칸마다 답 (8, 10, 2) · 줄마다 답 (8, 2)` — 라벨이 칸마다면 output 전체 · 줄마다면 마지막 요약

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>torch.equal</code></td><td>비트까지 똑같은가</td><td>정말 같은 계산인지</td></tr>
<tr><td><code>torch.allclose</code></td><td>아주 작은 오차는 봐줌</td><td>다른 경로로 계산한 값 비교</td></tr>
<tr><td><code>h_n[-1]</code></td><td>마지막 층의 끝 요약</td><td>여러 층일 때 분류</td></tr>
<tr><td><code>output</code> 전체</td><td>칸마다 요약</td><td>칸마다 라벨 (품사 · 프레임별)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 양방향이면 뒤 방향의 끝은 0번 칸 → `output[:, -1]`에는 뒤 방향이 1칸만 읽은 값
> - 빈 칸이 있으면 둘 다 빈 칸을 지난 값 (4장)

> ✅ **한 줄 정리** · 1층 · 한 방향 · 빈 칸 없음일 때만 `output[:, -1]`과 `h_n[0]`이 같고, `h_n` 첫 축은 층 수 × 방향 수다.

**딥러닝 코드에서는**

- LSTM은 `output, (h_n, c_n)`으로 하나가 더 늘어남 (6장)

### 3-6. `B` UCI HAR 모양 읽기 · `[개수, 길이, 채널]` <sub>강의 3부 · 'UCI HAR' · '9 가 어디서 나왔는가'</sub>

> 🎯 **`[7352, 128, 9]`의 세 숫자는 각각 무엇이고, 왜 패딩이 필요 없는가?**

💡 **쉽게 말하면**

- 허리에 찬 폰 센서 기록 · 7352개 · 128칸 · 칸마다 9개.
- 9 = 신호 3종 × 축 3개 · 128칸 = 2.56초씩 잘라서 모두 같은 길이.

**실행 흐름**

1. 채널 이름 만들기
2. 128칸의 시간 계산
3. 숫자 개수 · 용량

**먼저 예측** · 128칸은 몇 초일까? (50Hz) — 답은 코드 실행 결과 아래에

In [14]:
# UCI HAR 모양 읽기: [개수, 길이, 칸마다 값] = [7352, 128, 9] (데이터는 파일별 2에서 내려받음)
# 왜 · 실제 센서 데이터의 모양을 같은 규칙으로 읽기 위해
signals = ["가속도", "자이로", "전체 가속도"]              # 신호 3종
axes = ["x", "y", "z"]                                     # 축 3개
channels = [f"{s}_{a}" for s in signals for a in axes]     # 한 순간에 동시에 읽는 값
print("채널 수 :", len(channels), "→", channels[:3], "…")  # 3 × 3 = 9
print("한 줄 길이 : 128칸 ÷ 50Hz =", 128 / 50, "초")       # 2.56초씩 자른 창
X_shape = (7352, 128, 9)                                   # 학습 데이터 모양 (강의 화면)
n_num = 7352 * 128 * 9                                     # 숫자 개수
print("숫자 개수 :", f"{n_num:,}", "· float32 용량", round(n_num * 4 / 2**20, 1), "MB")  # 크기 감
print("모두 128칸 → 직사각형 텐서", tuple(torch.empty(X_shape).shape))  # 패딩 필요 없음

채널 수 : 9 → ['가속도_x', '가속도_y', '가속도_z'] …
한 줄 길이 : 128칸 ÷ 50Hz = 2.56 초
숫자 개수 : 8,469,504 · float32 용량 32.3 MB
모두 128칸 → 직사각형 텐서 (7352, 128, 9)


**결과** · 예측 정답은 2.56초

- `채널 수 : 9 → ['가속도_x', '가속도_y', '가속도_z'] …` — 신호 3종 × 축 3개 = 9
- `한 줄 길이 : 128칸 ÷ 50Hz = 2.56 초` — 같은 길이로 미리 자른 창
- `숫자 개수 : 8,469,504 · float32 용량 32.3 MB` — 7352 × 128 × 9개
- `모두 128칸 → 직사각형 텐서 (7352, 128, 9)` — 길이가 같아 패딩 없이 바로 텐서

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>HAR (128칸 고정)</td><td>이미 잘라 놓은 데이터</td><td>패딩 없이 바로</td></tr>
<tr><td>우리 3 · 7 과제</td><td>길이를 우리가 정함 · 바꿀 수 있음</td><td>길이 실험 (2 · 5장)</td></tr>
<tr><td>문장 · 클릭 기록</td><td>길이가 제각각</td><td>패딩 + lens (4장)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 진짜 데이터가 늘 HAR처럼 같은 길이로 잘려 오지는 않음

> ✅ **한 줄 정리** · HAR 한 줄은 (128칸, 9채널)이고 2.56초씩 잘라 길이가 같으므로 패딩 없이 `[7352, 128, 9]` 텐서가 된다.

**딥러닝 코드에서는**

- 실제 내려받기 · RNN · LSTM · GRU 비교는 파일별 2의 3 · 8장

---
## 4. 길이가 다른 줄

> **이 장의 흐름**
>
> - 강의 3부 뒤 · 1장에서 남겨 둔 두 번째 갈래 '길이도 매번 다른가'
> - 억지로 맞추고(padding) · 맞춘 자리를 빼고 읽음(lens · mask)

<img src="images/rl_pad3.png" alt="틀을 잡는 세 가지 방법 · 맞춤 · 자름 · 배치 안에서만" width="733">

**파이프라인 위치** · ③ 전처리 — 패딩 · 길이표

### 4-1. `A` padding과 lens · `torch.zeros()` · `lens` <sub>강의 3부 · '그대로 텐서로 만들면' · '손으로 채워 보면'</sub>

> 🎯 **길이가 다른 줄은 왜 텐서가 안 되고, 채운 뒤 무엇을 같이 들고 다녀야 하는가?**

💡 **쉽게 말하면**

- 배열은 **직사각형**이라 가장 긴 줄(6칸) 틀에 억지로 맞춤 = **padding**.
- 어디까지가 진짜인지는 텐서 안에 없어서 **길이표 lens**를 따로 들고 다님.

**실행 흐름**

1. 그대로 텐서로 → 에러
2. 6칸 틀 · 빈 벡터로 채우기
3. lens · 빈 칸 수

**먼저 예측** · 세 줄 중 빈 칸이 가장 많은 줄은 몇 칸이 빌까? — 답은 코드 실행 결과 아래에

In [15]:
# 길이가 3 · 6 · 4로 제각각인 세 줄을 그대로 텐서로
# 왜 · 길이가 다른 줄은 바로 텐서로 묶을 수 없다는 것을 확인하기 위해
seqs = [[3, 1, 7],                      # 길이 3
        [9, 7, 0, 5, 3, 2],             # 길이 6
        [1, 3, 4, 7]]                   # 길이 4
try:
    torch.tensor(seqs)                  # 배열은 직사각형이어야 함
except ValueError as e:                 # 에러를 잡아 이어서 실행
    print("그대로 텐서로 만들면 :", e)  # 에러 내용


# padding: 가장 긴 6칸 틀에 맞추고 진짜 길이표 lens를 같이 들고 다님
# 왜 · 빈 칸을 채우되 진짜 길이를 함께 들고 다녀 나중에 구분하기 위해
ML = max(len(s) for s in seqs)               # 6
X = torch.zeros(len(seqs), ML, 10)           # (3, 6, 10) 빈 틀
for b, s in enumerate(seqs):                 # 줄마다
    for t, d in enumerate(s):                # 진짜 칸만
        X[b, t, d] = 1.0                     # 숫자 d 자리에 1
lens = torch.tensor([len(s) for s in seqs])  # 어디까지가 진짜인지
# 5칸이 빔
print("X.shape :", tuple(X.shape), "· lens :", lens.tolist(), "· 빈 칸 수 :", (ML - lens).tolist())

그대로 텐서로 만들면 : expected sequence of length 3 at dim 1 (got 6)
X.shape : (3, 6, 10) · lens : [3, 6, 4] · 빈 칸 수 : [3, 0, 2]


**결과** · 예측 정답은 3칸 — 길이 3인 첫 줄

- `그대로 텐서로 만들면 : expected sequence of length 3 at dim 1 (got 6)` — 첫 줄 길이 3, 둘째 줄 6 → 직사각형이 아님
- `X.shape : (3, 6, 10) · lens : [3, 6, 4] · 빈 칸 수 : [3, 0, 2]` — 빈 칸 5개는 데이터가 아님

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>가장 긴 것에 맞춤</td><td>빈 칸이 생김</td><td>기본 (이 섹션)</td></tr>
<tr><td>넘치면 자름</td><td>뒷부분을 버림</td><td>길이 상한이 있을 때</td></tr>
<tr><td>배치 안에서만 맞춤</td><td>빈 칸이 줄어듦</td><td><code>pad_sequence</code> · 길이별 묶기 (RNN-3)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - lens를 버리면 진짜 0과 채운 0을 다시 가를 수 없음

> ✅ **한 줄 정리** · 길이가 다른 줄은 가장 긴 틀에 채워 한 텐서로 만들고, 진짜 길이 lens를 반드시 같이 들고 다닌다.

**딥러닝 코드에서는**

- `pad_sequence(seqs, batch_first=True)` + `lengths` (RNN-3)

### 4-2. `B` 빈 벡터 vs 숫자 0 · `X.clone()` <sub>강의 3부 · '무엇으로 채웠는지가 갈림'</sub>

> 🎯 **빈 칸을 '숫자 0'으로 채우면 무엇이 문제인가?**

💡 **쉽게 말하면**

- 숫자 0도 진짜 숫자라 one-hot이면 **첫 칸이 1**.
- 그러면 채운 칸과 진짜 0 칸이 **똑같아짐**.

**실행 흐름**

1. 빈 벡터 채움 · 칸별 합
2. 숫자 0 채움 · 칸별 합
3. 채운 칸과 진짜 0 칸 비교

**먼저 예측** · 숫자 0으로 채운 칸과 둘째 줄의 진짜 0 칸은 같을까? — 답은 코드 실행 결과 아래에

In [16]:
# 무엇으로 채웠는지가 갈림: 빈 벡터(열 칸 전부 0) vs 숫자 0(첫 칸만 1)
# 왜 · 채우는 값이 모델 입력의 의미를 바꿀 수 있다는 것을 비교하기 위해
X0 = X.clone()                                                            # 복사본
X0[0, 3:, 0] = 1.0                                                        # 첫 줄 빈 칸을 숫자 0으로
print("빈 벡터로 채움 · 첫 줄 칸별 합   :", X[0].sum(1).int().tolist())   # [1, 1, 1, 0, 0, 0]
print("숫자 0 으로 채움 · 첫 줄 칸별 합 :", X0[0].sum(1).int().tolist())  # 전부 1
print("채운 칸 == 둘째 줄의 진짜 0 칸 :", torch.equal(X0[0, 3], X[1, 2]))  # 구분 불가

빈 벡터로 채움 · 첫 줄 칸별 합   : [1, 1, 1, 0, 0, 0]
숫자 0 으로 채움 · 첫 줄 칸별 합 : [1, 1, 1, 1, 1, 1]
채운 칸 == 둘째 줄의 진짜 0 칸 : True


**결과** · 예측 정답은 같음 — 구분 불가

- `빈 벡터로 채움 · 첫 줄 칸별 합   : [1, 1, 1, 0, 0, 0]` — 빈 칸은 열 칸 모두 0 → 구분됨
- `숫자 0 으로 채움 · 첫 줄 칸별 합 : [1, 1, 1, 1, 1, 1]` — 빈 칸도 1 → 진짜 숫자처럼 보임
- `채운 칸 == 둘째 줄의 진짜 0 칸 : True` — 모델은 채운 칸을 숫자 0으로 읽음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>빈 벡터 (열 칸 모두 0)</td><td>채운 칸이 표시됨</td><td>one-hot 입력 채우기</td></tr>
<tr><td>숫자 0 (첫 칸만 1)</td><td>진짜 0과 같아짐</td><td>쓰지 않음</td></tr>
<tr><td><code>&lt;pad&gt;</code> 번호 0 + <code>padding_idx</code></td><td>사전에서 0번을 비워 둠</td><td>텍스트 (RNN-3)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 빈 벡터로 채워도 RNN은 빈 칸을 세 번 더 지남 → 읽는 문제가 남음 (4-3)

> ✅ **한 줄 정리** · 빈 칸은 진짜 값과 겹치지 않는 빈 벡터로 채워야 하며, 그래도 RNN은 빈 칸을 지나가므로 읽는 법을 따로 고쳐야 한다.

**딥러닝 코드에서는**

- `nn.Embedding(V, D, padding_idx=0)` — `<pad>` 벡터를 0으로 고정

### 4-3. `A` -1 대신 lens - 1 · mask 평균 · `output[idx, lens - 1]` · `mask` <sub>강의 3부 · '마지막 칸이 무엇인가' · '고치는 줄'</sub>

> 🎯 **`output[:, -1]`은 왜 틀리고, 진짜 끝 · 진짜 칸 평균은 어떻게 꺼내는가?**

💡 **쉽게 말하면**

- `-1`은 **틀의 끝**이지 데이터의 끝이 아님.
- 줄마다 자기 길이의 끝 `lens - 1`을 꺼내고, 평균은 **진짜 칸만 1인 mask**로 냄.

<img src="images/rl_padding.png" alt="가장 긴 6칸 틀에서 -1과 lens - 1이 가리키는 자리" width="686">

**실행 흐름**

1. -1 과 lens - 1 비교
2. mask 만들기 · 진짜 칸 평균
3. 길이 범위별 빈 칸 최대 수

**먼저 예측** · 세 줄 중 -1이 진짜 끝인 줄은? — 답은 코드 실행 결과 아래에

In [17]:
# 흔히 쓰는 output[:, -1]은 틀의 끝 · 진짜 끝은 output[idx, lens - 1]
# 왜 · padding이 있으면 마지막 칸이 진짜 끝이 아니라는 함정을 고치기 위해
torch.manual_seed(SEED)                          # 같은 시작점
output, _ = nn.RNN(10, 32, batch_first=True)(X)  # 채운 세 줄
idx = torch.arange(len(X))                       # 줄 번호
# 길이 6만
print("-1 == lens - 1 :", [torch.equal(a, b) for a, b in zip(output[:, -1], output[idx, lens - 1])])


# 평균에서 빈 칸 빼기: mask
# 왜 · 빈 칸이 평균을 망치지 않게 mask로 빼기 위해
T = X.shape[1]                                                        # 6
m = (torch.arange(T)[None, :] < lens[:, None]).float().unsqueeze(-1)  # (3, 6, 1) · 진짜 칸 = 1
h = (output * m).sum(1) / m.sum(1)                                    # (3, 32) 진짜 칸 평균
print("m 첫 줄 :", m[0, :, 0].int().tolist(), "· h", tuple(h.shape))  # 1 1 1 0 0 0
# 빈 칸 제외 확인
print("첫 줄 mask 평균 == 앞 3칸 평균 :", torch.allclose(h[0], output[0, :3].mean(0)))


# 편차가 작으면 버그가 안 드러남: 줄마다 생기는 빈 칸 수
# 왜 · 길이 차이가 작으면 버그가 숨는다는 것을 확인하기 위해
for lo, hi in [(4, 12), (5, 60)]:                      # 강의의 두 설정
    print(f"길이 {lo}~{hi} → 빈 칸 최대 {hi - lo}칸")  # 8칸 vs 55칸

-1 == lens - 1 : [False, True, False]
m 첫 줄 : [1, 1, 1, 0, 0, 0] · h (3, 32)
첫 줄 mask 평균 == 앞 3칸 평균 : True
길이 4~12 → 빈 칸 최대 8칸
길이 5~60 → 빈 칸 최대 55칸


**결과** · 예측 정답은 길이 6인 둘째 줄 하나

- `-1 == lens - 1 : [False, True, False]` — 가장 긴 줄만 맞음
- `m 첫 줄 : [1, 1, 1, 0, 0, 0] · h (3, 32)` — 진짜 칸만 1 → 평균은 (3, 32)
- `첫 줄 mask 평균 == 앞 3칸 평균 : True` — 빈 칸을 뺀 평균
- `길이 5~60 → 빈 칸 최대 55칸` — 편차가 크면 버그가 드러남 (강의 -1 → 51.4%)
- `길이 4~12 → 빈 칸 최대 8칸` — 편차가 작으면 네 방법 모두 만점 (버그가 숨음)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>output[:, -1]</code></td><td>틀의 끝 (빈 칸을 지난 값)</td><td>빈 칸이 없을 때만</td></tr>
<tr><td><code>output[idx, lens - 1]</code></td><td>줄마다 진짜 끝</td><td>패딩한 배치 분류</td></tr>
<tr><td><code>output.mean(1)</code></td><td>빈 칸까지 평균 (강의 0.483)</td><td>쓰지 않음</td></tr>
<tr><td>mask 평균</td><td>진짜 칸만 평균 (강의 1.000)</td><td>평균 요약이 필요할 때</td></tr>
<tr><td><code>pack_padded_sequence</code></td><td>빈 칸을 아예 안 지남</td><td>PyTorch 표준 (RNN-3)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 짧은 길이로만 시험하면 통과 → 길이 편차가 큰 데이터로 꼭 확인

> ✅ **한 줄 정리** · 패딩한 배치에서는 `output[:, -1]` 대신 `output[idx, lens - 1]`로 진짜 끝을 꺼내고, 평균은 mask로 빈 칸을 뺀다.

**딥러닝 코드에서는**

- `h_n[-1]` + `pack_padded_sequence(x, lengths)`가 실무 표준
- Transformer도 같은 이유로 padding mask를 씀

---
## 5. 넘긴 것이 끝까지 닿는가

> **이 장의 흐름**
>
> - 강의 4부 · 거리 98에서 20 epoch는 실패, 60 epoch는 성공 → 두 설명을 나눠 잼
> - 앞 → 뒤(기억)와 뒤 → 앞(기울기)을 따로 재고 원인을 반복 곱으로 설명

<img src="images/rl_distance.png" alt="거리별 정확도 · 78과 98 사이의 절벽 (강의 숫자)" width="814">

**파이프라인 위치** · ⑤ 학습 — 긴 거리에서 학습이 안 되는 원인 진단

### 5-1. `B` 앞 → 뒤: 귓속말 전달 · `norm()` · `finfo.eps` <sub>강의 4부 · '귓속말 전달 게임' · '학습 전 모델부터'</sub>

> 🎯 **맨 앞 두 칸만 다른 두 줄의 요약 차이는 몇 칸까지 남는가?**

💡 **쉽게 말하면**

- 귓속말이 스무 명을 지나면 말이 달라지듯, **한 칸 갈 때마다 차이가 절반 남짓**만 남음.
- float32가 구분할 수 있는 바닥보다 작아지면 잴 수도 없음.

<img src="images/rl_whisper.png" alt="맨 앞 두 칸만 바꾼 쌍 · 칸마다 남은 차이 (강의 숫자)" width="881">

**실행 흐름**

1. 맨 앞 두 칸만 맞바꾼 512쌍 만들기
2. 칸마다 두 요약의 상대 차이
3. float32 바닥과 비교

**먼저 예측** · 학습 전 모델에서 10칸 뒤에 남는 차이는? — 답은 코드 실행 결과 아래에

In [18]:
# 귓속말 전달: 맨 앞 두 칸만 바꾼 두 줄의 요약 차이가 칸마다 얼마나 남나 (학습 전 · CPU)
# 왜 · 앞의 작은 차이가 뒤로 갈수록 희미해지는 것을 숫자로 보기 위해
torch.manual_seed(SEED)                     # 같은 시작점
untrained = Net("RNN")                      # 학습 전 모델
xs, _ = make_front_task(512, 100, seed=31)  # 512줄 · 길이 100
xs_b = [[s[1], s[0]] + s[2:] for s in xs]   # 맨 앞 두 칸만 맞바꿈
with torch.no_grad():                       # 측정만
    oa, _ = untrained.r(to_tensor(xs))      # 원래 줄의 칸별 요약
    ob, _ = untrained.r(to_tensor(xs_b))    # 바꾼 줄의 칸별 요약
for t in (1, 2, 5, 10, 20, 40, 99):         # 칸마다
    d = ((oa[:, t] - ob[:, t]).norm(dim=1) / oa[:, t].norm(dim=1)).mean().item()  # 차이 ÷ 크기
    print(f"{t:>2}칸  {d:.3g}")             # 상대 차이
# 측정의 바닥
print("float32 가 구분하는 가장 작은 상대 차이 :", f"{torch.finfo(torch.float32).eps:.3g}")

 1칸  0.736


 2칸  0.378
 5칸  0.061
10칸  0.00255
20칸  7.67e-06
40칸  7.82e-08
99칸  7.94e-08
float32 가 구분하는 가장 작은 상대 차이 : 1.19e-07


**결과** · 예측 정답은 약 0.0026 — 처음(0.736)의 290분의 1

- ` 1칸  0.736` — 강의와 같은 값 (학습 전 · CPU)
- `10칸  0.00255` — 열 칸이면 처음의 수백분의 1
- `40칸  7.82e-08` — 40칸부터 바닥 근처 → 두 요약을 구분할 수 없음
- `float32 가 구분하는 가장 작은 상대 차이 : 1.19e-07` — 이보다 작으면 0에 가까운 게 아니라 잴 수 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>앞 → 뒤 (이 섹션)</td><td>맨 앞을 바꾸면 끝 요약이 바뀌나</td><td>기억이 닿는지</td></tr>
<tr><td>뒤 → 앞 (5-3)</td><td>끝 손실의 신호가 맨 앞에 오나</td><td>학습이 닿는지</td></tr>
<tr><td>정확도 (강의 L2)</td><td>둘을 섞은 결과</td><td>둘을 못 가름</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 강의 숫자 · 학습한 모델은 다름 · 51.7% 모델은 99칸에서 바닥 · 100% 모델은 99칸에서도 1.68
> - 40 · 99칸 값은 강의(9.9e-08 · 9.98e-08)와 끝자리가 다름 · 바닥 근처 값이라 버전 · 환경 차이가 드러남

> ✅ **한 줄 정리** · 학습 전 RNN은 맨 앞 차이가 칸마다 절반 남짓으로 줄어 40칸 근처에서 float32 바닥에 닿으며, 닿게 만드는 것이 학습이다.

**딥러닝 코드에서는**

- 모델이 앞 내용을 기억하는지 볼 때 '앞만 바꾼 쌍'으로 끝 출력 차이를 재는 방법이 유용

### 5-2. `B` 학습 없이 손으로 심은 가중치 · `torch.eye()` <sub>강의 4부 · '학습 없이 손으로 심어봐도'</sub>

> 🎯 **RNN이 398칸 뒤까지 기억을 나르는 것이 구조적으로 가능한가?**

💡 **쉽게 말하면**

- 이전 요약을 **그대로 넘기는** 가중치(단위 행렬)를 직접 심음.
- 3이면 +0.5 · 7이면 -0.5를 한 칸에 적으면 끝에서 부호만 보면 됨.

**실행 흐름**

1. 가중치 전부 0 · weight_hh = 단위 행렬
2. 3 · 7 자리에 ±0.5
3. 거리 8 · 98 · 398 평가

**먼저 예측** · 거리 398에서도 맞힐까? — 답은 코드 실행 결과 아래에

In [19]:
# 학습 없이 손으로 심은 가중치: 요약을 그대로 넘기고 3이면 +0.5 · 7이면 -0.5
# 왜 · 가중치를 잘 고르면 RNN도 멀리 기억할 수 있다는 것을 보이기 위해
def handmade(L, scale=0.5):                     # 길이 L → 정확도 · 끝에 남은 크기
    cell = nn.RNNCell(10, 8)                    # 요약 8칸
    with torch.no_grad():                       # 가중치를 직접 적음
        for p in cell.parameters():             # 전부 0으로 시작
            p.zero_()                           # 0
        cell.weight_hh.copy_(torch.eye(8))      # 이전 요약을 그대로 넘김
        cell.weight_ih[0, 3], cell.weight_ih[0, 7] = scale, -scale  # 3 → +0.5 · 7 → -0.5
    xs, ys = make_front_task(1000, L, seed=2)   # 평가 1000줄
    X, h = to_tensor(xs), torch.zeros(1000, 8)  # 입력 · 처음 요약
    with torch.no_grad():                       # 측정만
        for t in range(L):                      # L칸
            h = cell(X[:, t, :], h)             # 같은 상자 반복
    acc = ((h[:, 0] < 0).long() == torch.tensor(ys)).float().mean().item()  # 7이 나중이면 음수 → 네
    return acc, h[:, 0].abs().mean().item()     # 정확도 · 첫 칸 크기


# 왜 · 길이가 늘어도 손으로 심은 가중치가 맞히는지 확인하기 위해
for L in (10, 100, 400):                                               # 거리 8 · 98 · 398
    acc, size = handmade(L)                                            # 손 가중치로
    print(f"거리 {L - 2:>3} → {acc:.3f} · 끝에 남은 크기 {size:.4f}")  # 기억은 닿을 수 있음

거리   8 → 1.000 · 끝에 남은 크기 0.0377


거리  98 → 1.000 · 끝에 남은 크기 0.0362

거리 398 → 1.000 · 끝에 남은 크기 0.0322


**결과** · 예측 정답은 맞힘 — 1.000

- `거리   8 → 1.000 · 끝에 남은 크기 0.0377` — 강의와 같은 값
- `거리 398 → 1.000 · 끝에 남은 크기 0.0322` — 398칸을 지나도 거의 그대로 남음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>손으로 심은 가중치</td><td>기억이 닿을 수 있음을 증명</td><td>가능성 확인</td></tr>
<tr><td>학습한 가중치</td><td>같은 해를 찾아야 함</td><td>실제 사용</td></tr>
<tr><td>학습 전 가중치 (5-1)</td><td>40칸 근처에서 사라짐</td><td>학습의 출발점</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 가능하다고 학습이 그 해를 찾는다는 보장은 없음 → 다음 질문 '왜 멀수록 배우기가 느린가'

> ✅ **한 줄 정리** · 요약을 그대로 넘기는 가중치를 심으면 398칸도 닿으므로, 먼 거리에서 실패하는 원인은 기억이 아니라 학습 쪽이다.

**딥러닝 코드에서는**

- 잔차 연결 · LSTM 메모처럼 '그대로 넘기는 길'을 구조로 넣는 설계가 여기서 나옴

### 5-3. `A` 뒤 → 앞: 기울기를 직접 재기 · `retain_grad()` · `BPTT` <sub>강의 4부 · '기울기 = 고치라는 신호의 크기' · 'L3 셀로 재 봄'</sub>

> 🎯 **마지막 손실의 신호(기울기)는 몇 칸 거슬러 가면 사라지는가?**

💡 **쉽게 말하면**

- 역전파는 **지나온 길을 거꾸로** 되짚음 · 지름길이 없어 칸을 하나씩 거슬러 감 (BPTT).
- `h.retain_grad()`로 중간 요약의 기울기를 남겨 칸마다 크기를 잼.

**실행 흐름**

1. RNNCell을 손으로 100칸 펼침
2. 칸마다 retain_grad
3. 끝 손실 backward → 0 · 10 · 20 · 40 · 70 · 99칸 크기

**먼저 예측** · 20칸 거슬러 가면 몇 배 작아질까? — 답은 코드 실행 결과 아래에

In [20]:
# 뒤 → 앞: 마지막 손실의 신호(기울기)가 k칸 거슬러 간 요약에 얼마나 오나
# 왜 · 기울기가 멀리 거슬러 갈수록 작아지는지 직접 재기 위해
STEPS = (0, 10, 20, 40, 70, 99)  # 거슬러 간 칸 수


# 왜 · 셀 종류 · 문 시작값 · 자료형만 바꿔 같은 방법으로 기울기를 재기 위해
def grad_profile(kind, forget_bias=None, dtype=torch.float32,  # 칸별 기울기 크기
                 steps=STEPS, T=100, hid=32):
    torch.manual_seed(SEED)                                    # 같은 시작점
    cell = {"RNN": nn.RNNCell, "LSTM": nn.LSTMCell}[kind](10, hid).to(dtype)  # 손으로 펼칠 상자
    head = nn.Linear(hid, 2).to(dtype)                         # 끝 분류기
    if forget_bias is not None:  # bias 순서 [input | forget | cell | output] (6장)
        with torch.no_grad():                                  # 값만 넣음
            cell.bias_ih[hid:2 * hid].fill_(forget_bias)       # forget 구간
            cell.bias_hh[hid:2 * hid].fill_(0.0)               # 두 bias가 더해지므로 한쪽은 0
    xs, ys = make_front_task(64, T, seed=7)                    # 64줄 · 길이 100
    X = to_tensor(xs).to(dtype)                                # 자료형 맞춤
    # 요약 · 메모
    h, c, hs = torch.zeros(64, hid, dtype=dtype), torch.zeros(64, hid, dtype=dtype), []
    for t in range(T):                                         # T칸 차례로
        h = cell(X[:, t], h) if kind == "RNN" else cell(X[:, t], (h, c))  # RNN은 h · LSTM은 (h, c)
        if kind == "LSTM":                                     # LSTM은 둘을 돌려줌
            h, c = h                                           # 요약 · 메모
        h.retain_grad()  # 중간 요약의 기울기를 버리지 말고 남기기
        hs.append(h)                                           # 저장
    nn.CrossEntropyLoss()(head(hs[-1]), torch.tensor(ys)).backward()  # 끝에서 손실 → 역전파
    return [hs[T - 1 - k].grad.norm().item() for k in steps]   # k칸 거슬러 간 기울기 크기


# 왜 · 한글 폭을 맞춰 여러 결과를 표처럼 나란히 읽기 위해
def show(name, values):                                          # 한 줄 출력 (한글 2칸 폭)
    pad = 12 - sum(2 if ord(ch) > 0x1100 else 1 for ch in name)  # 모자란 폭
    cells = [f"{v:>9}" if isinstance(v, int) else f"{v:>9.1e}" for v in values]  # 정수 · 지수 표기
    print(name + " " * pad + "".join(cells))                     # 열 맞춤 출력


show("거슬러 간 칸", STEPS)  # 머리 줄
rnn_g = grad_profile("RNN")  # 학습 전 RNN
show("RNN", rnn_g)           # 칸별 크기
# 제곱처럼 커짐
print("10 · 20 · 40스텝 배율 :", " · ".join(f"{rnn_g[0] / rnn_g[k]:,.0f}배" for k in (1, 2, 3)))

거슬러 간 칸        0       10       20       40       70       99
RNN           5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  0.0e+00
10 · 20 · 40스텝 배율 : 486배 · 175,503배 · 38,772,643,119배


**결과** · 예측 정답은 약 17만 배

- `RNN           5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  0.0e+00` — 거슬러 갈수록 급격히 작아짐 · 99칸은 0
- `10 · 20 · 40스텝 배율 : 486배 · 175,503배 · 38,772,643,119배` — 거리가 두 배면 배율이 제곱처럼 커짐 (강의 17만 배)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>h.retain_grad()</code></td><td>중간 텐서의 기울기를 남김</td><td>칸별 기울기 측정</td></tr>
<tr><td><code>loss.backward()</code></td><td>끝에서 거꾸로 신호 전달</td><td>학습 · 측정 공통</td></tr>
<tr><td>BPTT</td><td>시간 축으로도 거슬러 가는 역전파</td><td>RNN 학습 전부</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `retain_grad()`를 안 쓰면 `h.grad`가 None
> - 상자가 하나라 백 칸의 신호가 같은 가중치에 더해짐 → 먼 칸 몫은 묻힘

> ✅ **한 줄 정리** · 기울기는 칸을 하나씩 거슬러 가며 줄어들어 20칸에서 약 17만 배 작아지고, 출발점에서 맨 앞을 고칠 신호가 거의 오지 않는 것이 기울기 소실이다.

**딥러닝 코드에서는**

- 긴 시퀀스 학습이 느리면 기울기 크기를 칸별로 찍어 보는 것이 첫 진단

### 5-4. `B` 0은 진짜 0일까 · output.grad 함정 · `torch.float64` · `output.grad` <sub>강의 4부 · '99스텝의 0 은 진짜 0 이 아님' · '쓰면 안 되는 지름길'</sub>

> 🎯 **99칸의 0은 진짜 0인가, 그리고 `output.grad`로 재면 왜 안 되는가?**

💡 **쉽게 말하면**

- 같은 계산을 **float64**로 하면 0이 아니라 아주 작은 수 → float32가 못 적는 것.
- `output`은 칸마다 값을 **복사해 보여 줄 뿐**, 다음 칸으로 가는 길 위의 h는 안에 감춰져 있음.

**실행 흐름**

1. float64로 같은 측정
2. float32가 처음 0이 되는 칸
3. output.retain_grad로 잰 0 칸 수

**먼저 예측** · `output.grad`가 정확히 0인 칸은 100칸 중 몇 칸일까? — 답은 코드 실행 결과 아래에

In [21]:
# 99스텝의 0은 진짜 0일까: 같은 계산을 float64로
# 왜 · float32에서 0으로 보인 값이 정말 0인지 float64로 다시 확인하기 위해
show("거슬러 간 칸", STEPS)                                      # 머리 줄
show("RNN float64", grad_profile("RNN", dtype=torch.float64))    # 0이 아니라 아주 작은 수
every = grad_profile("RNN", steps=range(100))                    # float32 · 0 ~ 99칸 전부
first0 = next(k for k in range(100) if every[k] == 0.0)          # 처음 0인 칸
print("float32 에서 처음 0 으로 찍히는 자리 :", first0, "스텝")  # 적을 수 있는 한계


# 쓰면 안 되는 지름길: output.retain_grad() → output.grad 는 h의 기울기가 아님
# 왜 · output.grad가 h의 기울기가 아니라는 함정을 미리 보기 위해
torch.manual_seed(SEED)                                                      # 같은 시작점
rnn, head = nn.RNN(10, 32, batch_first=True), nn.Linear(32, 2)               # 감싼 판 · 분류기
xs, ys = make_front_task(64, 100, seed=7)                                    # 같은 데이터
output, _ = rnn(to_tensor(xs))                                               # 칸별 요약
output.retain_grad()                                                         # output 기울기 남기기
nn.CrossEntropyLoss()(head(output[:, -1]), torch.tensor(ys)).backward()      # 끝 칸만 손실에
zero = sum(output.grad[:, t].abs().max().item() == 0.0 for t in range(100))  # 정확히 0인 칸
print("output.grad 가 정확히 0 인 칸 :", zero, "/ 100")                      # 감춰진 길은 안 보임

거슬러 간 칸        0       10       20       40       70       99
RNN float64   5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  3.4e-28
float32 에서 처음 0 으로 찍히는 자리 : 77 스텝
output.grad 가 정확히 0 인 칸 : 99 / 100


**결과** · 예측 정답은 99칸 — 손실에 직접 쓴 끝 칸만 값이 있음

- `RNN float64   5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  3.4e-28` — 99칸은 0이 아니라 3.4e-28 (강의 3.38e-28)
- `float32 에서 처음 0 으로 찍히는 자리 : 77 스텝` — 77칸부터 float32로는 0
- `output.grad 가 정확히 0 인 칸 : 99 / 100` — h의 기울기가 아님 → 이 값으로 소실을 재면 안 됨

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>float32 측정</td><td>아주 작은 값은 0으로 찍힘</td><td>학습 (기본)</td></tr>
<tr><td>float64 측정</td><td>더 작은 값까지 적음</td><td>0인지 확인할 때</td></tr>
<tr><td><code>output.grad</code></td><td>손실에 직접 쓴 칸만</td><td>쓰지 않음 (측정 착시)</td></tr>
<tr><td>손으로 편 cell + <code>h.retain_grad()</code></td><td>다음 칸으로 가는 길 위의 h</td><td>칸별 기울기 측정 (5-3)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 학습에는 3.4e-28이나 0이나 같음 → 0이 아니라도 고칠 신호는 없음

> ✅ **한 줄 정리** · 99칸의 0은 float32가 못 적는 아주 작은 수이고, 칸별 기울기는 `output.grad`가 아니라 손으로 편 cell의 `h.retain_grad()`로 재야 한다.

**딥러닝 코드에서는**

- `torch.autograd.grad(loss, hs)`로도 중간 값의 기울기를 꺼낼 수 있음

### 5-5. `A` 같은 수를 계속 곱하면 · `clip_grad_norm_()` · `eigvals()` <sub>강의 4부 · '같은 수를 계속 곱하면' · '재보면 그림대로 안 감'</sub>

> 🎯 **기울기는 왜 사라지거나 터지고, 둘 중 어느 쪽에만 간단한 대처가 있는가?**

💡 **쉽게 말하면**

- 한 칸 거슬러 갈 때마다 같은 비율이 곱해짐 · **1보다 조금 작으면 소실 · 조금 크면 폭발**.
- 폭발은 **잘라내면(clipping)** 되지만 소실은 통째로 키워도 비율이 그대로.

**실행 흐름**

1. 0.9 · 1.1 · 1.0을 50번 곱하기
2. 크기 50 기울기를 5로 자르기
3. 실제 행렬의 배율 · 실측 비율

**먼저 예측** · 0.9를 50번 곱하면? — 답은 코드 실행 결과 아래에

In [22]:
# 같은 수를 계속 곱하면: 1보다 조금 작으면 사라지고 조금 크면 터짐
# 왜 · 1보다 작으면 사라지고 크면 터지는 원리를 숫자로 보기 위해
# 소실 · 폭발 · 유지
print(f"0.9 ** 50 = {0.9 ** 50:.3f} · 1.1 ** 50 = {1.1 ** 50:.0f} · 1.0 ** 50 = {1.0 ** 50:.0f}")


# 폭발 쪽 대처: gradient clipping (전체 크기가 기준을 넘으면 같은 비율로 줄임)
# 왜 · 기울기가 너무 커질 때 전체 크기만 줄여 학습을 안정시키기 위해
w = nn.Parameter(torch.zeros(3))                                        # 파라미터 3개
w.grad = torch.tensor([30.0, 40.0, 0.0])                                # 크기 50짜리 기울기
nn.utils.clip_grad_norm_([w], max_norm=5.0)                             # 크기 5로 자르기
g = [round(v, 3) for v in w.grad.tolist()]                              # 자른 기울기
print("clip 후 기울기 :", g, "· 크기", round(w.grad.norm().item(), 3))  # 방향은 그대로


# 실제로 곱해지는 것은 숫자 하나가 아니라 행렬 (학습 전 · float64)
# 왜 · 행렬의 고유값으로 사라짐 · 폭발을 판단하는 방법을 보기 위해
torch.manual_seed(SEED)                                             # 6-3과 같은 시작점
W = nn.RNNCell(10, 32).to(torch.float64).weight_hh                  # 이전 요약에 곱하는 행렬
print(f"최대 특이값 {torch.linalg.svdvals(W).max().item():.2f} · "  # 한 번 곱할 때 최대 배율
      # 거듭 곱할 때 남는 배율
      f"고유값 크기 최댓값 {torch.linalg.eigvals(W).abs().max().item():.2f}")
g64 = grad_profile("RNN", dtype=torch.float64, steps=range(71))     # 0 ~ 70칸 기울기
# 실측
print(f"실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝) : {(g64[70] / g64[0]) ** (1 / 70):.2f}")

0.9 ** 50 = 0.005 · 1.1 ** 50 = 117 · 1.0 ** 50 = 1
clip 후 기울기 : [3.0, 4.0, 0.0] · 크기 5.0
최대 특이값 1.10 · 고유값 크기 최댓값 0.57
실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝) : 0.54


**결과** · 예측 정답은 0.005 — 천분의 5

- `0.9 ** 50 = 0.005 · 1.1 ** 50 = 117 · 1.0 ** 50 = 1` — 10%만 달라도 소실 · 폭발 · 딱 1이어야 유지
- `clip 후 기울기 : [3.0, 4.0, 0.0] · 크기 5.0` — 크기만 5로 · 방향 (3 : 4)은 그대로
- `최대 특이값 1.10 · 고유값 크기 최댓값 0.57` — 한 번은 1.10배까지 늘지만 거듭 곱하면 0.57배씩
- `실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝) : 0.54` — 0.57 × tanh 미분 0.96 ≈ 0.55와 비슷

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>폭발</td><td>손실이 갑자기 NaN · 눈에 띔</td><td><code>clip_grad_norm_(params, 5.0)</code></td></tr>
<tr><td>소실</td><td>조용히 먼 칸을 못 배움</td><td>구조를 바꿈 (LSTM · GRU 6 · 7장)</td></tr>
<tr><td>0.9를 50번 그림</td><td>감 잡기용</td><td>설명할 때 · 답은 재서 냄</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 작은 기울기를 통째로 키우면 가까운 칸 몫도 같이 커져 먼 칸이 묻힌 비율은 그대로

> ✅ **한 줄 정리** · 같은 비율이 칸마다 곱해져 1보다 작으면 소실 · 크면 폭발이 되며, 폭발은 clipping으로 막지만 소실은 구조를 바꿔야 한다.

**딥러닝 코드에서는**

- `torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)`는 RNN 학습 루프의 기본 줄

---
## 6. LSTM · 남길 것을 학습

> **이 장의 흐름**
>
> - 강의 5부 · 남길 양을 고르는 손잡이(sigmoid 문)와 길게 남길 메모 c를 추가
> - 같은 자로 다시 재 보니 문을 열어 두었을 때만 살아남

<img src="images/rl_lstm_cell.png" alt="LSTM 한 칸 · 메모 c는 곱하기와 더하기만 거쳐 곧게 지나감" width="703">

**파이프라인 위치** · ④ 모델 — 셀 구조 고르기

### 6-1. `A` 문(gate) = sigmoid 수도꼭지 · 메모 c · `torch.sigmoid()` · `c = f × c + i × g` <sub>강의 5부 · '필요한 부품' · '메모에 허용된 일은 두 가지뿐임'</sub>

> 🎯 **LSTM은 무엇을 새로 달고, 메모 c는 왜 곧게 지나갈 수 있는가?**

💡 **쉽게 말하면**

- **수도꼭지(문)** · sigmoid로 0 ~ 1 사이 여는 정도를 만들어 곱함.
- **메모 c** · 지우기(곱)와 덧붙이기(더하기)만 허용 · 통째로 다시 쓰지 않음.

<img src="images/rl_gates.png" alt="sigmoid 문 · bias 0 · 2 · 4에서 시작하는 문 값" width="662">

**실행 흐름**

1. sigmoid로 아무 값 → 0 ~ 1
2. f = 0 · 0.5 · 1로 통과량
3. c ← f × c + i × g · h = o × tanh(c)

**먼저 예측** · f = 1 · i = 0이면 메모는? — 답은 코드 실행 결과 아래에

In [23]:
# 수도꼭지 = sigmoid: 어떤 값이 와도 0 ~ 1 사이 → 여는 정도로 씀
# 왜 · 문(gate)이 0 ~ 1 사이 값으로 여는 정도를 정한다는 것을 확인하기 위해
z = torch.tensor([-4.0, -2.0, 0.0, 2.0, 4.0])                         # 안쪽 계산값 (아무 값)
print("sigmoid :", [round(v, 2) for v in torch.sigmoid(z).tolist()])  # 0 · 0.5 · 1 근처
value = torch.tensor(10.0)                                            # 원래 값
for f in (0.0, 0.5, 1.0):                                             # 잠금 · 반 · 다 열림
    print(f"f = {f} → 통과한 양 {f * value.item():.1f}")              # 곱하기 하나


# 메모 c에 허용된 일 두 가지: 지우기(곱) · 덧붙이기(더하기)
# 왜 · LSTM 메모가 곱(지우기)과 더하기(덧붙이기)로만 바뀐다는 것을 보기 위해
c = torch.tensor([1.0, 1.0, 1.0])                                    # 이전 메모
f = torch.tensor([1.0, 0.5, 0.0])                                    # forget 문: 남길 양
i, g = torch.tensor([0.0, 1.0, 1.0]), torch.tensor([0.3, 0.3, 0.3])  # input 문 · 적을 내용
c_new = f * c + i * g                                                # c ← f × c + i × g
o = torch.tensor([1.0, 1.0, 0.0])                                    # output 문: 꺼낼 양
h_new = o * torch.tanh(c_new)                                        # h ← o × tanh(c)
print("새 메모 c :", [round(v, 3) for v in c_new.tolist()],          # 메모
      "· 꺼낸 h :", [round(v, 3) for v in h_new.tolist()])

sigmoid : [0.02, 0.12, 0.5, 0.88, 0.98]
f = 0.0 → 통과한 양 0.0
f = 0.5 → 통과한 양 5.0
f = 1.0 → 통과한 양 10.0
새 메모 c : [1.0, 0.8, 0.3] · 꺼낸 h : [0.762, 0.664, 0.0]


**결과** · 예측 정답은 그대로 1.0 — 지우지도 덧붙이지도 않음

- `sigmoid : [0.02, 0.12, 0.5, 0.88, 0.98]` — 어떤 값이 와도 0 ~ 1 → 여는 정도로 바로 씀
- `f = 0.5 → 통과한 양 5.0` — 반만 열면 절반만 지나감
- `새 메모 c : [1.0, 0.8, 0.3] · 꺼낸 h : [0.762, 0.664, 0.0]` — 칸마다 남김 · 덧붙임이 다름 · h는 c에서 꺼낸 만큼

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>forget 문 f</td><td>메모에서 <strong>남길</strong> 양 (<code>c ← f × c</code>)</td><td>오래 기억할지</td></tr>
<tr><td>input 문 i · 내용 g</td><td>메모에 <strong>적을</strong> 양 · 적을 내용 (tanh)</td><td>새 정보를 넣을지</td></tr>
<tr><td>output 문 o</td><td>메모에서 지금 <strong>꺼낼</strong> 양 (<code>h ← o × tanh(c)</code>)</td><td>지금 답에 쓸지</td></tr>
<tr><td>RNN의 h</td><td>매 칸 통째로 새로 만듦</td><td>남길 양을 고르는 손잡이가 없음</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 이름은 '잊는 문'이지만 실제로 정하는 것은 **남기는 양**
> - f가 1일 때만 메모 줄에 곱이 없음

> ✅ **한 줄 정리** · LSTM은 0 ~ 1 사이 문 세 개로 메모 c에서 남길 · 적을 · 꺼낼 양을 학습하고, c는 곱하기와 더하기만 거쳐 곧게 지나간다.

**딥러닝 코드에서는**

- 회의 노트(c)와 지금 하는 말(h)을 나눈 것 · 답에는 민감하게, 기억에는 둔감하게

### 6-2. `A` LSTM이 돌려주는 것 · `output, (h_n, c_n)` <sub>강의 5부 · '코드에서는 한 줄임' · '값은 치르고 있음'</sub>

> 🎯 **LSTM은 텐서를 몇 개 돌려주고, 같은 hidden에서 파라미터는 얼마나 늘어나는가?**

💡 **쉽게 말하면**

- 돌려받는 텐서가 **두 개에서 세 개**로 · `output, (h_n, c_n)`.
- 문 세 개 + 적을 내용까지 **네 벌**을 계산해 파라미터가 4배.

**실행 흐름**

1. LSTM(10, 32)에 8줄
2. 세 모양 · h_n과 c_n 비교
3. RNN · LSTM 파라미터 · weight_ih 모양

**먼저 예측** · h_n과 c_n은 같은 값일까? — 답은 코드 실행 결과 아래에

In [24]:
# LSTM은 텐서 세 개를 돌려줌: output, (h_n, c_n)
# 왜 · LSTM 출력 모양을 RNN과 비교해 확인하기 위해
torch.manual_seed(SEED)                                      # 같은 시작점
lstm = nn.LSTM(10, 32, batch_first=True)                     # 입력 10 · hidden 32
x = to_tensor(xtr[:8])                                       # (8, 10, 10)
output, (h_n, c_n) = lstm(x)                                 # 괄호 안 두 개 = 튜플 풀기
print("x", tuple(x.shape), "· output", tuple(output.shape),  # 네 모양
      "· h_n", tuple(h_n.shape), "· c_n", tuple(c_n.shape))
print("h_n 과 c_n 이 같은 값인가 :", torch.equal(h_n, c_n))  # h = o × tanh(c)
n_rnn = sum(p.numel() for p in nn.RNN(10, 32).parameters())  # 1벌
n_lstm = sum(p.numel() for p in lstm.parameters())           # 문 3개 + 적을 내용 = 4벌
# 같은 hidden에서 4배
print(f"파라미터 RNN {n_rnn:,} · LSTM {n_lstm:,} · 배율 {n_lstm / n_rnn:.0f}")
print("weight_ih 모양 :", tuple(lstm.weight_ih_l0.shape), "= (4 × 32, 10)")  # [i | f | g | o]

x (8, 10, 10) · output (8, 10, 32) · h_n (1, 8, 32) · c_n (1, 8, 32)
h_n 과 c_n 이 같은 값인가 : False
파라미터 RNN 1,408 · LSTM 5,632 · 배율 4
weight_ih 모양 : (128, 10) = (4 × 32, 10)


**결과** · 예측 정답은 다름 — h = o × tanh(c)

- `x (8, 10, 10) · output (8, 10, 32) · h_n (1, 8, 32) · c_n (1, 8, 32)` — c_n이 하나 더 · 모양은 h_n과 같음
- `h_n 과 c_n 이 같은 값인가 : False` — 꺼낸 말(h)과 메모(c)는 다른 값
- `파라미터 RNN 1,408 · LSTM 5,632 · 배율 4` — 네 벌 계산
- `weight_ih 모양 : (128, 10) = (4 × 32, 10)` — [i | f | g | o] 네 구간이 한 텐서에 붙어 있음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>output, h_n = rnn(x)</code></td><td>두 개</td><td>RNN · GRU</td></tr>
<tr><td><code>output, (h_n, c_n) = lstm(x)</code></td><td>세 개 (뒤 둘은 튜플)</td><td>LSTM</td></tr>
<tr><td><code>output, _ = lstm(x)</code></td><td>두 번째를 통째로 버림</td><td>output만 쓸 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `output, h_n = lstm(x)`로 받으면 h_n 자리에 튜플이 들어가 뒤에서 에러

> ✅ **한 줄 정리** · LSTM은 `output, (h_n, c_n)` 세 개를 돌려주고, 같은 hidden에서 네 벌을 계산해 파라미터가 RNN의 4배다.

**딥러닝 코드에서는**

- 모델 코드에서 `nn.RNN` → `nn.LSTM`으로 바꿀 때 받는 줄도 함께 바꿀 것

### 6-3. `A` 문을 달아도 죽는 경우 · `bias_ih[H:2*H]` <sub>강의 5부 · '아흔아홉 칸 뒤에서 기울기가 살아 있을까' · '아까 똑같은 걸 봤음'</sub>

> 🎯 **보통의 LSTM(forget 문 bias 0)은 왜 99칸 뒤 기울기가 RNN처럼 0인가?**

💡 **쉽게 말하면**

- bias가 0이면 문 값이 **0.5 근처** = 반쯤 열린 문.
- 반쯤 열린 문을 백 번 지나면 0.5를 백 번 곱한 것과 같음.

**실행 흐름**

1. 같은 조건에서 RNN · LSTM 기본 기울기
2. forget 문 값을 백 칸 기록
3. 0.5를 백 번 곱하기

**먼저 예측** · LSTM 기본은 99칸 뒤에 기울기가 남을까? — 답은 코드 실행 결과 아래에

In [25]:
# 같은 조건(데이터 · 길이 100 · hidden 32)에서 셀 종류만 바꿈 · forget 문 bias 0 = 보통의 LSTM
# 왜 · 셀 종류만 바꾼 공정한 비교로 LSTM의 효과를 확인하기 위해
show("거슬러 간 칸", STEPS)                               # 머리 줄
show("RNN", grad_profile("RNN"))                          # 비교 기준
show("LSTM 기본", grad_profile("LSTM", forget_bias=0.0))  # 문을 달았지만


# 왜 죽었나: 문에 더하는 값이 0이면 문 값이 0.5 근처 → 반쯤 열린 문을 백 번 지남
# 왜 · 문이 반쯤 열린 채 시작하면 멀리 기억하지 못하는 원인을 찾기 위해
torch.manual_seed(SEED)                                                         # 같은 시작점
cell = nn.LSTMCell(10, 32)                                                      # 학습 전 LSTM 상자
with torch.no_grad():                                                           # 측정만
    cell.bias_ih[32:64].fill_(0.0)                                              # forget 구간 0
    cell.bias_hh[32:64].fill_(0.0)                                              # 한쪽도 0
    xs, _ = make_order_task(64, 100, seed=7)                                    # 길이 100
    X, h, c, fs = to_tensor(xs), torch.zeros(64, 32), torch.zeros(64, 32), []   # 입력 · 상태 · 기록
    for t in range(100):                                                        # 100칸
        # 네 문
        g4 = X[:, t] @ cell.weight_ih.T + cell.bias_ih + h @ cell.weight_hh.T + cell.bias_hh
        fs.append(torch.sigmoid(g4[:, 32:64]).mean().item())                    # forget 문 값 평균
        h, c = cell(X[:, t], (h, c))                                            # 한 칸 진행
print(f"문 값 백 칸 평균 {sum(fs) / 100:.4f} · 0.5 ** 100 = {0.5 ** 100:.1e}")  # 10^-30 쪽

거슬러 간 칸        0       10       20       40       70       99


RNN           5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  0.0e+00
LSTM 기본     5.3e-02  4.6e-05  4.0e-07  2.8e-11  2.1e-17  0.0e+00


문 값 백 칸 평균 0.4997 · 0.5 ** 100 = 7.9e-31


**결과** · 예측 정답은 안 남음 — 0.0e+00

- `LSTM 기본     5.3e-02  4.6e-05  4.0e-07  2.8e-11  2.1e-17  0.0e+00` — RNN과 똑같이 99칸은 0 · 10칸은 오히려 더 작음
- `문 값 백 칸 평균 0.4997 · 0.5 ** 100 = 7.9e-31` — 반쯤 열린 문 백 번 = 10의 -30제곱 쪽

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>RNN</td><td>남길 양 손잡이가 없음</td><td>짧은 거리</td></tr>
<tr><td>LSTM 기본 (bias 0)</td><td>손잡이는 있지만 반쯤 닫힌 채 시작</td><td>PyTorch 기본 초기화</td></tr>
<tr><td>LSTM 열어둠 (6-4)</td><td>1 근처에서 시작</td><td>먼 거리 기억이 필요할 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 'LSTM이 기울기 소실을 해결했다'는 말은 정확하지 않음 → 문이 열려 있을 때만 크게 늦춤

> ✅ **한 줄 정리** · forget 문 bias가 0이면 문 값이 0.5 근처라 0.5를 백 번 곱한 셈이 되어, LSTM도 99칸 뒤 기울기가 RNN처럼 사라진다.

**딥러닝 코드에서는**

- 0.9를 50번 (RNN) · 0.5를 100번 (LSTM 기본) — 1보다 작은 수를 반복해 곱하는 같은 구조

### 6-4. `A` 문을 열어두면 · `fill_(2.0)` · `sigmoid(2) = 0.88` <sub>강의 5부 · '문을 열어두고 시작해 볼 것' · '더 열면 더 남음'</sub>

> 🎯 **forget 문 bias를 2 · 4로 두면 기울기는 얼마나 남고, 그래도 '해결'이 아닌 이유는?**

💡 **쉽게 말하면**

- bias를 2로 두면 문이 **0.88**에서 시작 · 4면 **0.98** · 곡선은 그대로, 시작점만 옮김.
- 이름으로는 못 찾아서 **두 번째 구간** `bias_ih[H:2*H]`에 넣음.

<img src="images/rl_grad.png" alt="거슬러 간 칸별 기울기 · RNN · LSTM 기본 · 열어둠 · 활짝" width="941">

**실행 흐름**

1. forget 구간 자리 확인
2. bias 0 · 2 · 4 기울기 표
3. 열어둔 경우 0스텝 ÷ 99스텝

**먼저 예측** · 열어둔 LSTM의 99칸 기울기는 0일까? — 답은 코드 실행 결과 아래에

In [26]:
# 문을 열어두고 시작: forget 문의 bias를 2 · 4로 (이름이 아니라 자리로 찾음)
# 왜 · forget 문의 시작값만 바꿔 멀리 기억하는지 확인하기 위해
H = 32                                                                 # hidden 크기
# 두 번째 구간
print("bias 자리 [ input | forget | cell | output ] → forget =", f"bias_ih[{H}:{2 * H}]")
start = torch.sigmoid(torch.tensor([0., 2., 4.]))                      # bias → 시작 문 값
print("시작 문 값 sigmoid(0 · 2 · 4) :", [round(v, 2) for v in start.tolist()])  # 0.5 · 0.88 · 0.98
show("거슬러 간 칸", STEPS)                                            # 머리 줄
show("RNN", grad_profile("RNN"))                                       # 0으로 죽음
show("LSTM 기본", grad_profile("LSTM", forget_bias=0.0))               # 0으로 죽음
opened = grad_profile("LSTM", forget_bias=2.0)                         # 열어둠
show("LSTM 열어둠", opened)                                            # 99칸에도 남음
show("LSTM 활짝", grad_profile("LSTM", forget_bias=4.0))               # 더 남음
print(f"열어둔 경우 0스텝 ÷ 99스텝 = {opened[0] / opened[5]:,.0f}배")  # 해결이 아니라 크게 늦춤

bias 자리 [ input | forget | cell | output ] → forget = bias_ih[32:64]
시작 문 값 sigmoid(0 · 2 · 4) : [0.5, 0.88, 0.98]
거슬러 간 칸        0       10       20       40       70       99


RNN           5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  0.0e+00


LSTM 기본     5.3e-02  4.6e-05  4.0e-07  2.8e-11  2.1e-17  0.0e+00

LSTM 열어둠   5.3e-02  1.0e-03  5.2e-04  1.9e-04  3.9e-05  1.6e-05
LSTM 활짝     5.3e-02  7.2e-04  5.9e-04  6.7e-04  9.8e-04  4.5e-03
열어둔 경우 0스텝 ÷ 99스텝 = 3,303배


**결과** · 예측 정답은 0 아님 — 1.6e-05로 남음

- `bias 자리 [ input | forget | cell | output ] → forget = bias_ih[32:64]` — 네 문의 bias가 한 텐서에 i · f · g · o 순서
- `시작 문 값 sigmoid(0 · 2 · 4) : [0.5, 0.88, 0.98]` — 시작점만 옮김
- `LSTM 열어둠   5.3e-02  1.0e-03  5.2e-04  1.9e-04  3.9e-05  1.6e-05` — 99칸 뒤에도 남음
- `LSTM 활짝     5.3e-02  7.2e-04  5.9e-04  6.7e-04  9.8e-04  4.5e-03` — 멀어질수록 오히려 커지기도 함
- `열어둔 경우 0스텝 ÷ 99스텝 = 3,303배` — 사라지진 않았지만 크게 줄기는 함 → 해결이 아니라 늦춤

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>bias 0 (기본)</td><td>문 0.5에서 시작</td><td>짧은 의존</td></tr>
<tr><td>bias 1 ~ 2</td><td>문 0.73 ~ 0.88에서 시작</td><td>긴 의존 · 흔한 권장값</td></tr>
<tr><td>bias 4</td><td>문 0.98 · 거의 다 남김</td><td>아주 먼 의존 · 잊기가 느려짐</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `bias_ih`와 `bias_hh`는 더해서 쓰이므로 한쪽에 값을 넣으면 다른 쪽은 0으로

> ✅ **한 줄 정리** · 문을 열어 두면 99칸 뒤에도 기울기가 남지만 3,303배 줄므로, LSTM은 소실을 해결한 게 아니라 크게 늦춘 것이다.

**딥러닝 코드에서는**

- `lstm.bias_ih_l0.data[H:2*H].fill_(1.0)` — 흔한 초기화 요령 (`bias_hh_l0` 쪽은 0)

---
## 7. GRU와 고르는 법

> **이 장의 흐름**
>
> - 강의 6부 · 손잡이를 둘로 줄인 GRU
> - 구조만 바꾼 비교 → 조건을 맞춘 비교 → 비용 · 강의 7부 양방향

<img src="images/rl_gru.png" alt="LSTM은 손잡이 3 · 줄 2, GRU는 손잡이 2 · 줄 1" width="718">

**파이프라인 위치** · ⑥ 평가 — 통제된 비교로 모델 고르기

### 7-1. `B` GRU와 파라미터 수 · `nn.GRU` <sub>강의 6부 · '손잡이를 둘로 줄인 답' · '같은 화면에 파라미터를 더함'</sub>

> 🎯 **GRU는 LSTM에서 무엇을 줄였고, 같은 hidden 64는 왜 같은 크기가 아닌가?**

💡 **쉽게 말하면**

- GRU = **손잡이 2개 · 넘기는 줄 1개** (버릴 양과 넣을 양을 한 손잡이 z가 같이 정함, 합이 1).
- 계산하는 벌 수가 RNN 1 · GRU 3 · LSTM 4라 같은 64여도 파라미터가 다름.

**실행 흐름**

1. GRU 반환 모양
2. HAR 크기(입력 9 · 출력 6)로 파라미터 세기
3. 강의의 다섯 크기

**먼저 예측** · GRU는 몇 개를 돌려줄까? — 답은 코드 실행 결과 아래에

In [27]:
# GRU: 손잡이 2개 · 넘기는 줄 1개 → RNN처럼 두 개만 돌려줌
# 왜 · GRU의 출력 모양을 LSTM과 비교하기 위해
torch.manual_seed(SEED)                                           # 같은 시작점
out, h_n = nn.GRU(10, 32, batch_first=True)(to_tensor(xtr[:8]))   # (h_n, c_n) 아님
print("GRU output", tuple(out.shape), "· h_n", tuple(h_n.shape))  # 메모 c 없음


# HAR 크기(입력 9 · 출력 6)의 파라미터: RNN 1벌 · GRU 3벌 · LSTM 4벌 + Linear
# 왜 · 셀 종류마다 파라미터가 몇 벌인지 세어 비용을 비교하기 위해
def har_params(kind, hid):                                           # 층 + Linear(hid → 6)
    layer = getattr(nn, kind)(9, hid, batch_first=True)              # 층 만들기
    return sum(p.numel() for p in layer.parameters()) + hid * 6 + 6  # 분류기까지


# 왜 · 크기를 맞췄을 때 파라미터 수가 어떻게 달라지는지 비교하기 위해
sizes = [("RNN", 64), ("LSTM", 64), ("GRU", 64), ("RNN", 128), ("GRU", 74)]  # 강의의 다섯 크기
for kind, hid in sizes:                                                      # 크기마다
    # 같은 64 ≠ 같은 크기
    print(f"{kind:<4} hidden {hid:<3} → 파라미터 {har_params(kind, hid):>6,}")

GRU output (8, 10, 32) · h_n (1, 8, 32)
RNN  hidden 64  → 파라미터  5,190
LSTM hidden 64  → 파라미터 19,590
GRU  hidden 64  → 파라미터 14,790
RNN  hidden 128 → 파라미터 18,566
GRU  hidden 74  → 파라미터 19,320


**결과** · 예측 정답은 두 개 — RNN처럼 `output, h_n`

- `GRU output (8, 10, 32) · h_n (1, 8, 32)` — 메모 c가 없음
- `RNN  hidden 64  → 파라미터  5,190` — 1벌 + 분류기
- `LSTM hidden 64  → 파라미터 19,590` — 4벌 · 같은 64인데 3.77배
- `GRU  hidden 64  → 파라미터 14,790` — 3벌 · LSTM보다 약 25% 적음
- `RNN  hidden 128 → 파라미터 18,566` — 크기를 맞춘 비교용
- `GRU  hidden 74  → 파라미터 19,320` — LSTM 64 근처

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>RNN</td><td>손잡이 0 · 줄 1</td><td>가중치가 가장 적음 · 먼저 돌려 볼 것</td></tr>
<tr><td>LSTM</td><td>손잡이 3 · 줄 2 (메모 c 따로)</td><td>자유도가 필요할 때</td></tr>
<tr><td>GRU</td><td>손잡이 2 · 줄 1</td><td>LSTM과 비슷한 성능 · 더 가볍게</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - hidden 크기가 같다고 공정한 비교가 아님 → 파라미터 수를 함께 적을 것

> ✅ **한 줄 정리** · GRU는 LSTM의 손잡이를 둘 · 넘기는 줄을 하나로 줄인 구조이고, 같은 hidden에서도 계산 벌 수(1 · 3 · 4)가 달라 파라미터가 다르다.

**딥러닝 코드에서는**

- 모델 비교표에는 정확도와 파라미터 수 · 학습 시간을 같이 적음

### 7-2. `B` 통제된 비교 두 번 읽기 · `%p` <sub>강의 6부 · '첫 번째 비교 — 구조만 바꾼 비교' · '두 번째 비교 — 조건을 맞춘 비교'</sub>

> 🎯 **8%p와 0.5%p 두 간격은 각각 무엇을 말하고, LSTM과 GRU는 무엇으로 고르는가?**

💡 **쉽게 말하면**

- **한 번에 하나만 바꿔** 재고, 조건을 바꿔 **한 번 더** 잼.
- 두 번 다 같은 모양이면 성능이 아니라 **비용**으로 고름.

<img src="images/rl_compare.png" alt="HAR 정확도 · 파라미터 (강의 숫자) · 두 비교" width="714">

**실행 흐름**

1. 강의 화면 정확도 · 흔들림 폭
2. 구조만 바꾼 간격 · 크기 맞춘 간격
3. 평가 개수 · 파라미터 배율 · 시간

**먼저 예측** · LSTM과 GRU의 차이는 평가 2,947개 중 몇 개쯤일까? — 답은 코드 실행 결과 아래에

In [28]:
# 강의 화면 숫자로 두 비교 읽기 (HAR · 12 epoch · 강의 제작 환경)
# 왜 · 강의 결과를 통제된 비교 기준으로 다시 읽기 위해
acc = {"RNN 64": 82.5, "LSTM 64": 90.6, "GRU 64": 90.1, "RNN 128": 83.2, "GRU 74": 90.8}  # 정확도 %
width = {"RNN": 2.75, "LSTM": 0.95, "GRU": 1.46}           # seed 3번 흔들림 폭 %p
gap = {k: acc[k] - acc["RNN 64"] for k in ("LSTM 64", "GRU 64", "RNN 128")}  # RNN 64 기준 간격
print(f"구조만 바꿈 · RNN→LSTM {gap['LSTM 64']:.1f}%p · RNN→GRU {gap['GRU 64']:.1f}%p"  # 첫 간격
      f" · LSTM-GRU {acc['LSTM 64'] - acc['GRU 64']:.1f}%p")
print(f"크기 맞춤 · RNN 64→128 {gap['RNN 128']:.1f}%p",    # 크기 덕이 아님
      f"(RNN 흔들림 폭 {width['RNN']}%p 안) · GRU74-LSTM {acc['GRU 74'] - acc['LSTM 64']:.1f}%p")
print(f"평가 2,947개 중 0.5%p = 약 {2947 * 0.005:.0f}개",  # 차이를 개수로
      f"· 파라미터 19590 ÷ 5190 = {19590 / 5190:.2f}배")
sec = {"RNN 128": 20.1, "GRU 74": 36.8, "LSTM 64": 52.4}   # 학습 시간(초) · 제작 맥 기준
# 비용으로 고름
print("파라미터를 맞춰도 시간은 다름 :", " · ".join(f"{k} {v}초" for k, v in sec.items()))

구조만 바꿈 · RNN→LSTM 8.1%p · RNN→GRU 7.6%p · LSTM-GRU 0.5%p
크기 맞춤 · RNN 64→128 0.7%p (RNN 흔들림 폭 2.75%p 안) · GRU74-LSTM 0.2%p
평가 2,947개 중 0.5%p = 약 15개 · 파라미터 19590 ÷ 5190 = 3.77배
파라미터를 맞춰도 시간은 다름 : RNN 128 20.1초 · GRU 74 36.8초 · LSTM 64 52.4초


**결과** · 예측 정답은 약 15개 — 0.5%p

- `구조만 바꿈 · RNN→LSTM 8.1%p · RNN→GRU 7.6%p · LSTM-GRU 0.5%p` — 문을 단 두 모델이 같이 움직인 폭 · 둘 사이는 거의 없음
- `크기 맞춤 · RNN 64→128 0.7%p (RNN 흔들림 폭 2.75%p 안) · GRU74-LSTM 0.2%p` — 3.5배 넘게 키워도 RNN은 그대로 → 크기 덕이 아님
- `평가 2,947개 중 0.5%p = 약 15개 · 파라미터 19590 ÷ 5190 = 3.77배` — LSTM은 네 배 가까이 큼
- `파라미터를 맞춰도 시간은 다름 : RNN 128 20.1초 · GRU 74 36.8초 · LSTM 64 52.4초` — 부품 수가 같아도 걸리는 시간은 다름

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>구조만 바꾼 비교</td><td>hidden을 같게</td><td>구조 차이 첫 확인</td></tr>
<tr><td>조건을 맞춘 비교</td><td>파라미터 수를 같게</td><td>크기 덕인지 가르기</td></tr>
<tr><td>seed 여러 번</td><td>흔들림 폭 확인</td><td>차이가 폭보다 큰지</td></tr>
<tr><td>비용 (파라미터 · 시간)</td><td>성능이 비슷할 때의 기준</td><td>LSTM vs GRU</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 한 번씩만 돌린 값으로 'LSTM이 GRU보다 낫다'고 쓸 수 없음
> - 숫자는 강의 제작 환경 값 (파일별 2 8장에서 이 PC로 다시 잼)
> - 강의는 'GRU 74가 0.3%p 위'라고 했지만 화면 숫자로는 90.8 - 90.6 → 0.2%p (반올림 전 값의 차이로 보임)

> ✅ **한 줄 정리** · 구조만 바꾼 비교와 조건을 맞춘 비교가 같은 모양(RNN만 뒤처짐 · LSTM ≈ GRU)이면, 구조가 다르다고 성능이 다른 것이 아니며 비용으로 고른다.

**딥러닝 코드에서는**

- 순서 · RNN으로 먼저 → 안 되면 문 달린 쪽 → 여유가 없으면 GRU부터

### 7-3. `B` 양방향 (Bidirectional) · `bidirectional=True` <sub>강의 7부 · '이렇게 읽는 것에 붙은 이름' · '못 쓰는 자리'</sub>

> 🎯 **양방향은 무엇을 더 읽고, 언제 쓸 수 없는가?**

💡 **쉽게 말하면**

- 같은 줄을 **앞 → 뒤 · 뒤 → 앞으로 한 번씩** 읽어 칸마다 두 요약을 가짐.
- 뒤 → 앞은 **끝에서 출발**하므로 끝이 아직 없는 실시간 판정에는 못 씀.

**실행 흐름**

1. 양방향 RNN 출력 · h_n 모양
2. 뒤 방향 끝이 0번 칸인지
3. 실시간 3칸 예

**먼저 예측** · 양방향 RNN(hidden 32)의 output 마지막 축은? — 답은 코드 실행 결과 아래에

In [29]:
# 양방향: 같은 줄을 앞 → 뒤 · 뒤 → 앞으로 한 번씩 읽음
# 왜 · 양방향 RNN이 출력을 어떻게 만드는지 확인하기 위해
torch.manual_seed(SEED)                                    # 같은 시작점
bi = nn.RNN(10, 32, bidirectional=True, batch_first=True)  # 양방향
out, h_n = bi(to_tensor(xtr[:8]))                          # 칸마다 두 방향 요약
print("output", tuple(out.shape), "= 32 × 2 · h_n", tuple(h_n.shape), "= 방향 2")  # 앞 32 · 뒤 32
# 끝에서 출발
print("뒤 방향의 끝은 0번 칸 : out[:, 0, 32:] == h_n[1] →", torch.allclose(out[:, 0, 32:], h_n[1]))


# 실시간이면: 뒤에서 앞으로 읽을 끝이 아직 없음 (지금까지 3칸만 들어옴)
# 왜 · 실시간 예측에는 양방향을 쓸 수 없는 이유를 확인하기 위해
stream = xtr[0][:3]  # 지금까지 들어온 값
# 쓸 수 없는 자리
print("지금까지", stream, "→ 앞 → 뒤 RNN 은 출력 가능 · 양방향은 끝을 기다려야 함")

output (8, 10, 64) = 32 × 2 · h_n (2, 8, 32) = 방향 2
뒤 방향의 끝은 0번 칸 : out[:, 0, 32:] == h_n[1] → True
지금까지 [2, 3, 1] → 앞 → 뒤 RNN 은 출력 가능 · 양방향은 끝을 기다려야 함


**결과** · 예측 정답은 64 — 두 방향을 이어 붙임

- `output (8, 10, 64) = 32 × 2 · h_n (2, 8, 32) = 방향 2` — 칸마다 앞 32 + 뒤 32
- `뒤 방향의 끝은 0번 칸 : out[:, 0, 32:] == h_n[1] → True` — 뒤 → 앞은 0번 칸에서 끝남
- `지금까지 [2, 3, 1] → 앞 → 뒤 RNN 은 출력 가능 · 양방향은 끝을 기다려야 함` — 성능 문제가 아니라 쓸 수 없는 자리

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>한 방향</td><td>지금까지만 읽음</td><td>실시간 · 다음 값 예측</td></tr>
<tr><td>양방향</td><td>줄 전체를 두 번 읽음</td><td>줄이 다 들어온 뒤 (HAR 창 · 문장 분류)</td></tr>
<tr><td>양방향 + <code>output[:, -1]</code></td><td>뒤 방향은 1칸만 읽은 값</td><td>쓰지 않음 · <code>h_n</code> 두 방향을 이어 씀</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 양방향은 순서를 버리는 게 아니라 순서를 두 번 쓰는 것

> ✅ **한 줄 정리** · 양방향은 같은 줄을 앞뒤로 한 번씩 읽어 output이 두 배가 되며, 줄 전체가 손에 있을 때만 쓸 수 있다.

**딥러닝 코드에서는**

- `torch.cat([h_n[-2], h_n[-1]], dim=1)` — 양방향 분류의 마지막 요약
- Seq2Seq · Attention · Transformer는 다음 과정

---
## 8. 종합 실습 · 아홉 단계로 다시 풀기

> **이 장의 흐름**
>
> - 강의 7부 '마지막 30분'의 아홉 단계를 3과 7 과제에 처음부터 다시 적용
> - 1 ~ 7장 코드를 그대로 써서 단계마다 숫자 하나로 확인

<img src="images/rl_steps.png" alt="아홉 단계 · 오늘의 장면" width="733">

**파이프라인 위치** · ① 문제 정의 → ⑥ 평가

> 🎯 **아홉 단계를 따라가면 3과 7 과제의 어디가 막히고, 무엇으로 풀리는가?**

💡 **쉽게 말하면**

- 막히면 **그 단계 번호가 지금 자리**.
- 섞어 보기 → 기억할 것 적기 → 사람이 정해 보기 → 학습으로 → 모양 → 거리 → 원인 → 문 달린 것 비교.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1 ~ 4단계</td><td>한 샘플 · 길이 · 순서 섞기 · 기억할 것 · 사람이 정한 요약의 상한</td><td>1-2 · 1-3 · 2-3</td></tr>
<tr><td>5 · 6단계</td><td>넘길 것을 학습으로 · 입력 (N, 10, 10) → 출력 (N, 2) · RNN 학습</td><td>2-4 · 3-2</td></tr>
<tr><td>7단계</td><td>거리 98 · 20 epoch에서 닿는가</td><td>5장</td></tr>
<tr><td>8단계</td><td>기억 탓인가 학습 탓인가 · 기울기</td><td>5-3</td></tr>
<tr><td>9단계</td><td>문 달린 것 · 셀 종류와 문 시작값만 바꾼 통제된 비교</td><td>6-4 · 7-2</td></tr>
</tbody></table>

**먼저 예측** · 거리 98 · 20 epoch에서 RNN · LSTM 기본 · LSTM 열어둠 · GRU 중 누가 닿을까? — 답은 코드 실행 결과 아래에

In [30]:
# 1 ~ 3단계: 한 샘플 · 길이 → 순서를 섞어봄 → 무엇을 기억할지
# 왜 · 앞에서 배운 점검 순서를 새 과제에 그대로 적용하기 위해
# 길이 10 고정
print("1단계 · 한 샘플 = 숫자", len(xtr[0]), "칸 · 길이 고정 :", len(set(map(len, xtr))) == 1)
rng = random.Random(1)                                              # 섞기 전용
changed = sum(int(s.index(3) < s.index(7)) != y                     # 섞은 뒤 답이 바뀐 줄
              for s, y in zip((rng.sample(s, 10) for s in xte), yte))
print(f"2단계 · 칸 순서를 섞으면 답이 바뀌는 줄 {changed / len(xte):.3f} → 순서가 중요")  # 절반가량
print("3단계 · 기억할 것 = 3 · 7 중 무엇을 먼저 봤는가 (상태 셋)")  # 2-3의 결론
# 1-2 함수
print(f"4단계 · 사람이 정한 요약(합)의 최고 정확도 {best_by_sum(xtr, ytr, xte, yte):.3f}")

1단계 · 한 샘플 = 숫자 10 칸 · 길이 고정 : True
2단계 · 칸 순서를 섞으면 답이 바뀌는 줄 0.487 → 순서가 중요
3단계 · 기억할 것 = 3 · 7 중 무엇을 먼저 봤는가 (상태 셋)
4단계 · 사람이 정한 요약(합)의 최고 정확도 0.501


**실행 흐름**

1. 학습 → 평가 정확도를 돌려주는 `fit()` 정의
2. 입력 (N, 10, 10) → 출력 (N, 2) 모양 확인
3. RNN을 30 epoch 학습한 평가 정확도 출력

In [31]:
# 5 · 6단계: 넘길 것을 학습으로 · 입력 (N, 10, 10) → 출력 (N, 2)
# 왜 · 손으로 정한 상태 대신 학습으로 넘길 것을 찾게 하기 위해
# 왜 · 모든 비교가 같은 학습 절차 · 같은 평가 방법을 쓰도록 한 함수로 묶기 위해
def fit(model, X, Y, Xe, Ye, epochs, lr=3e-3, bs=128):                 # 학습 → 평가 정확도
    model, X, Y, Xe, Ye = model.to(DEV), X.to(DEV), Y.to(DEV), Xe.to(DEV), Ye.to(DEV)  # 같은 장치로
    opt = torch.optim.Adam(model.parameters(), lr=lr)                  # Adam
    for _ in range(epochs):                                            # epoch 반복
        perm = torch.randperm(len(X))                                  # 순서 섞기
        for i in range(0, len(X), bs):                                 # 미니배치
            j = perm[i:i + bs].to(DEV)                                 # 이번 배치
            opt.zero_grad()                                            # 지난 기울기 지우기
            nn.functional.cross_entropy(model(X[j]), Y[j]).backward()  # 손실 → 기울기
            opt.step()                                                 # 가중치 수정
    with torch.no_grad():                                              # 평가
        return (model(Xe).argmax(1) == Ye).float().mean().item()       # 정확도


# 왜 · 학습 · 평가 데이터를 모델 입력 모양으로 바꾸기 위해
Xa, Xb = to_tensor(xtr), to_tensor(xte)             # (6000, 10, 10) · (2000, 10, 10)
# 세 겹 → 점수 2개
print("6단계 · 입력", tuple(Xa.shape), "→ 출력", tuple(Net("RNN")(Xa[:2]).shape))
torch.manual_seed(SEED)                             # 같은 시작점
acc = fit(Net("RNN"), Xa, torch.tensor(ytr), Xb, torch.tensor(yte), epochs=30)  # 강의와 같은 설정
print(f"5단계 · RNN 학습 → 평가 정확도 {acc:.4f}")  # 요약을 학습으로

6단계 · 입력 (6000, 10, 10) → 출력 (2, 2)


5단계 · RNN 학습 → 평가 정확도 1.0000


**실행 흐름**

1. 거리 98 학습 · 평가 데이터 만들기
2. 셀 종류 · forget 문 시작값만 바꿔 같은 예산(20 epoch)으로 학습
3. 8단계 기울기 · 9단계 정확도 차이 출력

In [32]:
# 7 ~ 9단계: 거리 98 · 같은 예산(20 epoch)에서 셀 종류 · 문 시작값만 바꾼 통제된 비교
# 왜 · 멀리 기억해야 하는 과제에서 셀 종류 · 문 시작값만 바꿔 같은 예산으로 비교하기 위해
xa, ya = make_front_task(4000, 100, seed=1)                               # 학습 4000줄 · 거리 98
xb, yb = make_front_task(1000, 100, seed=2)                               # 평가 1000줄
Xa, Ya, Xb, Yb = to_tensor(xa), torch.tensor(ya), to_tensor(xb), torch.tensor(yb)  # 텐서
res = {}                                                                  # 이름별 정확도
for name, kind, fb in [("RNN", "RNN", None), ("LSTM 기본", "LSTM", 0.0),  # 비교 4가지
                       ("LSTM 열어둠", "LSTM", 2.0), ("GRU", "GRU", None)]:
    torch.manual_seed(SEED)                                               # 같은 시작점
    model = Net(kind)                                                     # 셀 종류만 바꿈
    if fb is not None:  # LSTM이면 forget 문 시작값 (6-4)
        with torch.no_grad():                                             # 값만 넣음
            model.r.bias_ih_l0[32:64].fill_(fb)                           # forget 구간
            model.r.bias_hh_l0[32:64].fill_(0.0)                          # 한쪽은 0
    res[name] = fit(model, Xa, Ya, Xb, Yb, epochs=20)                     # 같은 데이터 · 같은 예산
    print(f"7단계 · 거리 98 · 20 epoch · {name} → {res[name]:.3f}")       # 닿는가
# 신호가 안 옴
print("8단계 · 학습 전 RNN 기울기 0 · 10 · 20스텝 :", [f"{v:.1e}" for v in grad_profile("RNN")[:3]])
# 문이 열려 있을 때 차이
print(f"9단계 · 열어둔 LSTM - RNN = {res['LSTM 열어둠'] - res['RNN']:+.3f}")

7단계 · 거리 98 · 20 epoch · RNN → 0.517


7단계 · 거리 98 · 20 epoch · LSTM 기본 → 0.512


7단계 · 거리 98 · 20 epoch · LSTM 열어둠 → 1.000


7단계 · 거리 98 · 20 epoch · GRU → 0.496
8단계 · 학습 전 RNN 기울기 0 · 10 · 20스텝 : ['5.2e-02', '1.1e-04', '2.9e-07']
9단계 · 열어둔 LSTM - RNN = +0.483


**결과** · 예측 정답은 LSTM 열어둠 하나 — 나머지는 찍기 근처

- `2단계 · 칸 순서를 섞으면 답이 바뀌는 줄 0.487 → 순서가 중요` — 절반가량이 바뀜
- `4단계 · 사람이 정한 요약(합)의 최고 정확도 0.501` — 합으로는 상한이 찍기
- `6단계 · 입력 (6000, 10, 10) → 출력 (2, 2)` — 세 겹 입력 → 줄마다 점수 2개
- `5단계 · RNN 학습 → 평가 정확도 1.0000` — 길이 10에서는 학습한 요약으로 만점 (강의도 만점)
- `7단계 · 거리 98 · 20 epoch · RNN → 0.517` — 강의와 같은 51.7% · 닿지 못함
- `7단계 · 거리 98 · 20 epoch · LSTM 기본 → 0.512` — 문만 달아서는 안 됨 (6-3)
- `7단계 · 거리 98 · 20 epoch · LSTM 열어둠 → 1.000` — 문을 열어 두니 같은 예산에서 닿음 (6-4)
- `7단계 · 거리 98 · 20 epoch · GRU → 0.496` — GRU도 이 예산에서는 못 닿음
- `8단계 · 학습 전 RNN 기울기 0 · 10 · 20스텝 : ['5.2e-02', '1.1e-04', '2.9e-07']` — 맨 앞을 고칠 신호가 거의 안 옴 → 학습 탓
- `9단계 · 열어둔 LSTM - RNN = +0.483` — 셀 종류와 문 시작값 하나만 바꿔 생긴 차이

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>RNN</td><td>남길 양 손잡이 없음</td><td>짧은 거리 · 먼저 돌려 볼 것</td></tr>
<tr><td>LSTM 기본 (bias 0)</td><td>문이 반쯤 닫힌 채 시작</td><td>기본 초기화 그대로</td></tr>
<tr><td>LSTM 열어둠 (bias 2)</td><td>기억을 남긴 채 시작</td><td>먼 거리 의존</td></tr>
<tr><td>GRU</td><td>손잡이 2개 · 가벼움</td><td>LSTM과 비슷할 때 비용으로</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 이 결과는 같은 예산(20 epoch) 비교 · 더 오래 학습하면 RNN도 닿음 (강의 60 epoch 100%)
> - GPU · 환경이 다르면 학습 결과 숫자가 조금 다를 수 있음

> ✅ **한 줄 정리** · 아홉 단계로 물으면 3과 7 과제는 순서가 중요하고, 학습한 요약이 필요하며, 먼 거리는 학습 탓이고, 문을 열어 둔 LSTM이 같은 예산에서 닿는다.

**딥러닝 코드에서는**

- 새 데이터 · 새 모델도 같은 순서 — 섞기 검사 → 기준선 → 학습 모델 → shape → 거리 · 기울기 진단 → 통제된 비교

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 합 · 마지막 값의 한계</td><td>합의 상한 0.501 · 마지막 값 0.2 × 1 + 0.8 × 0.5 = 0.6</td><td>1-2 · 2-1</td></tr>
<tr><td>2. hidden state · 공유</td><td>길이 10 · 200 모두 파라미터 1,474개 · cell + for 문 = nn.RNN (차이 0.0)</td><td>2-4 · 3-1 · 3-2</td></tr>
<tr><td>3. 모양</td><td><code>Expected 10, got 1</code> → one_hot · batch_first 끄면 h_n (1, 10, 32) · 2층 양방향 h_n (4, 8, 32)</td><td>3-3 ~ 3-5</td></tr>
<tr><td>4. padding</td><td>-1 == lens - 1 : [False, True, False] · mask 평균</td><td>4장</td></tr>
<tr><td>5. 소실 · LSTM · GRU</td><td>20칸 17만 배 · LSTM 기본 99칸 0 · 열어둠 1.6e-05 · LSTM ≈ GRU</td><td>5 · 6 · 7장</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>순서와 질문</td><td>순서가 중요한지는 질문이 정함 · 섞어서 확인</td><td>평균 풀링 전에 질문 확인</td></tr>
<tr><td><code>A</code></td><td>hidden state</td><td>매 칸 새로 쓰는 고정 크기 요약</td><td><code>h_t = f(h_(t-1), x_t)</code></td></tr>
<tr><td><code>A</code></td><td>파라미터 공유</td><td>같은 상자를 칸마다 다시 씀</td><td>길이가 달라도 같은 모델</td></tr>
<tr><td><code>A</code></td><td>세 겹 모양</td><td>(개수, 길이, 표현) · batch_first</td><td><code>nn.RNN(I, H, batch_first=True)</code></td></tr>
<tr><td><code>A</code></td><td>padding</td><td>lens와 함께 · -1 대신 lens - 1</td><td><code>pack_padded_sequence</code></td></tr>
<tr><td><code>A</code></td><td>기울기 소실</td><td>반복 곱 · 출발점에서 신호가 안 옴</td><td><code>clip_grad_norm_</code> (폭발)</td></tr>
<tr><td><code>A</code></td><td>LSTM 문</td><td>열려 있을 때만 기울기가 남음</td><td>forget bias 1 ~ 2 초기화</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>B</code></td><td>통제된 비교</td><td>하나만 바꾸고 · 조건 맞춰 한 번 더 · 비용</td><td>정확도 + 파라미터 + 시간</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · 합 · 평균이 순서만 다른 두 줄을 못 가르는 이유는?
- **1-2** · 질문 A가 합으로 48.8%인데 질문 B가 96%인 이유는?
- **1-3** · 순서가 중요한지 확인하는 방법은?
- **2-1** · 마지막 값 하나가 60% 근처인 이유는?
- **2-2** · 전부 펼치기가 길어지면 무너지는 이유는?
- **2-3** · '봤는가'가 안 되는 이유는?
- **2-4** · 파라미터 공유란?
- **3-2** · `output`과 `h_n`의 차이는?
- **3-3** · `Expected 10, got 1` 에러를 고치는 법은?
- **3-4** · `batch_first=False`로 (8, 10, 10)을 넣으면?
- **4-3** · 패딩한 배치에서 마지막 요약은?
- **5-3** · 기울기 소실이란?
- **5-5** · 폭발 · 소실의 대처는?
- **6-3** · LSTM 기본도 99칸에서 0인 이유는?
- **7-2** · LSTM과 GRU는 무엇으로 고르나?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-5** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**6-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**7-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `nn.RNNCell(4, 3)`을 만들고 `(2, 5, 4)` 입력을 for 문으로 펼쳐 마지막 `h`의 모양 출력
2. 같은 모양의 `nn.RNN(4, 3, num_layers=2, bidirectional=True, batch_first=True)` 출력 · h_n 모양 출력

In [33]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 3과 7 과제를 길이 20으로 바꾸면 마지막 값 하나의 기대 정확도는?
2. hidden 16인 `nn.RNN(10, 16)` + `nn.Linear(16, 2)`의 파라미터 수는?
3. 한 칸당 기울기 비율이 0.54이면 20칸 거슬러 갔을 때 대략 몇 배 작아지나?

In [34]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · 순서를 보지 않는 계산이라 재료가 같으면 같은 값
- **1-2** · A는 합에 답이 없고 B는 합이 답을 정함 · 질문이 정함
- **1-3** · 한 줄 안에서 칸 순서를 섞어 성능이 떨어지는지
- **2-1** · 끝 칸이 3 · 7인 2할만 확정 · 0.2 × 1 + 0.8 × 0.5
- **2-2** · 길이만큼 모델이 커지는데 데이터는 그대로
- **2-3** · 3 · 7이 항상 둘 다 있어 입력이 늘 같음 · 필요한 건 먼저 봤는가
- **2-4** · 같은 상자(가중치)를 칸마다 다시 써서 길이와 무관
- **3-2** · 매 칸 요약 전부 vs 마지막 요약
- **3-3** · 숫자를 one_hot으로 열 칸에 적기
- **3-4** · 8을 길이로 읽어 h_n (1, 10, 32) · 에러 없음
- **4-3** · `output[idx, lens - 1]` (또는 pack + h_n)
- **5-3** · 거슬러 갈수록 신호가 작아져 출발점에서 맨 앞을 고칠 신호가 거의 없음
- **5-5** · 폭발은 clipping · 소실은 구조(LSTM · GRU · 문 열기)
- **6-3** · forget 문 bias 0 → 문 0.5를 백 번 곱함
- **7-2** · 두 비교에서 성능이 비슷 → 파라미터 · 시간

#### 코드 문제 답

```python
import torch
import torch.nn as nn

cell, x = nn.RNNCell(4, 3), torch.randn(2, 5, 4)
h = torch.zeros(2, 3)
for t in range(5):
    h = cell(x[:, t], h)
print(tuple(h.shape))  # (2, 3)
out, h_n = nn.RNN(4, 3, num_layers=2, bidirectional=True, batch_first=True)(x)
print(tuple(out.shape), tuple(h_n.shape))  # (2, 5, 6) (4, 2, 3)
```

#### 변형 문제 답

*문제 → 답*

- **1** · 끝 칸이 3 · 7일 확률 2/20 = 0.1 → 0.1 × 1 + 0.9 × 0.5 = 0.55
- **2** · 16 × 10 + 16 × 16 + 16 + 16 + 16 × 2 + 2 = 482
- **3** · 0.54의 20제곱 ≈ 4.4e-06 → 약 23만 배 (실측 17만 배와 같은 크기)

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) 패딩한 배치에서 <code>output[:, -1]</code>로 분류</td><td>모든 줄 길이가 같다고 착각</td><td><code>output[idx, lens - 1]</code> · mask · pack</td><td>이 배치에 빈 칸이 있나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. RNN의 hidden state와 파라미터 공유를 설명해 주세요.**
> - hidden state는 매 시점 이전 요약과 새 입력으로 새로 계산하는 고정 크기 요약입니다.
> - 같은 가중치를 모든 시점에 다시 쓰는 것이 파라미터 공유이고, 그래서 파라미터 수가 길이와 무관하며 학습 때보다 긴 시퀀스도 넣을 수 있습니다.

**Q2. 패딩한 배치에서 주의할 점은?**
> - 가장 긴 길이에 맞춰 채우면 마지막 인덱스는 틀의 끝이지 데이터의 끝이 아닙니다.
> - 길이표를 함께 들고 다니며 줄마다 lens - 1을 꺼내거나, 평균은 mask로 빈 칸을 빼고, PyTorch에서는 pack_padded_sequence를 씁니다.

**Q3. 기울기 소실이 왜 생기고 어떻게 대처하나요?**
> - 역전파가 시간 축을 한 칸씩 거슬러 가며 비슷한 비율이 반복해 곱해지므로 1보다 작으면 지수적으로 작아집니다.
> - 폭발은 gradient clipping으로 막지만 소실은 구조를 바꿔야 하며, LSTM은 forget 문이 열려 있을 때 메모 경로로 신호를 크게 늦춰 전달합니다.

**Q4. LSTM과 GRU는 어떻게 고르나요?**
> - hidden을 같게 한 비교와 파라미터 수를 맞춘 비교를 seed 여러 번으로 해 보고, 둘 다 차이가 흔들림 폭 안이면 성능이 아니라 파라미터 수와 학습 시간으로 고릅니다.
> - 강의 HAR에서는 0.5%p 차이라 더 가벼운 GRU부터 시도할 만했습니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>sum()</code> · <code>sorted()</code></td><td>Python 내장</td><td>순서를 안 보는 계산 · 재료 비교</td><td>1-1 <code>sorted(a) == sorted(b)</code></td></tr>
<tr><td><code>defaultdict(Counter)</code></td><td>collections</td><td>합별 다수결 표 (상한 계산)</td><td>1-2 <code>table[sum(s)][y] += 1</code></td></tr>
<tr><td><code>random.sample()</code></td><td>random</td><td>한 줄 안 칸 순서 섞기</td><td>1-3 <code>rng.sample(s, len(s))</code></td></tr>
<tr><td><code>nn.RNN</code></td><td>torch.nn</td><td>칸마다 같은 상자 · output, h_n</td><td>2-4 <code>layer(10, hid, batch_first=True)</code></td></tr>
<tr><td><code>nn.RNNCell</code></td><td>torch.nn</td><td>상자 하나 · for 문으로 펼침</td><td>3-1 <code>h = cell(x[:, t, :], h)</code></td></tr>
<tr><td><code>F.one_hot()</code></td><td>torch.nn.functional</td><td>번호 → 열 칸</td><td>3-3 <code>F.one_hot(digits, num_classes=10)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>batch_first</code></td><td>nn.RNN 인자</td><td>앞 두 겹 순서</td><td>3-4 <code>nn.RNN(10, 32, batch_first=bf)</code></td></tr>
<tr><td><code>torch.equal()</code> · <code>torch.allclose()</code></td><td>torch</td><td>비트 비교 · 오차 허용 비교</td><td>3-5 <code>torch.equal(output[:, -1, :], h_n[0])</code></td></tr>
<tr><td><code>output[idx, lens - 1]</code></td><td>인덱싱</td><td>줄마다 진짜 끝</td><td>4-3 <code>output[idx, lens - 1]</code></td></tr>
<tr><td><code>retain_grad()</code></td><td>Tensor 메서드</td><td>중간 요약의 기울기 남기기</td><td>5-3 <code>h.retain_grad()</code></td></tr>
<tr><td><code>clip_grad_norm_()</code></td><td>torch.nn.utils</td><td>기울기 폭발 막기</td><td>5-5 <code>nn.utils.clip_grad_norm_([w], max_norm=5.0)</code></td></tr>
<tr><td><code>torch.sigmoid()</code></td><td>torch</td><td>문 값 0 ~ 1</td><td>6-1 <code>torch.sigmoid(z)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>nn.LSTM</code></td><td>torch.nn</td><td>output, (h_n, c_n)</td><td>6-2 <code>output, (h_n, c_n) = lstm(x)</code></td></tr>
<tr><td><code>bias_ih[H:2*H]</code></td><td>LSTMCell 파라미터</td><td>forget 문 시작값</td><td>6-4 <code>cell.bias_ih[hid:2 * hid].fill_(forget_bias)</code></td></tr>
<tr><td><code>nn.GRU</code></td><td>torch.nn</td><td>손잡이 2개 · output, h_n</td><td>7-1 <code>nn.GRU(10, 32, batch_first=True)</code></td></tr>
<tr><td><code>bidirectional=True</code></td><td>nn.RNN 인자</td><td>앞뒤 두 번 읽기</td><td>7-3 <code>nn.RNN(10, 32, bidirectional=True, batch_first=True)</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>새 데이터는 칸 순서를 섞어 먼저 검사</td><td>평균 풀링부터 쓰기</td></tr>
<tr><td><code>batch_first=True</code> + 입력 shape 출력</td><td>기본값(False)에 (N, L, I) 넣기</td></tr>
<tr><td>padding하면 lens를 같이 들고 다니기</td><td><code>output[:, -1]</code>로 패딩 배치 분류</td></tr>
<tr><td>칸별 기울기는 cell + <code>retain_grad()</code>로</td><td><code>output.grad</code>로 소실 판단</td></tr>
<tr><td>LSTM forget bias를 열어 두고 시작</td><td>'LSTM이 소실을 해결했다'고 말하기</td></tr>
<tr><td>비교는 하나만 바꾸고 seed 여러 번</td><td>한 번 돌린 0.5%p로 우열 결론</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td><code>output</code> vs <code>h_n</code></td><td>매 칸 요약 vs 마지막 요약 (층 × 방향)</td><td>칸마다 답이면 output</td></tr>
<tr><td><code>-1</code> vs <code>lens - 1</code></td><td>틀의 끝 vs 데이터의 끝</td><td>패딩이면 lens - 1</td></tr>
<tr><td><code>RNNCell</code> vs <code>RNN</code></td><td>한 칸 + for vs 판 전체</td><td>측정은 Cell · 학습은 RNN</td></tr>
<tr><td>기억 탓 vs 학습 탓</td><td>앞 → 뒤 차이 vs 뒤 → 앞 기울기</td><td>둘을 따로 잼</td></tr>
<tr><td>LSTM vs GRU</td><td>손잡이 3 · 줄 2 vs 손잡이 2 · 줄 1</td><td>성능 비슷 → 비용</td></tr>
<tr><td>폭발 vs 소실</td><td>NaN · 눈에 띔 vs 조용히 못 배움</td><td>clipping vs 구조</td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>15분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 · 그림을 가리고 output · h_n 모양 말하기</td><td>20분</td></tr>
<tr><td>7일 뒤</td><td>8장 아홉 단계를 빈 셀에 직접 다시 짜기</td><td>30분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 파일별 2(rnn.ipynb) 시작 전 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 · `rnn_lecture.pdf` (HTML 판으로 텍스트 확인) 7부를 순서 그대로 7장 25섹션으로
- 슬라이드의 예측 문제 ① ~ ⑤는 각 섹션의 '먼저 예측'으로
- 슬라이드 숫자 중 학습이 필요 없는 것은 코드로 다시 계산 (강의 노트북과 같은 seed · 같은 난수 순서)
- 학습이 필요한 정확도는 강의 값으로 적고 출처 표시 · 직접 학습은 파일별 2 · 8장만 짧게
- 마지막 30분의 아홉 단계를 8장 종합 실습으로 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)
- 슬라이드의 띄어 쓴 조사(`RNN 이` · `8%p 를` · `3 과 7`)는 붙여 씀 · 강의 숫자는 계산으로 다시 확인

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>1부 · 순서가 중요하다</td><td>두 줄 · 합 48.8% · 질문 B 96% · 섞기 · 길이 갈래</td><td>1-1 ~ 1-3</td></tr>
<tr><td>2부 · 무엇을 넘길 것인가</td><td>마지막 값 60.4% · 0.2 × 1 + 0.8 × 0.5 · 펼치기 · 봤는가 51.2% · 상태 셋</td><td>2-1 ~ 2-3</td></tr>
<tr><td>2부 · hidden state · 파라미터 공유</td><td>h_t = f(h_(t-1), x_t) · 1,474개 · CNN과 같은 발상</td><td>2-4</td></tr>
<tr><td>3부 · 코드로 옮기면</td><td>RNNCell · for 세 줄 · nn.RNN 0.0 · output / h_n · 학습 100%</td><td>3-1 · 3-2</td></tr>
<tr><td>3부 · 에러 · 세 겹 · batch_first</td><td>Expected 10, got 1 · one_hot · 껐을 때가 더 무서움</td><td>3-3 · 3-4</td></tr>
<tr><td>3부 · 정말 같은 값일까</td><td>equal · allclose · 그대로 쓸 수 있는 조건 · 1 = 층 × 방향</td><td>3-5</td></tr>
<tr><td>3부 · UCI HAR</td><td>[7352, 128, 9] · 9 = 3 × 3 · 2.56초</td><td>3-6</td></tr>
<tr><td>3부 · padding</td><td>직사각형 · 세 방법 · 빈 벡터 vs 0 · lens · -1 vs lens - 1 · 51.4% · mask · 4~12</td><td>4-1 ~ 4-3</td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>4부 · 넘긴 것이 안 닿는다</td><td>거리 8 ~ 398 · 60 epoch · 귓속말 · float32 바닥 · 손으로 심은 가중치</td><td>5-1 · 5-2 · 5장 그림</td></tr>
<tr><td>4부 · 기울기</td><td>retain_grad · 486배 · 17만 배 · 77스텝 · 3.38e-28 · BPTT · output.grad</td><td>5-3 · 5-4</td></tr>
<tr><td>4부 · 같은 수를 계속 곱하면</td><td>0.9를 50번 · 1.1 · clipping · 1.10 · 0.57 · 0.54</td><td>5-5</td></tr>
<tr><td>5부 · 남길 것을 학습시키자</td><td>수도꼭지 · sigmoid · cell state · 세 문 · LSTM 세 개</td><td>6-1 · 6-2</td></tr>
<tr><td>5부 · 정말 살아날까</td><td>LSTM 기본 0 · 0.4997 · 문을 열어두면 · 활짝 · 3,303배 · HAR 82.5 vs 90.6</td><td>6-3 · 6-4</td></tr>
<tr><td>6부 · 더 적은 손잡이</td><td>GRU · 구조만 바꾼 비교 · 조건을 맞춘 비교 · 시간 · 비용</td><td>7-1 · 7-2</td></tr>
<tr><td>7부 · 남는 것</td><td>아홉 단계 · 섞기 3.6%p · 양방향 · 세 문장 · Seq2Seq 등은 다음 과정</td><td>7-3 · 8장 · 마무리</td></tr>
</tbody></table>

### 전체 코드

1 ~ 8장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 순서만 다른 두 줄
2. 1-2 질문이 정하는 순서의 중요성
3. 1-3 섞기 검사 · 길이 문제

In [35]:
# ---------- 1-1 ----------
# 강의 첫 두 줄: 들어 있는 숫자는 같고 순서만 다름
# 왜 · 합 · 정렬처럼 순서를 버리는 요약으로는 두 줄을 구분할 수 없음을 보이기 위해
a = [9, 3, 1, 0, 7, 5, 8, 6, 8, 4]                      # 3이 두 번째 · 7이 다섯 번째 → 네
b = [9, 7, 1, 0, 3, 5, 8, 6, 8, 4]                      # 7이 두 번째 · 3이 다섯 번째 → 아니요
# 순서 안 보는 계산
for name, f in [("합", sum), ("평균", lambda s: sum(s) / len(s)), ("최댓값", max), ("개수", len)]:
    print(f"{name:<4} {f(a)} vs {f(b)}")                # 둘 다 같음
print("재료(정렬)가 같은가 :", sorted(a) == sorted(b),  # 재료 비교
      "· 8 의 개수 :", Counter(a)[8], Counter(b)[8])
ans_a, ans_b = int(a.index(3) < a.index(7)), int(b.index(3) < b.index(7))  # 3이 먼저면 1
print("3 의 자리 :", a.index(3), "vs", b.index(3), "→ 답", ans_a, ans_b)  # 순서만 다름


# ---------- 1-2 ----------
# 합 하나만 보고 낼 수 있는 최고 정확도: 학습에서 합마다 많은 쪽 답을 외워 평가에 적용
# 왜 · 순서 정보 없이 얻을 수 있는 점수의 상한을 기준으로 두기 위해
def best_by_sum(xs, ys, xs_e, ys_e):                            # 합 → 다수결 표
    table = defaultdict(Counter)                                # 합마다 정답 개수
    for s, y in zip(xs, ys):                                    # 학습 줄
        table[sum(s)][y] += 1                                   # 합 칸에 정답 세기
    pred = [table[sum(s)].most_common(1)[0][0] if sum(s) in table else 0 for s in xs_e]  # 다수결 답
    return sum(p == y for p, y in zip(pred, ys_e)) / len(ys_e)  # 정확도


# 왜 · 합이 정답을 정하는 다른 질문과 비교해 순서가 중요한지는 질문이 정한다는 것을 보이기 위해
q45 = [int(sum(s) >= 45) for s in xtr]                      # 질문 B: 합이 45 이상인가 (학습)
q45e = [int(sum(s) >= 45) for s in xte]                     # 질문 B (평가)
# 찍기 근처
print(f"질문 A '3 다음에 7인가'  · 합으로 낼 수 있는 최고 {best_by_sum(xtr, ytr, xte, yte):.3f}")
# 완벽
print(f"질문 B '합이 45 이상인가' · 합으로 낼 수 있는 최고 {best_by_sum(xtr, q45, xte, q45e):.3f}")
c51 = Counter(y for s, y in zip(xtr, ytr) if sum(s) == 51)  # 합 51인 줄의 정답 개수
print("같은 합 51 인 학습 줄의 정답 개수 :", dict(sorted(c51.items())))  # 반반


# ---------- 1-3 ----------
# 새 데이터를 만나면 먼저: 한 줄 안에서 칸 순서를 섞어 답이 바뀌는지
# 왜 · 순서가 중요한 데이터인지 섞어 보기로 먼저 확인하기 위해
rng = random.Random(0)                                 # 섞기 전용 난수
shuffled = [rng.sample(s, len(s)) for s in xte]        # 줄마다 칸 순서만 섞음 (줄끼리는 그대로)
qa = [int(s.index(3) < s.index(7)) for s in shuffled]  # 질문 A 답 다시 계산
qb = [int(sum(s) >= 45) for s in shuffled]             # 질문 B 답 다시 계산
# 절반가량
print(f"질문 A · 섞은 뒤 답이 바뀐 줄 {sum(x != y for x, y in zip(qa, yte)) / len(yte):.3f}")
qb0 = [int(sum(s) >= 45) for s in xte]                 # 섞기 전 질문 B 답
print(f"질문 B · 섞은 뒤 답이 바뀐 줄 {sum(x != y for x, y in zip(qb, qb0))}")  # 0줄


# 두 번째 갈래: 길이가 매번 다르면 입력 칸을 몇 개로 고정하나
# 왜 · 길이가 다른 입력을 고정 크기 모델에 넣기 어렵다는 문제를 보이기 위해
clicks = {"A": 3, "B": 15, "C": 7, "D": 22}  # 사용자별 클릭 수 (예)
for user, n in clicks.items():               # 15칸으로 고정했다면
    # 채우거나 잘라야 함
    print(f"사용자 {user} · 클릭 {n:>2} → 빈 칸 {max(0, 15 - n):>2} · 넘침 {max(0, n - 15)}")

합    51 vs 51
평균   5.1 vs 5.1
최댓값  9 vs 9
개수   10 vs 10
재료(정렬)가 같은가 : True · 8 의 개수 : 2 2
3 의 자리 : 1 vs 4 → 답 1 0
질문 A '3 다음에 7인가'  · 합으로 낼 수 있는 최고 0.501
질문 B '합이 45 이상인가' · 합으로 낼 수 있는 최고 1.000
같은 합 51 인 학습 줄의 정답 개수 : {0: 114, 1: 120}
질문 A · 섞은 뒤 답이 바뀐 줄 0.494
질문 B · 섞은 뒤 답이 바뀐 줄 0
사용자 A · 클릭  3 → 빈 칸 12 · 넘침 0
사용자 B · 클릭 15 → 빈 칸  0 · 넘침 0
사용자 C · 클릭  7 → 빈 칸  8 · 넘침 0
사용자 D · 클릭 22 → 빈 칸  0 · 넘침 7


**실행 흐름**

1. 2-1 마지막 값 하나
2. 2-2 전부 넘기기
3. 2-3 무엇을 기억할지

In [36]:
# ---------- 2-1 ----------
# 마지막 값 하나만 넘기면: 끝 칸이 3 · 7이면 답이 확정, 나머지는 찍기
# 왜 · 마지막 값만 보면 답이 정해지는 경우와 아닌 경우를 나눠 한계를 확인하기 위해
last_hit = sum(s[-1] in (3, 7) for s in xte) / len(xte)                 # 끝 칸이 3 · 7인 비율
first_hit = sum(s[0] in (3, 7) for s in xte) / len(xte)                 # 첫 칸이 3 · 7인 비율
# 확정 + 반은 맞힘
print(f"마지막 칸이 3 · 7 : {last_hit:.3f} → 기대 정확도 {last_hit + (1 - last_hit) / 2:.3f}")
# 첫 값도 같은 이유
print(f"첫 칸이 3 · 7     : {first_hit:.3f} → 기대 정확도 {first_hit + (1 - first_hit) / 2:.3f}")
print("이론 : 0.2 × 1 + 0.8 × 0.5 =", round(0.2 * 1 + 0.8 * 0.5, 3))    # 강의 계산
print("끝 칸이 7 인 줄의 정답 :", set(y for s, y in zip(xte, yte) if s[-1] == 7),  # 항상 네
      "· 끝 칸이 3 :", set(y for s, y in zip(xte, yte) if s[-1] == 3))  # 항상 아니요


# ---------- 2-2 ----------
# 전부 넘기기 = 줄을 통째로 펼쳐 한 번에: 입력 L × 10칸짜리 MLP (은닉 32 · 32)
# 왜 · 모두 펼쳐 넣으면 길이에 따라 모델이 커지는 문제를 숫자로 보기 위해
def mlp_params(n_in, width=32):  # Linear 3개 (입력 → 32 → 32 → 2)
    return (n_in * width + width) + (width * width + width) + (width * 2 + 2)  # 가중치 + bias


# 왜 · 길이가 늘 때 파라미터가 얼마나 커지는지 비교하기 위해
for L in (10, 20, 50, 100, 200):  # 길이만 바꿈
    # 길이만큼 커짐
    print(f"길이 {L:>3} → 입력 {L * 10:>4}칸 · 파라미터 {mlp_params(L * 10):>6,}개")


# 칸마다 다른 모델이면: k번째 칸 모델은 앞 k - 1칸을 입력으로 받음
# 왜 · 칸마다 따로 모델을 두면 생기는 문제를 확인하기 위해
# 가중치 공유 불가
print("3번째 칸 모델 입력", 3 - 1, "개 · 10번째", 10 - 1, "개 · 200칸이면 모델", 200, "개")


# ---------- 2-3 ----------
# 무엇을 기억해야 하나: '봤는가' 두 칸 vs '먼저 봤는가' 상태 셋
# 왜 · 끝까지 넘겨야 할 정보가 무엇인지 상태로 정리하기 위해
seen = Counter((int(3 in s), int(7 in s)) for s in xtr)       # 봤는가 두 칸의 종류
print("'3 을 봤는가 · 7 을 봤는가' 입력 종류 :", dict(seen))  # 모든 줄이 (1, 1)


# 왜 · 줄을 앞에서부터 읽으며 상태 하나만 갱신해도 답을 낼 수 있음을 보이기 위해
def first_state(s):                     # 0 = 아직 안 봄 · 1 = 3이 먼저 · 2 = 7이 먼저
    state = 0                           # 처음엔 아무것도 안 봄
    for d in s:                         # 왼쪽부터 하나씩 읽음
        if state == 0 and d in (3, 7):  # 처음 본 표식만 기록
            state = 1 if d == 3 else 2  # 상태 갱신
    return state                        # 끝까지 읽은 뒤 상태


# 왜 · 손으로 만든 상태 규칙이 정답과 맞는지 확인하기 위해
states = [first_state(s) for s in xte]                               # 평가 줄마다 끝 상태
print("끝 상태 개수 :", dict(sorted(Counter(states).items())))       # 1 · 2만 남음
rule = sum((st == 1) == y for st, y in zip(states, yte)) / len(yte)  # 상태 1이면 네
print(f"상태 1 → 네 규칙의 정확도 : {rule:.3f}")                     # 상태 하나로 충분

마지막 칸이 3 · 7 : 0.211 → 기대 정확도 0.606
첫 칸이 3 · 7     : 0.212 → 기대 정확도 0.606
이론 : 0.2 × 1 + 0.8 × 0.5 = 0.6
끝 칸이 7 인 줄의 정답 : {1} · 끝 칸이 3 : {0}
길이  10 → 입력  100칸 · 파라미터  4,354개
길이  20 → 입력  200칸 · 파라미터  7,554개
길이  50 → 입력  500칸 · 파라미터 17,154개
길이 100 → 입력 1000칸 · 파라미터 33,154개
길이 200 → 입력 2000칸 · 파라미터 65,154개
3번째 칸 모델 입력 2 개 · 10번째 9 개 · 200칸이면 모델 200 개
'3 을 봤는가 · 7 을 봤는가' 입력 종류 : {(1, 1): 6000}
끝 상태 개수 : {1: 1024, 2: 976}
상태 1 → 네 규칙의 정확도 : 1.000


**실행 흐름**

1. 2-4 hidden state와 파라미터 공유

In [37]:
# ---------- 2-4 ----------
# 넘기는 요약의 크기를 32로 고정 → 같은 상자(같은 가중치)를 칸마다 다시 씀
# 왜 · 길이와 상관없이 같은 상자(같은 가중치)를 칸마다 쓰는 RNN을 만들기 위해
class Net(nn.Module):                              # RNN → 마지막 요약 → 네 · 아니요
    def __init__(self, kind="RNN", hid=32):        # 셀 종류 · 요약 크기
        super().__init__()                         # nn.Module 준비 (필수)
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]  # 이름으로 층 고르기
        self.r = layer(10, hid, batch_first=True)  # 입력 10 · 요약 hid
        self.o = nn.Linear(hid, 2)                 # 요약 → 점수 2개

    # 왜 · 칸마다 요약을 만든 뒤 마지막 요약으로 답을 내기 위해
    def forward(self, x):           # 입력이 지나가는 길
        o, _ = self.r(x)            # 칸마다 요약
        return self.o(o[:, -1, :])  # 마지막 칸 요약 → 점수


torch.manual_seed(SEED)                           # 같은 시작점
net = Net("RNN")                                  # 모델 하나
for L in (10, 50, 200):                           # 길이만 바꿔 넣음
    xs, _ = make_order_task(4, L, seed=3)         # 길이 L인 4줄
    n = sum(p.numel() for p in net.parameters())  # 파라미터 수
    print(f"길이 {L:>3} → 출력 {tuple(net(to_tensor(xs)).shape)} · 파라미터 {n:,}개")  # 길이와 무관
hand = 32 * 10 + 32 * 32 + 32 + 32 + 32 * 2 + 2   # W_ih + W_hh + bias 2개 + Linear
print("손계산 : 32 × 10 + 32 × 32 + 32 + 32 + 32 × 2 + 2 =", hand)  # 1,474

길이  10 → 출력 (4, 2) · 파라미터 1,474개
길이  50 → 출력 (4, 2) · 파라미터 1,474개
길이 200 → 출력 (4, 2) · 파라미터 1,474개
손계산 : 32 × 10 + 32 × 32 + 32 + 32 + 32 × 2 + 2 = 1474


**실행 흐름**

1. 3-1 RNNCell + for 문
2. 3-2 nn.RNN
3. 3-3 세 겹 모양 · 원-핫

In [38]:
# ---------- 3-1 ----------
# 상자 하나(RNNCell) + for 문 세 줄 = 손으로 펼친 RNN
# 왜 · RNN이 하는 일을 for 문으로 직접 펼쳐 이해하기 위해
torch.manual_seed(SEED)             # 같은 시작점
cell = nn.RNNCell(10, 32)           # 10 = 칸 하나의 표현 크기 · 32 = 넘길 요약 크기
x = to_tensor(xtr[:4])              # 학습 4줄 → (4, 10, 10)
h = torch.zeros(4, 32)              # 처음 요약 = 0
for t in range(10):                 # 시간을 펼친 것
    h = cell(x[:, t, :], h)         # 덮어쓰기 = 매 순간 새 요약
print("h.shape :", tuple(h.shape))  # 4 = 줄 수 · 32 = 요약 · 길이 10은 사라짐


# 상자 안의 계산을 직접: h_t = tanh(x_t W_ih^T + b_ih + h_(t-1) W_hh^T + b_hh)
# 왜 · RNNCell 안의 수식을 직접 계산해 결과가 같은지 검증하기 위해
h1 = torch.zeros(4, 32)                                                    # 다시 처음부터
for t in range(10):                                                        # 같은 10칸
    pre = x[:, t] @ cell.weight_ih.T + cell.bias_ih                        # 입력 쪽
    h1 = torch.tanh(pre + h1 @ cell.weight_hh.T + cell.bias_hh)            # + 이전 요약 쪽 → tanh
print("식으로 계산한 h == cell 결과 :", torch.allclose(h, h1, atol=1e-6))  # 같은 계산
# (32, 10) · (32, 32)
print("가중치 모양 W_ih", tuple(cell.weight_ih.shape), "· W_hh", tuple(cell.weight_hh.shape))


# ---------- 3-2 ----------
# PyTorch가 감싸 둔 판 nn.RNN: for 문이 안으로 들어감
# 왜 · 직접 만든 반복과 nn.RNN 결과가 같은지 확인하기 위해
torch.manual_seed(SEED)                          # 같은 시작점
rnn = nn.RNN(10, 32, batch_first=True)           # 판 전체
cell = nn.RNNCell(10, 32)                        # 비교할 상자
with torch.no_grad():                            # 같은 가중치를 복사
    for a, b in [("weight_ih", "weight_ih_l0"), ("weight_hh", "weight_hh_l0"),  # 이름 짝 4개
                 ("bias_ih", "bias_ih_l0"), ("bias_hh", "bias_hh_l0")]:
        getattr(cell, a).copy_(getattr(rnn, b))  # rnn → cell
x = to_tensor(xtr[:4])                           # (4, 10, 10)
h, hs = torch.zeros(4, 32), []                   # 처음 요약 · 칸별 요약
for t in range(10):                              # 손으로 펼친 판
    h = cell(x[:, t, :], h)                      # 새 요약
    hs.append(h)                                 # 저장
output, h_n = rnn(x)                             # 감싼 판
print("(output - manual).abs().max() :", (output - torch.stack(hs, 1)).abs().max().item())  # 0.0
# 돌려주는 것 두 개
print("x", tuple(x.shape), "· output", tuple(output.shape), "· h_n", tuple(h_n.shape))
# 사실상 같음
print("torch.allclose(output[:, -1, :], h_n[0]) :", torch.allclose(output[:, -1, :], h_n[0]))


# ---------- 3-3 ----------
# 거의 모두가 한 번 막히는 에러: 칸마다 숫자 1개를 넣음
# 왜 · 칸마다 숫자 하나만 넣을 때의 모양 에러를 미리 보기 위해
digits = torch.tensor([xtr[0]])                             # (1, 10) · 숫자 그대로
rnn = nn.RNN(10, 32, batch_first=True)                      # 칸마다 10개를 기대
x_bad = digits.unsqueeze(-1).float()                        # 축만 붙임 → (1, 10, 1)
try:
    rnn(x_bad)                                              # 마지막 겹이 1
except RuntimeError as e:                                   # 에러를 잡아 이어서 실행
    print(tuple(x_bad.shape), "→", str(e).splitlines()[0])  # 에러 첫 줄


# 고치는 법: 축을 붙이는 게 아니라 숫자를 열 칸으로 적음
# 왜 · 범주 숫자는 원-핫으로 바꿔야 한다는 것을 확인하기 위해
x = F.one_hot(digits, num_classes=10).float()                   # (1, 10) → (1, 10, 10)
output, h_n = rnn(x)                                            # 통과
print(tuple(x.shape), "→ output", tuple(output.shape), "· h_n", tuple(h_n.shape))  # 세 겹
print("숫자", xtr[0][1], "의 열 칸 :", x[0, 1].int().tolist())  # 그 자리만 1

h.shape : (4, 32)
식으로 계산한 h == cell 결과 : True
가중치 모양 W_ih (32, 10) · W_hh (32, 32)
(output - manual).abs().max() : 0.0
x (4, 10, 10) · output (4, 10, 32) · h_n (1, 4, 32)
torch.allclose(output[:, -1, :], h_n[0]) : True
(1, 10, 1) → input.size(-1) must be equal to input_size. Expected 10, got 1
(1, 10, 10) → output (1, 10, 32) · h_n (1, 1, 32)
숫자 3 의 열 칸 : [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]


**실행 흐름**

1. 3-4 batch_first
2. 3-5 output vs h_n
3. 3-6 HAR 모양

In [39]:
# ---------- 3-4 ----------
# batch_first: 앞 두 겹(시퀀스 개수 · 길이)의 순서만 바꿈 · 기본값은 False
# 왜 · 축 순서를 잘못 두면 에러 없이 틀린 결과가 나오는 함정을 보이기 위해
x = torch.randn(8, 10, 10)                           # 8줄 · 10칸 · 칸마다 10개
for bf in (True, False):                             # 켜기 · 끄기
    output, h_n = nn.RNN(10, 32, batch_first=bf)(x)  # 둘 다 에러 없음
    # 8 ↔ 10
    print(f"batch_first={str(bf):<5} → output {tuple(output.shape)} · h_n {tuple(h_n.shape)}")
x_tf = x.transpose(0, 1)  # (8, 10, 10) → (10, 8, 10) = 길이 · 개수 · 표현
output, h_n = nn.RNN(10, 32)(x_tf)                   # False에 맞게 바꿔 넣으면
print("transpose 후 False → h_n", tuple(h_n.shape))  # 다시 8줄


# ---------- 3-5 ----------
# output = 칸마다 요약 전부 · h_n = 마지막 요약 · 정말 같은 값일까
# 왜 · 두 출력이 언제 같은지 직접 비교하기 위해
torch.manual_seed(SEED)                                                  # 같은 시작점
x = to_tensor(xtr[:8])                                                   # (8, 10, 10)
output, h_n = nn.RNN(10, 32, batch_first=True)(x)                        # 1층 · 한 방향
print("equal :", torch.equal(output[:, -1, :], h_n[0]), "· allclose :",  # 비트까지 · 오차 허용
      torch.allclose(output[:, -1, :], h_n[0]))


# 조건이 깨지면: 층 여러 개 · 양방향 → h_n 첫 축 = 층 수 × 방향 수
# 왜 · 층 · 방향이 늘면 h_n 모양이 달라진다는 것을 확인하기 위해
for layers, bi in [(1, False), (2, False), (1, True), (2, True)]:  # 조합 4가지
    out, h_n = nn.RNN(10, 32, num_layers=layers, bidirectional=bi, batch_first=True)(x)  # 실행
    # 모양
    print(f"층 {layers} · 양방향 {str(bi):<5} → output {tuple(out.shape)} · h_n {tuple(h_n.shape)}")


# 칸마다 답 → output 전체 · 끝에 한 번 → 마지막 요약
# 왜 · 과제에 따라 어느 출력을 쓸지 고르는 기준을 세우기 위해
head = nn.Linear(32, 2)  # 요약 → 점수
# 두 경우
print("칸마다 답", tuple(head(output).shape), "· 줄마다 답", tuple(head(output[:, -1]).shape))


# ---------- 3-6 ----------
# UCI HAR 모양 읽기: [개수, 길이, 칸마다 값] = [7352, 128, 9] (데이터는 파일별 2에서 내려받음)
# 왜 · 실제 센서 데이터의 모양을 같은 규칙으로 읽기 위해
signals = ["가속도", "자이로", "전체 가속도"]              # 신호 3종
axes = ["x", "y", "z"]                                     # 축 3개
channels = [f"{s}_{a}" for s in signals for a in axes]     # 한 순간에 동시에 읽는 값
print("채널 수 :", len(channels), "→", channels[:3], "…")  # 3 × 3 = 9
print("한 줄 길이 : 128칸 ÷ 50Hz =", 128 / 50, "초")       # 2.56초씩 자른 창
X_shape = (7352, 128, 9)                                   # 학습 데이터 모양 (강의 화면)
n_num = 7352 * 128 * 9                                     # 숫자 개수
print("숫자 개수 :", f"{n_num:,}", "· float32 용량", round(n_num * 4 / 2**20, 1), "MB")  # 크기 감
print("모두 128칸 → 직사각형 텐서", tuple(torch.empty(X_shape).shape))  # 패딩 필요 없음

batch_first=True  → output (8, 10, 32) · h_n (1, 8, 32)
batch_first=False → output (8, 10, 32) · h_n (1, 10, 32)
transpose 후 False → h_n (1, 8, 32)
equal : True · allclose : True
층 1 · 양방향 False → output (8, 10, 32) · h_n (1, 8, 32)
층 2 · 양방향 False → output (8, 10, 32) · h_n (2, 8, 32)
층 1 · 양방향 True  → output (8, 10, 64) · h_n (2, 8, 32)
층 2 · 양방향 True  → output (8, 10, 64) · h_n (4, 8, 32)
칸마다 답 (8, 10, 2) · 줄마다 답 (8, 2)
채널 수 : 9 → ['가속도_x', '가속도_y', '가속도_z'] …
한 줄 길이 : 128칸 ÷ 50Hz = 2.56 초
숫자 개수 : 8,469,504 · float32 용량 32.3 MB
모두 128칸 → 직사각형 텐서 (7352, 128, 9)


**실행 흐름**

1. 4-1 padding · lens
2. 4-2 채우는 값
3. 4-3 진짜 끝 · mask 평균

In [40]:
# ---------- 4-1 ----------
# 길이가 3 · 6 · 4로 제각각인 세 줄을 그대로 텐서로
# 왜 · 길이가 다른 줄은 바로 텐서로 묶을 수 없다는 것을 확인하기 위해
seqs = [[3, 1, 7],                      # 길이 3
        [9, 7, 0, 5, 3, 2],             # 길이 6
        [1, 3, 4, 7]]                   # 길이 4
try:
    torch.tensor(seqs)                  # 배열은 직사각형이어야 함
except ValueError as e:                 # 에러를 잡아 이어서 실행
    print("그대로 텐서로 만들면 :", e)  # 에러 내용


# padding: 가장 긴 6칸 틀에 맞추고 진짜 길이표 lens를 같이 들고 다님
# 왜 · 빈 칸을 채우되 진짜 길이를 함께 들고 다녀 나중에 구분하기 위해
ML = max(len(s) for s in seqs)               # 6
X = torch.zeros(len(seqs), ML, 10)           # (3, 6, 10) 빈 틀
for b, s in enumerate(seqs):                 # 줄마다
    for t, d in enumerate(s):                # 진짜 칸만
        X[b, t, d] = 1.0                     # 숫자 d 자리에 1
lens = torch.tensor([len(s) for s in seqs])  # 어디까지가 진짜인지
# 5칸이 빔
print("X.shape :", tuple(X.shape), "· lens :", lens.tolist(), "· 빈 칸 수 :", (ML - lens).tolist())


# ---------- 4-2 ----------
# 무엇으로 채웠는지가 갈림: 빈 벡터(열 칸 전부 0) vs 숫자 0(첫 칸만 1)
# 왜 · 채우는 값이 모델 입력의 의미를 바꿀 수 있다는 것을 비교하기 위해
X0 = X.clone()                                                            # 복사본
X0[0, 3:, 0] = 1.0                                                        # 첫 줄 빈 칸을 숫자 0으로
print("빈 벡터로 채움 · 첫 줄 칸별 합   :", X[0].sum(1).int().tolist())   # [1, 1, 1, 0, 0, 0]
print("숫자 0 으로 채움 · 첫 줄 칸별 합 :", X0[0].sum(1).int().tolist())  # 전부 1
print("채운 칸 == 둘째 줄의 진짜 0 칸 :", torch.equal(X0[0, 3], X[1, 2]))  # 구분 불가


# ---------- 4-3 ----------
# 흔히 쓰는 output[:, -1]은 틀의 끝 · 진짜 끝은 output[idx, lens - 1]
# 왜 · padding이 있으면 마지막 칸이 진짜 끝이 아니라는 함정을 고치기 위해
torch.manual_seed(SEED)                          # 같은 시작점
output, _ = nn.RNN(10, 32, batch_first=True)(X)  # 채운 세 줄
idx = torch.arange(len(X))                       # 줄 번호
# 길이 6만
print("-1 == lens - 1 :", [torch.equal(a, b) for a, b in zip(output[:, -1], output[idx, lens - 1])])


# 평균에서 빈 칸 빼기: mask
# 왜 · 빈 칸이 평균을 망치지 않게 mask로 빼기 위해
T = X.shape[1]                                                        # 6
m = (torch.arange(T)[None, :] < lens[:, None]).float().unsqueeze(-1)  # (3, 6, 1) · 진짜 칸 = 1
h = (output * m).sum(1) / m.sum(1)                                    # (3, 32) 진짜 칸 평균
print("m 첫 줄 :", m[0, :, 0].int().tolist(), "· h", tuple(h.shape))  # 1 1 1 0 0 0
# 빈 칸 제외 확인
print("첫 줄 mask 평균 == 앞 3칸 평균 :", torch.allclose(h[0], output[0, :3].mean(0)))


# 편차가 작으면 버그가 안 드러남: 줄마다 생기는 빈 칸 수
# 왜 · 길이 차이가 작으면 버그가 숨는다는 것을 확인하기 위해
for lo, hi in [(4, 12), (5, 60)]:                      # 강의의 두 설정
    print(f"길이 {lo}~{hi} → 빈 칸 최대 {hi - lo}칸")  # 8칸 vs 55칸

그대로 텐서로 만들면 : expected sequence of length 3 at dim 1 (got 6)
X.shape : (3, 6, 10) · lens : [3, 6, 4] · 빈 칸 수 : [3, 0, 2]
빈 벡터로 채움 · 첫 줄 칸별 합   : [1, 1, 1, 0, 0, 0]
숫자 0 으로 채움 · 첫 줄 칸별 합 : [1, 1, 1, 1, 1, 1]
채운 칸 == 둘째 줄의 진짜 0 칸 : True
-1 == lens - 1 : [False, True, False]
m 첫 줄 : [1, 1, 1, 0, 0, 0] · h (3, 32)
첫 줄 mask 평균 == 앞 3칸 평균 : True
길이 4~12 → 빈 칸 최대 8칸
길이 5~60 → 빈 칸 최대 55칸


**실행 흐름**

1. 5-1 귓속말 전달
2. 5-2 손으로 심은 가중치
3. 5-3 기울기 재기

In [41]:
# ---------- 5-1 ----------
# 귓속말 전달: 맨 앞 두 칸만 바꾼 두 줄의 요약 차이가 칸마다 얼마나 남나 (학습 전 · CPU)
# 왜 · 앞의 작은 차이가 뒤로 갈수록 희미해지는 것을 숫자로 보기 위해
torch.manual_seed(SEED)                     # 같은 시작점
untrained = Net("RNN")                      # 학습 전 모델
xs, _ = make_front_task(512, 100, seed=31)  # 512줄 · 길이 100
xs_b = [[s[1], s[0]] + s[2:] for s in xs]   # 맨 앞 두 칸만 맞바꿈
with torch.no_grad():                       # 측정만
    oa, _ = untrained.r(to_tensor(xs))      # 원래 줄의 칸별 요약
    ob, _ = untrained.r(to_tensor(xs_b))    # 바꾼 줄의 칸별 요약
for t in (1, 2, 5, 10, 20, 40, 99):         # 칸마다
    d = ((oa[:, t] - ob[:, t]).norm(dim=1) / oa[:, t].norm(dim=1)).mean().item()  # 차이 ÷ 크기
    print(f"{t:>2}칸  {d:.3g}")             # 상대 차이
# 측정의 바닥
print("float32 가 구분하는 가장 작은 상대 차이 :", f"{torch.finfo(torch.float32).eps:.3g}")


# ---------- 5-2 ----------
# 학습 없이 손으로 심은 가중치: 요약을 그대로 넘기고 3이면 +0.5 · 7이면 -0.5
# 왜 · 가중치를 잘 고르면 RNN도 멀리 기억할 수 있다는 것을 보이기 위해
def handmade(L, scale=0.5):                     # 길이 L → 정확도 · 끝에 남은 크기
    cell = nn.RNNCell(10, 8)                    # 요약 8칸
    with torch.no_grad():                       # 가중치를 직접 적음
        for p in cell.parameters():             # 전부 0으로 시작
            p.zero_()                           # 0
        cell.weight_hh.copy_(torch.eye(8))      # 이전 요약을 그대로 넘김
        cell.weight_ih[0, 3], cell.weight_ih[0, 7] = scale, -scale  # 3 → +0.5 · 7 → -0.5
    xs, ys = make_front_task(1000, L, seed=2)   # 평가 1000줄
    X, h = to_tensor(xs), torch.zeros(1000, 8)  # 입력 · 처음 요약
    with torch.no_grad():                       # 측정만
        for t in range(L):                      # L칸
            h = cell(X[:, t, :], h)             # 같은 상자 반복
    acc = ((h[:, 0] < 0).long() == torch.tensor(ys)).float().mean().item()  # 7이 나중이면 음수 → 네
    return acc, h[:, 0].abs().mean().item()     # 정확도 · 첫 칸 크기


# 왜 · 길이가 늘어도 손으로 심은 가중치가 맞히는지 확인하기 위해
for L in (10, 100, 400):                                               # 거리 8 · 98 · 398
    acc, size = handmade(L)                                            # 손 가중치로
    print(f"거리 {L - 2:>3} → {acc:.3f} · 끝에 남은 크기 {size:.4f}")  # 기억은 닿을 수 있음


# ---------- 5-3 ----------
# 뒤 → 앞: 마지막 손실의 신호(기울기)가 k칸 거슬러 간 요약에 얼마나 오나
# 왜 · 기울기가 멀리 거슬러 갈수록 작아지는지 직접 재기 위해
STEPS = (0, 10, 20, 40, 70, 99)  # 거슬러 간 칸 수

 1칸  0.736
 2칸  0.378
 5칸  0.061
10칸  0.00255
20칸  7.67e-06
40칸  7.82e-08
99칸  7.94e-08
float32 가 구분하는 가장 작은 상대 차이 : 1.19e-07
거리   8 → 1.000 · 끝에 남은 크기 0.0377


거리  98 → 1.000 · 끝에 남은 크기 0.0362


거리 398 → 1.000 · 끝에 남은 크기 0.0322


**실행 흐름**

1. 5-4 float64로 다시 · output.grad 함정
2. 5-5 같은 수를 계속 곱하면

In [42]:
# 왜 · 셀 종류 · 문 시작값 · 자료형만 바꿔 같은 방법으로 기울기를 재기 위해
def grad_profile(kind, forget_bias=None, dtype=torch.float32,  # 칸별 기울기 크기
                 steps=STEPS, T=100, hid=32):
    torch.manual_seed(SEED)                                    # 같은 시작점
    cell = {"RNN": nn.RNNCell, "LSTM": nn.LSTMCell}[kind](10, hid).to(dtype)  # 손으로 펼칠 상자
    head = nn.Linear(hid, 2).to(dtype)                         # 끝 분류기
    if forget_bias is not None:  # bias 순서 [input | forget | cell | output] (6장)
        with torch.no_grad():                                  # 값만 넣음
            cell.bias_ih[hid:2 * hid].fill_(forget_bias)       # forget 구간
            cell.bias_hh[hid:2 * hid].fill_(0.0)               # 두 bias가 더해지므로 한쪽은 0
    xs, ys = make_front_task(64, T, seed=7)                    # 64줄 · 길이 100
    X = to_tensor(xs).to(dtype)                                # 자료형 맞춤
    # 요약 · 메모
    h, c, hs = torch.zeros(64, hid, dtype=dtype), torch.zeros(64, hid, dtype=dtype), []
    for t in range(T):                                         # T칸 차례로
        h = cell(X[:, t], h) if kind == "RNN" else cell(X[:, t], (h, c))  # RNN은 h · LSTM은 (h, c)
        if kind == "LSTM":                                     # LSTM은 둘을 돌려줌
            h, c = h                                           # 요약 · 메모
        h.retain_grad()  # 중간 요약의 기울기를 버리지 말고 남기기
        hs.append(h)                                           # 저장
    nn.CrossEntropyLoss()(head(hs[-1]), torch.tensor(ys)).backward()  # 끝에서 손실 → 역전파
    return [hs[T - 1 - k].grad.norm().item() for k in steps]   # k칸 거슬러 간 기울기 크기


# 왜 · 한글 폭을 맞춰 여러 결과를 표처럼 나란히 읽기 위해
def show(name, values):                                          # 한 줄 출력 (한글 2칸 폭)
    pad = 12 - sum(2 if ord(ch) > 0x1100 else 1 for ch in name)  # 모자란 폭
    cells = [f"{v:>9}" if isinstance(v, int) else f"{v:>9.1e}" for v in values]  # 정수 · 지수 표기
    print(name + " " * pad + "".join(cells))                     # 열 맞춤 출력


show("거슬러 간 칸", STEPS)  # 머리 줄
rnn_g = grad_profile("RNN")  # 학습 전 RNN
show("RNN", rnn_g)           # 칸별 크기
# 제곱처럼 커짐
print("10 · 20 · 40스텝 배율 :", " · ".join(f"{rnn_g[0] / rnn_g[k]:,.0f}배" for k in (1, 2, 3)))


# ---------- 5-4 ----------
# 99스텝의 0은 진짜 0일까: 같은 계산을 float64로
# 왜 · float32에서 0으로 보인 값이 정말 0인지 float64로 다시 확인하기 위해
show("거슬러 간 칸", STEPS)                                      # 머리 줄
show("RNN float64", grad_profile("RNN", dtype=torch.float64))    # 0이 아니라 아주 작은 수
every = grad_profile("RNN", steps=range(100))                    # float32 · 0 ~ 99칸 전부
first0 = next(k for k in range(100) if every[k] == 0.0)          # 처음 0인 칸
print("float32 에서 처음 0 으로 찍히는 자리 :", first0, "스텝")  # 적을 수 있는 한계


# 쓰면 안 되는 지름길: output.retain_grad() → output.grad 는 h의 기울기가 아님
# 왜 · output.grad가 h의 기울기가 아니라는 함정을 미리 보기 위해
torch.manual_seed(SEED)                                                      # 같은 시작점
rnn, head = nn.RNN(10, 32, batch_first=True), nn.Linear(32, 2)               # 감싼 판 · 분류기
xs, ys = make_front_task(64, 100, seed=7)                                    # 같은 데이터
output, _ = rnn(to_tensor(xs))                                               # 칸별 요약
output.retain_grad()                                                         # output 기울기 남기기
nn.CrossEntropyLoss()(head(output[:, -1]), torch.tensor(ys)).backward()      # 끝 칸만 손실에
zero = sum(output.grad[:, t].abs().max().item() == 0.0 for t in range(100))  # 정확히 0인 칸
print("output.grad 가 정확히 0 인 칸 :", zero, "/ 100")                      # 감춰진 길은 안 보임


# ---------- 5-5 ----------
# 같은 수를 계속 곱하면: 1보다 조금 작으면 사라지고 조금 크면 터짐
# 왜 · 1보다 작으면 사라지고 크면 터지는 원리를 숫자로 보기 위해
# 소실 · 폭발 · 유지
print(f"0.9 ** 50 = {0.9 ** 50:.3f} · 1.1 ** 50 = {1.1 ** 50:.0f} · 1.0 ** 50 = {1.0 ** 50:.0f}")

거슬러 간 칸        0       10       20       40       70       99
RNN           5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  0.0e+00
10 · 20 · 40스텝 배율 : 486배 · 175,503배 · 38,772,643,119배
거슬러 간 칸        0       10       20       40       70       99
RNN float64   5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  3.4e-28
float32 에서 처음 0 으로 찍히는 자리 : 77 스텝
output.grad 가 정확히 0 인 칸 : 99 / 100
0.9 ** 50 = 0.005 · 1.1 ** 50 = 117 · 1.0 ** 50 = 1


**실행 흐름**

1. gradient clipping · 행렬의 고유값

In [43]:
# 폭발 쪽 대처: gradient clipping (전체 크기가 기준을 넘으면 같은 비율로 줄임)
# 왜 · 기울기가 너무 커질 때 전체 크기만 줄여 학습을 안정시키기 위해
w = nn.Parameter(torch.zeros(3))                                        # 파라미터 3개
w.grad = torch.tensor([30.0, 40.0, 0.0])                                # 크기 50짜리 기울기
nn.utils.clip_grad_norm_([w], max_norm=5.0)                             # 크기 5로 자르기
g = [round(v, 3) for v in w.grad.tolist()]                              # 자른 기울기
print("clip 후 기울기 :", g, "· 크기", round(w.grad.norm().item(), 3))  # 방향은 그대로


# 실제로 곱해지는 것은 숫자 하나가 아니라 행렬 (학습 전 · float64)
# 왜 · 행렬의 고유값으로 사라짐 · 폭발을 판단하는 방법을 보기 위해
torch.manual_seed(SEED)                                             # 6-3과 같은 시작점
W = nn.RNNCell(10, 32).to(torch.float64).weight_hh                  # 이전 요약에 곱하는 행렬
print(f"최대 특이값 {torch.linalg.svdvals(W).max().item():.2f} · "  # 한 번 곱할 때 최대 배율
      # 거듭 곱할 때 남는 배율
      f"고유값 크기 최댓값 {torch.linalg.eigvals(W).abs().max().item():.2f}")
g64 = grad_profile("RNN", dtype=torch.float64, steps=range(71))     # 0 ~ 70칸 기울기
# 실측
print(f"실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝) : {(g64[70] / g64[0]) ** (1 / 70):.2f}")

clip 후 기울기 : [3.0, 4.0, 0.0] · 크기 5.0
최대 특이값 1.10 · 고유값 크기 최댓값 0.57
실제 기울기에서 잰 한 칸당 비율 (0 → 70스텝) : 0.54


**실행 흐름**

1. 6-1 문 = sigmoid · 메모
2. 6-2 LSTM 출력
3. 6-3 문을 달아도 죽는 경우

In [44]:
# ---------- 6-1 ----------
# 수도꼭지 = sigmoid: 어떤 값이 와도 0 ~ 1 사이 → 여는 정도로 씀
# 왜 · 문(gate)이 0 ~ 1 사이 값으로 여는 정도를 정한다는 것을 확인하기 위해
z = torch.tensor([-4.0, -2.0, 0.0, 2.0, 4.0])                         # 안쪽 계산값 (아무 값)
print("sigmoid :", [round(v, 2) for v in torch.sigmoid(z).tolist()])  # 0 · 0.5 · 1 근처
value = torch.tensor(10.0)                                            # 원래 값
for f in (0.0, 0.5, 1.0):                                             # 잠금 · 반 · 다 열림
    print(f"f = {f} → 통과한 양 {f * value.item():.1f}")              # 곱하기 하나


# 메모 c에 허용된 일 두 가지: 지우기(곱) · 덧붙이기(더하기)
# 왜 · LSTM 메모가 곱(지우기)과 더하기(덧붙이기)로만 바뀐다는 것을 보기 위해
c = torch.tensor([1.0, 1.0, 1.0])                                    # 이전 메모
f = torch.tensor([1.0, 0.5, 0.0])                                    # forget 문: 남길 양
i, g = torch.tensor([0.0, 1.0, 1.0]), torch.tensor([0.3, 0.3, 0.3])  # input 문 · 적을 내용
c_new = f * c + i * g                                                # c ← f × c + i × g
o = torch.tensor([1.0, 1.0, 0.0])                                    # output 문: 꺼낼 양
h_new = o * torch.tanh(c_new)                                        # h ← o × tanh(c)
print("새 메모 c :", [round(v, 3) for v in c_new.tolist()],          # 메모
      "· 꺼낸 h :", [round(v, 3) for v in h_new.tolist()])


# ---------- 6-2 ----------
# LSTM은 텐서 세 개를 돌려줌: output, (h_n, c_n)
# 왜 · LSTM 출력 모양을 RNN과 비교해 확인하기 위해
torch.manual_seed(SEED)                                      # 같은 시작점
lstm = nn.LSTM(10, 32, batch_first=True)                     # 입력 10 · hidden 32
x = to_tensor(xtr[:8])                                       # (8, 10, 10)
output, (h_n, c_n) = lstm(x)                                 # 괄호 안 두 개 = 튜플 풀기
print("x", tuple(x.shape), "· output", tuple(output.shape),  # 네 모양
      "· h_n", tuple(h_n.shape), "· c_n", tuple(c_n.shape))
print("h_n 과 c_n 이 같은 값인가 :", torch.equal(h_n, c_n))  # h = o × tanh(c)
n_rnn = sum(p.numel() for p in nn.RNN(10, 32).parameters())  # 1벌
n_lstm = sum(p.numel() for p in lstm.parameters())           # 문 3개 + 적을 내용 = 4벌
# 같은 hidden에서 4배
print(f"파라미터 RNN {n_rnn:,} · LSTM {n_lstm:,} · 배율 {n_lstm / n_rnn:.0f}")
print("weight_ih 모양 :", tuple(lstm.weight_ih_l0.shape), "= (4 × 32, 10)")  # [i | f | g | o]


# ---------- 6-3 ----------
# 같은 조건(데이터 · 길이 100 · hidden 32)에서 셀 종류만 바꿈 · forget 문 bias 0 = 보통의 LSTM
# 왜 · 셀 종류만 바꾼 공정한 비교로 LSTM의 효과를 확인하기 위해
show("거슬러 간 칸", STEPS)                               # 머리 줄
show("RNN", grad_profile("RNN"))                          # 비교 기준
show("LSTM 기본", grad_profile("LSTM", forget_bias=0.0))  # 문을 달았지만


# 왜 죽었나: 문에 더하는 값이 0이면 문 값이 0.5 근처 → 반쯤 열린 문을 백 번 지남
# 왜 · 문이 반쯤 열린 채 시작하면 멀리 기억하지 못하는 원인을 찾기 위해
torch.manual_seed(SEED)                                                         # 같은 시작점
cell = nn.LSTMCell(10, 32)                                                      # 학습 전 LSTM 상자
with torch.no_grad():                                                           # 측정만
    cell.bias_ih[32:64].fill_(0.0)                                              # forget 구간 0
    cell.bias_hh[32:64].fill_(0.0)                                              # 한쪽도 0
    xs, _ = make_order_task(64, 100, seed=7)                                    # 길이 100
    X, h, c, fs = to_tensor(xs), torch.zeros(64, 32), torch.zeros(64, 32), []   # 입력 · 상태 · 기록
    for t in range(100):                                                        # 100칸
        # 네 문
        g4 = X[:, t] @ cell.weight_ih.T + cell.bias_ih + h @ cell.weight_hh.T + cell.bias_hh
        fs.append(torch.sigmoid(g4[:, 32:64]).mean().item())                    # forget 문 값 평균
        h, c = cell(X[:, t], (h, c))                                            # 한 칸 진행
print(f"문 값 백 칸 평균 {sum(fs) / 100:.4f} · 0.5 ** 100 = {0.5 ** 100:.1e}")  # 10^-30 쪽

sigmoid : [0.02, 0.12, 0.5, 0.88, 0.98]
f = 0.0 → 통과한 양 0.0
f = 0.5 → 통과한 양 5.0
f = 1.0 → 통과한 양 10.0
새 메모 c : [1.0, 0.8, 0.3] · 꺼낸 h : [0.762, 0.664, 0.0]
x (8, 10, 10) · output (8, 10, 32) · h_n (1, 8, 32) · c_n (1, 8, 32)
h_n 과 c_n 이 같은 값인가 : False
파라미터 RNN 1,408 · LSTM 5,632 · 배율 4
weight_ih 모양 : (128, 10) = (4 × 32, 10)
거슬러 간 칸        0       10       20       40       70       99
RNN           5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  0.0e+00


LSTM 기본     5.3e-02  4.6e-05  4.0e-07  2.8e-11  2.1e-17  0.0e+00
문 값 백 칸 평균 0.4997 · 0.5 ** 100 = 7.9e-31


**실행 흐름**

1. 6-4 forget 문을 열어두고 시작

In [45]:
# ---------- 6-4 ----------
# 문을 열어두고 시작: forget 문의 bias를 2 · 4로 (이름이 아니라 자리로 찾음)
# 왜 · forget 문의 시작값만 바꿔 멀리 기억하는지 확인하기 위해
H = 32                                                                 # hidden 크기
# 두 번째 구간
print("bias 자리 [ input | forget | cell | output ] → forget =", f"bias_ih[{H}:{2 * H}]")
start = torch.sigmoid(torch.tensor([0., 2., 4.]))                      # bias → 시작 문 값
print("시작 문 값 sigmoid(0 · 2 · 4) :", [round(v, 2) for v in start.tolist()])  # 0.5 · 0.88 · 0.98
show("거슬러 간 칸", STEPS)                                            # 머리 줄
show("RNN", grad_profile("RNN"))                                       # 0으로 죽음
show("LSTM 기본", grad_profile("LSTM", forget_bias=0.0))               # 0으로 죽음
opened = grad_profile("LSTM", forget_bias=2.0)                         # 열어둠
show("LSTM 열어둠", opened)                                            # 99칸에도 남음
show("LSTM 활짝", grad_profile("LSTM", forget_bias=4.0))               # 더 남음
print(f"열어둔 경우 0스텝 ÷ 99스텝 = {opened[0] / opened[5]:,.0f}배")  # 해결이 아니라 크게 늦춤

bias 자리 [ input | forget | cell | output ] → forget = bias_ih[32:64]
시작 문 값 sigmoid(0 · 2 · 4) : [0.5, 0.88, 0.98]
거슬러 간 칸        0       10       20       40       70       99
RNN           5.2e-02  1.1e-04  2.9e-07  1.3e-12  1.5e-20  0.0e+00
LSTM 기본     5.3e-02  4.6e-05  4.0e-07  2.8e-11  2.1e-17  0.0e+00


LSTM 열어둠   5.3e-02  1.0e-03  5.2e-04  1.9e-04  3.9e-05  1.6e-05
LSTM 활짝     5.3e-02  7.2e-04  5.9e-04  6.7e-04  9.8e-04  4.5e-03
열어둔 경우 0스텝 ÷ 99스텝 = 3,303배


**실행 흐름**

1. 7-1 GRU · 파라미터 수
2. 7-2 통제된 비교
3. 7-3 양방향

In [46]:
# ---------- 7-1 ----------
# GRU: 손잡이 2개 · 넘기는 줄 1개 → RNN처럼 두 개만 돌려줌
# 왜 · GRU의 출력 모양을 LSTM과 비교하기 위해
torch.manual_seed(SEED)                                           # 같은 시작점
out, h_n = nn.GRU(10, 32, batch_first=True)(to_tensor(xtr[:8]))   # (h_n, c_n) 아님
print("GRU output", tuple(out.shape), "· h_n", tuple(h_n.shape))  # 메모 c 없음


# HAR 크기(입력 9 · 출력 6)의 파라미터: RNN 1벌 · GRU 3벌 · LSTM 4벌 + Linear
# 왜 · 셀 종류마다 파라미터가 몇 벌인지 세어 비용을 비교하기 위해
def har_params(kind, hid):                                           # 층 + Linear(hid → 6)
    layer = getattr(nn, kind)(9, hid, batch_first=True)              # 층 만들기
    return sum(p.numel() for p in layer.parameters()) + hid * 6 + 6  # 분류기까지


# 왜 · 크기를 맞췄을 때 파라미터 수가 어떻게 달라지는지 비교하기 위해
sizes = [("RNN", 64), ("LSTM", 64), ("GRU", 64), ("RNN", 128), ("GRU", 74)]  # 강의의 다섯 크기
for kind, hid in sizes:                                                      # 크기마다
    # 같은 64 ≠ 같은 크기
    print(f"{kind:<4} hidden {hid:<3} → 파라미터 {har_params(kind, hid):>6,}")


# ---------- 7-2 ----------
# 강의 화면 숫자로 두 비교 읽기 (HAR · 12 epoch · 강의 제작 환경)
# 왜 · 강의 결과를 통제된 비교 기준으로 다시 읽기 위해
acc = {"RNN 64": 82.5, "LSTM 64": 90.6, "GRU 64": 90.1, "RNN 128": 83.2, "GRU 74": 90.8}  # 정확도 %
width = {"RNN": 2.75, "LSTM": 0.95, "GRU": 1.46}           # seed 3번 흔들림 폭 %p
gap = {k: acc[k] - acc["RNN 64"] for k in ("LSTM 64", "GRU 64", "RNN 128")}  # RNN 64 기준 간격
print(f"구조만 바꿈 · RNN→LSTM {gap['LSTM 64']:.1f}%p · RNN→GRU {gap['GRU 64']:.1f}%p"  # 첫 간격
      f" · LSTM-GRU {acc['LSTM 64'] - acc['GRU 64']:.1f}%p")
print(f"크기 맞춤 · RNN 64→128 {gap['RNN 128']:.1f}%p",    # 크기 덕이 아님
      f"(RNN 흔들림 폭 {width['RNN']}%p 안) · GRU74-LSTM {acc['GRU 74'] - acc['LSTM 64']:.1f}%p")
print(f"평가 2,947개 중 0.5%p = 약 {2947 * 0.005:.0f}개",  # 차이를 개수로
      f"· 파라미터 19590 ÷ 5190 = {19590 / 5190:.2f}배")
sec = {"RNN 128": 20.1, "GRU 74": 36.8, "LSTM 64": 52.4}   # 학습 시간(초) · 제작 맥 기준
# 비용으로 고름
print("파라미터를 맞춰도 시간은 다름 :", " · ".join(f"{k} {v}초" for k, v in sec.items()))


# ---------- 7-3 ----------
# 양방향: 같은 줄을 앞 → 뒤 · 뒤 → 앞으로 한 번씩 읽음
# 왜 · 양방향 RNN이 출력을 어떻게 만드는지 확인하기 위해
torch.manual_seed(SEED)                                    # 같은 시작점
bi = nn.RNN(10, 32, bidirectional=True, batch_first=True)  # 양방향
out, h_n = bi(to_tensor(xtr[:8]))                          # 칸마다 두 방향 요약
print("output", tuple(out.shape), "= 32 × 2 · h_n", tuple(h_n.shape), "= 방향 2")  # 앞 32 · 뒤 32
# 끝에서 출발
print("뒤 방향의 끝은 0번 칸 : out[:, 0, 32:] == h_n[1] →", torch.allclose(out[:, 0, 32:], h_n[1]))


# 실시간이면: 뒤에서 앞으로 읽을 끝이 아직 없음 (지금까지 3칸만 들어옴)
# 왜 · 실시간 예측에는 양방향을 쓸 수 없는 이유를 확인하기 위해
stream = xtr[0][:3]  # 지금까지 들어온 값
# 쓸 수 없는 자리
print("지금까지", stream, "→ 앞 → 뒤 RNN 은 출력 가능 · 양방향은 끝을 기다려야 함")

GRU output (8, 10, 32) · h_n (1, 8, 32)
RNN  hidden 64  → 파라미터  5,190
LSTM hidden 64  → 파라미터 19,590
GRU  hidden 64  → 파라미터 14,790
RNN  hidden 128 → 파라미터 18,566
GRU  hidden 74  → 파라미터 19,320
구조만 바꿈 · RNN→LSTM 8.1%p · RNN→GRU 7.6%p · LSTM-GRU 0.5%p
크기 맞춤 · RNN 64→128 0.7%p (RNN 흔들림 폭 2.75%p 안) · GRU74-LSTM 0.2%p
평가 2,947개 중 0.5%p = 약 15개 · 파라미터 19590 ÷ 5190 = 3.77배
파라미터를 맞춰도 시간은 다름 : RNN 128 20.1초 · GRU 74 36.8초 · LSTM 64 52.4초
output (8, 10, 64) = 32 × 2 · h_n (2, 8, 32) = 방향 2
뒤 방향의 끝은 0번 칸 : out[:, 0, 32:] == h_n[1] → True
지금까지 [2, 3, 1] → 앞 → 뒤 RNN 은 출력 가능 · 양방향은 끝을 기다려야 함


**실행 흐름**

1. 1 ~ 3단계 점검
2. 5 · 6단계 학습
3. 7 ~ 9단계 셀 비교

In [47]:
# ---------- 8. 종합 실습 ----------
# 1 ~ 3단계: 한 샘플 · 길이 → 순서를 섞어봄 → 무엇을 기억할지
# 왜 · 앞에서 배운 점검 순서를 새 과제에 그대로 적용하기 위해
# 길이 10 고정
print("1단계 · 한 샘플 = 숫자", len(xtr[0]), "칸 · 길이 고정 :", len(set(map(len, xtr))) == 1)
rng = random.Random(1)                                              # 섞기 전용
changed = sum(int(s.index(3) < s.index(7)) != y                     # 섞은 뒤 답이 바뀐 줄
              for s, y in zip((rng.sample(s, 10) for s in xte), yte))
print(f"2단계 · 칸 순서를 섞으면 답이 바뀌는 줄 {changed / len(xte):.3f} → 순서가 중요")  # 절반가량
print("3단계 · 기억할 것 = 3 · 7 중 무엇을 먼저 봤는가 (상태 셋)")  # 2-3의 결론
# 1-2 함수
print(f"4단계 · 사람이 정한 요약(합)의 최고 정확도 {best_by_sum(xtr, ytr, xte, yte):.3f}")


# 5 · 6단계: 넘길 것을 학습으로 · 입력 (N, 10, 10) → 출력 (N, 2)
# 왜 · 손으로 정한 상태 대신 학습으로 넘길 것을 찾게 하기 위해
# 왜 · 모든 비교가 같은 학습 절차 · 같은 평가 방법을 쓰도록 한 함수로 묶기 위해
def fit(model, X, Y, Xe, Ye, epochs, lr=3e-3, bs=128):                 # 학습 → 평가 정확도
    model, X, Y, Xe, Ye = model.to(DEV), X.to(DEV), Y.to(DEV), Xe.to(DEV), Ye.to(DEV)  # 같은 장치로
    opt = torch.optim.Adam(model.parameters(), lr=lr)                  # Adam
    for _ in range(epochs):                                            # epoch 반복
        perm = torch.randperm(len(X))                                  # 순서 섞기
        for i in range(0, len(X), bs):                                 # 미니배치
            j = perm[i:i + bs].to(DEV)                                 # 이번 배치
            opt.zero_grad()                                            # 지난 기울기 지우기
            nn.functional.cross_entropy(model(X[j]), Y[j]).backward()  # 손실 → 기울기
            opt.step()                                                 # 가중치 수정
    with torch.no_grad():                                              # 평가
        return (model(Xe).argmax(1) == Ye).float().mean().item()       # 정확도


# 왜 · 학습 · 평가 데이터를 모델 입력 모양으로 바꾸기 위해
Xa, Xb = to_tensor(xtr), to_tensor(xte)             # (6000, 10, 10) · (2000, 10, 10)
# 세 겹 → 점수 2개
print("6단계 · 입력", tuple(Xa.shape), "→ 출력", tuple(Net("RNN")(Xa[:2]).shape))
torch.manual_seed(SEED)                             # 같은 시작점
acc = fit(Net("RNN"), Xa, torch.tensor(ytr), Xb, torch.tensor(yte), epochs=30)  # 강의와 같은 설정
print(f"5단계 · RNN 학습 → 평가 정확도 {acc:.4f}")  # 요약을 학습으로


# 7 ~ 9단계: 거리 98 · 같은 예산(20 epoch)에서 셀 종류 · 문 시작값만 바꾼 통제된 비교
# 왜 · 멀리 기억해야 하는 과제에서 셀 종류 · 문 시작값만 바꿔 같은 예산으로 비교하기 위해
xa, ya = make_front_task(4000, 100, seed=1)                               # 학습 4000줄 · 거리 98
xb, yb = make_front_task(1000, 100, seed=2)                               # 평가 1000줄
Xa, Ya, Xb, Yb = to_tensor(xa), torch.tensor(ya), to_tensor(xb), torch.tensor(yb)  # 텐서
res = {}                                                                  # 이름별 정확도
for name, kind, fb in [("RNN", "RNN", None), ("LSTM 기본", "LSTM", 0.0),  # 비교 4가지
                       ("LSTM 열어둠", "LSTM", 2.0), ("GRU", "GRU", None)]:
    torch.manual_seed(SEED)                                               # 같은 시작점
    model = Net(kind)                                                     # 셀 종류만 바꿈
    if fb is not None:  # LSTM이면 forget 문 시작값 (6-4)
        with torch.no_grad():                                             # 값만 넣음
            model.r.bias_ih_l0[32:64].fill_(fb)                           # forget 구간
            model.r.bias_hh_l0[32:64].fill_(0.0)                          # 한쪽은 0
    res[name] = fit(model, Xa, Ya, Xb, Yb, epochs=20)                     # 같은 데이터 · 같은 예산
    print(f"7단계 · 거리 98 · 20 epoch · {name} → {res[name]:.3f}")       # 닿는가
# 신호가 안 옴
print("8단계 · 학습 전 RNN 기울기 0 · 10 · 20스텝 :", [f"{v:.1e}" for v in grad_profile("RNN")[:3]])
# 문이 열려 있을 때 차이
print(f"9단계 · 열어둔 LSTM - RNN = {res['LSTM 열어둠'] - res['RNN']:+.3f}")

1단계 · 한 샘플 = 숫자 10 칸 · 길이 고정 : True
2단계 · 칸 순서를 섞으면 답이 바뀌는 줄 0.487 → 순서가 중요
3단계 · 기억할 것 = 3 · 7 중 무엇을 먼저 봤는가 (상태 셋)
4단계 · 사람이 정한 요약(합)의 최고 정확도 0.501


6단계 · 입력 (6000, 10, 10) → 출력 (2, 2)


5단계 · RNN 학습 → 평가 정확도 1.0000


7단계 · 거리 98 · 20 epoch · RNN → 0.517


7단계 · 거리 98 · 20 epoch · LSTM 기본 → 0.512


7단계 · 거리 98 · 20 epoch · LSTM 열어둠 → 1.000


7단계 · 거리 98 · 20 epoch · GRU → 0.496
8단계 · 학습 전 RNN 기울기 0 · 10 · 20스텝 : ['5.2e-02', '1.1e-04', '2.9e-07']
9단계 · 열어둔 LSTM - RNN = +0.483
